In [ ]:
# Execute data processing or visualization steps
# Import standard libraries and common data science packages
import numpy as np
import pandas as pd
import textwrap
import os
import dill as pickle
import nltk
import re
import psutil
import math
import textwrap
import torch
import spacy
import pytextrank
import seaborn as sns
import json

# Plotly imports for interactive charts
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Matplotlib imports for static plotting
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

# Utility imports
from itertools import tee

# Progress bar integration with pandas
from tqdm import tqdm
tqdm.pandas(desc="progress")

# NLP imports
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Scikit-learn imports for modeling and similarity computation
from sklearn.model_selection import train_test_split
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import FeatureAgglomeration
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer 

# Sentence-transformers for embedding-based similarity
from sentence_transformers import SentenceTransformer, util

from itertools import chain

from tabulate import tabulate
from prettytable import PrettyTable

from collections import defaultdict, Counter

from itables import show
# import itables
# itables.init_notebook_mode()

from pybibx.base import pbx_probe

# Load topic index from parquet file
topic_index_name = "topic_index.parquet"
topic_index_df = pd.read_parquet(topic_index_name)

# Load bibliography data from pickle file
bibfile_name = "imported_bibs_kw_normalized_relevant.pkl"
with open(bibfile_name, "rb") as inp:
    bibfile = pickle.load(inp)

# Load main dataset from parquet
df_data_name = "statistical_study_relevant_df_data.parquet"
df_data = pd.read_parquet(df_data_name)

# Notebook view mode
view = "notebook"

df_data.columns

In [ ]:
# Execute data processing or visualization steps
from impact_factor.core import Factor

# Initialize the impact factor lookup service
fa = Factor()

def obtem_fi(row):
    # Extract ISSN and journal identifiers from the row
    issn, eissn, journal = row['issn'], row['eissn'], row['journal']

    # Prefer electronic ISSN first, then print ISSN, then journal name
    if eissn != 'UNKNOWN':
        chave = eissn
    elif issn != 'UNKNOWN':
        chave = issn
    elif journal != 'UNKNOWN':
        chave = journal.lower()
    else:
        return None
    
    # Search the impact factor database for the selected identifier
    ifs = fa.search(chave)

    if len(ifs) == 0:
        return None
    
    return ifs[0]["factor"]

# Add a new column with journal impact factor values
df_data['journal_if'] = df_data.progress_apply(obtem_fi, axis=1)

### Adjust df_data

In [ ]:
# Execute data processing or visualization steps
# Create the df_data with the data of bibfile.data and more additional information

df_data_name = 'df_data.parquet'

# Create the df_data
bibfile_name = 'imported_bibs_kw_normalized_relevant.pkl'
with open(bibfile_name, 'rb') as inp:
    bibfile = pickle.load(inp)

# Create the df_data
df_data = bibfile.data.copy()

# Ensure the Year is in int format
df_data['year'] = df_data['year'].astype(int)

# List of authors
df_data["authors"] = bibfile.aut

# List of authors' countries
df_data["countries"] = bibfile.ctr

# List of authors' languages
df_data["languages"] = bibfile.lan

# Total quantity of citations
df_data["citations_tot"] = bibfile.citation

# First author's institution
print("Separating first institution:")
df_data["first_institution"] = df_data["affiliation"].progress_apply(
    lambda x: x.split(";")[0].strip().replace("\&", "&") if isinstance(x, str) and x.strip() != "" else ""
)

# List of author's keywords
df_data['kw_authors'] = bibfile.data['author_keywords'].str.split(';')

# Save intermediate df_data
print("Saved 1...")
df_data.to_pickle(df_data_name)


# Normalize author keywords
# Download nltk resources (only once)
nltk.download('wordnet')
nltk.download('omw-1.4')
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()
print("Separating author keywords:")
df_data["kw_authors_normalized"] = pd.DataFrame({'kws': bibfile.auk}).progress_apply(
    lambda lista: [palavra.strip().lower() for palavra in lista if isinstance(palavra, str) and palavra.strip()]
)
def ajusta_palavra(texto):
    if texto == "unknown":
        return ""
    texto = lemmatizer.lemmatize(texto)
    # A plural that is not being removed
    texto = texto.replace('electrocatalysts', 'electrocatalyst')
    # Remove duplicate spaces
    texto = texto.replace(r'\s+', ' ')
    # Replace acronyms
    texto = texto.replace('oxygen evolution reaction (oer)', 'oxygen evolution reaction')
    texto = texto.replace('oer', 'oxygen evolution reaction')
    texto = texto.replace('her', 'hydrogen evolution reaction')
    return texto
    # Remove words that should not be there
    # text = text.upper() != 'UNKNOWN']
print("Normalizing author keywords:")
df_data['kw_authors_normalized'] = df_data['kw_authors'].progress_apply(lambda x: [ajusta_palavra(y) for y in x])

# Save intermediate df_data
print("Saved 2...")
df_data.to_pickle(df_data_name)

# Extract and normalize abstract keywords
# Load spaCy + pyTextRank model
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")
# Remove acronyms
siglas  = {'oer': 'Oxygen Evolution Reaction',
            'her': 'Hydrogen Evolution Reaction',
            'orr': 'Oxygen Reduction Reaction'}
def normaliza_palavra(texto):
    texto_modificado = texto.replace(r'\s+', ' ')
    for sigla, termo in siglas.items():
        # Regex to detect common acronym patterns with the term
        padroes_combinados = [
            rf"\(?{sigla}\)?\s*[-–:]\s*{re.escape(termo)}",   # (OER) - Oxygen Evolution Reaction
            rf"{re.escape(termo)}\s*\(?{sigla}\)?",           # Oxygen Evolution Reaction (OER)
            rf"\(?{sigla}\)?\s*{re.escape(termo)}",           # OER Oxygen Evolution Reaction
            rf"{re.escape(termo)}\s*\(?{sigla}\)?",           # Oxygen Evolution Reaction OER
        ]
        # Step 1: Remove the acronym when it appears together with the term
        for padrao in padroes_combinados:
            texto_modificado = re.sub(padrao, termo, texto_modificado, flags=re.IGNORECASE)
        # Step 2: Replace the isolated acronym with the term (if it still exists)
        def substituir_sigla_isolada(m):
            return termo
        texto_modificado = re.sub(rf"\b{sigla}\b", substituir_sigla_isolada, texto_modificado, flags=re.IGNORECASE)
    # Clean duplicate spaces left by removals
    texto_modificado = re.sub(r'\s{2,}', ' ', texto_modificado).strip()
    return texto_modificado    
def extract_keywords(text):
    text = normaliza_palavra(text)
    doc = nlp(text)
    keywords = [phrase.text.lower() for phrase in doc._.phrases]
    counts = Counter(keywords)
    return list(counts.items())
# Apply to DataFrame
print("Extracting abstract keywords:")
df_data["kw_abstract"] = df_data["abstract"].progress_apply(extract_keywords)
# Group by similarity using sentencetransformers
# Load model with GPU support
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
similarity_threshold = 0.85
# Normalization function
def normalize_keywords(kw_list):
    if not kw_list or len(kw_list) == 1:
        return kw_list
    keywords, counts = zip(*kw_list)
    keywords = list(keywords)
    counts = list(counts)
    # Embeddings
    embeddings = model.encode(keywords, convert_to_tensor=False, device="cuda")
    # Similarity and distance
    sim_matrix = cosine_similarity(embeddings)
    dist_matrix = 1 - sim_matrix
    # Clustering
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='precomputed',
        linkage='average',
        distance_threshold=1 - similarity_threshold
    )
    labels = clustering.fit_predict(dist_matrix)
    # Grouping by clusters
    cluster_map = defaultdict(list)
    for idx, label in enumerate(labels):
        cluster_map[label].append((keywords[idx], counts[idx]))
    # Choose cluster representative and sum counts
    normalized_keywords = []
    for cluster_keywords in cluster_map.values():
        # Choose the most frequent term as representative
        cluster_counter = Counter()
        for word, count in cluster_keywords:
            cluster_counter[word] += count
        representative = cluster_counter.most_common(1)[0][0]
        total_count = sum(count for _, count in cluster_keywords)
        normalized_keywords.append((representative, total_count))
    return normalized_keywords
# Apply to DataFrame
print("Normalizing abstract keywords:")
df_data["kw_abstract_normalized"] = df_data["kw_abstract"].progress_apply(normalize_keywords)

# Save intermediate df_data
print("Saved 3...")
df_data.to_pickle(df_data_name)


# Include a column with the topics of cluster 3 for each paper
# Read cluster 3 data
xls_file = "papers_relevant_cluster3.xlsx"
topics_cluster3 = pd.read_excel(xls_file, sheet_name="Topics")["topics_cluster_3"].to_list()
dict_cluster3 = {}
for topic in tqdm(topics_cluster3):
    papers_topic = pd.read_excel(xls_file, sheet_name=topic[:30])
    # Separate data with known and unknown DOI
    known_doi = papers_topic[~papers_topic['doi'].str.contains("UNKNOW", na=False)].copy()
    unknown_doi = papers_topic[papers_topic['doi'].str.contains("UNKNOW", na=False)].copy()
    # Merge using DOI for the known ones
    merged_known = known_doi.merge(
        df_data[['doi', 'citations_tot', 'year']],
        on='doi',
        how='left'
    )
    # Merge using title for the unknown ones
    merged_unknown = unknown_doi.merge(
        df_data[['title', 'citations_tot', 'year']],
        on='title',
        how='left'
    )
    # Join the two sets
    papers_topic = pd.concat([merged_known, merged_unknown], ignore_index=True) 
    dict_cluster3[topic] = papers_topic
# Initialize a new column with None
df_data["topic_name_cluster3"] = None
# Iterate over the topics dictionary
for topic_name, df_topic in tqdm(dict_cluster3.items()):
    # Ensure columns are in the correct format
    df_topic["doi"] = df_topic["doi"].fillna("UNKNOWN").astype(str).str.strip()
    df_topic["title"] = df_topic["title"].astype(str).str.strip()

    # Condition 1: Combine by DOI
    mask_doi = df_data["doi"].isin(df_topic[df_topic["doi"] != "UNKNOWN"]["doi"])
    df_data.loc[mask_doi, "topic_name_cluster3"] = topic_name
    # Condition 2: For the "UNKNOWN" DOIs, combine by title
    unknown_titles = df_topic[df_topic["doi"] == "UNKNOWN"]["title"]
    mask_title = (df_data["doi"] == "UNKNOWN") & (df_data["title"].isin(unknown_titles))
    df_data.loc[mask_title, "topic_name_cluster3"] = topic_name

# Save intermediate df_data
print("Saved 4...")
df_data.to_pickle(df_data_name)

# Create column with region for each country
country_region_map = {
    'United States': 'North America',
    'United States of America': 'North America',
    'Canada': 'North America',
    'Mexico': 'North America',
    'Brazil': 'South America',
    'Argentina': 'South America',
    'Chile': 'South America',
    'Colombia': 'South America',
    'Germany': 'Europe',
    'France': 'Europe',
    'United Kingdom': 'Europe',
    'Italy': 'Europe',
    'Spain': 'Europe',
    'Russia': 'Europe',
    'China': 'Asia',
    'Japan': 'Asia',
    'India': 'Asia',
    'South Korea': 'Asia',
    'Turkey': 'Europe',
    'Iran': 'Asia',
    'Egypt': 'Africa',
    'South Africa': 'Africa',
    'Australia': 'Oceania',
    'Georgia': 'Europe',
    'Taiwan': 'Asia',
    'Singapore': 'Asia',
    'Belgium': 'Europe',
    'Sweden': 'Europe',
    'Netherlands': 'Europe',
    'Denmark': 'Europe',
    'Russian Federation': 'Europe',
    'Ireland': 'Europe',
    'Morocco': 'Africa',
    'Switzerland': 'Europe',
    'Saudi Arabia': 'Asia',
    'Finland': 'Europe',
    'Portugal': 'Europe',
    'Qatar': 'Asia',
    'Luxembourg': 'Europe',
    'Oman': 'Asia',
    'Serbia': 'Europe',
    'Pakistan': 'Asia',
    'Malaysia': 'Asia',
    'Chad': 'Africa',
    'Greece': 'Europe',
    'Oman': 'Asia',
    'United Arab Emirates': 'Asia',
    'Bulgaria': 'Europe',
    'Niger': 'Africa',
    'Poland': 'Europe',
    'Bangladesh': 'Asia',
    'New Zealand': 'Oceania',
    'Viet Nam': 'Asia',
    'Israel': 'Asia',
    'Czechia': 'Europe',
    'Namibia': 'Africa',
    'Mali': 'Africa',
    'Lithuania': 'Europe',
    'Mauritius': 'Africa',
    'Ukraine': 'Europe',
    'Anguilla': 'North America',
    'Brunei Darussalam': 'Asia',
    'Thailand': 'Asia',
    'Croatia': 'Europe',
    'Slovenia': 'Europe',
    'Norway': 'Europe',
    'Belarus': 'Europe',
    'Bahrain': 'Asia',
    'Austria': 'Europe',
    'Indonesia': 'Asia',
    'Bhutan': 'Asia',
    'Algeria': 'Asia',
    'Slovakia': 'Europe',
    'Haiti': 'North America',
    'Estonia': 'Europe',
    'Peru': 'South America',
    'Philippines': 'Asia',
    'Ethiopia': 'Africa',
    'Jordan': 'Asia',
    'Cuba': 'North America',
    'Ghana': 'Africa',
    'Hungary': 'Europe',
    'Ecuador': 'South America',
    'Iraq': 'Asia',
    'Lebanon': 'Asia',
    'Cameroon': 'Africa',
    'Puerto Rico': 'North America',
    'Jamaica': 'North America',
    'Mongolia': 'Asia',
    'Latvia': 'Europe',
    'Tunisia': 'Africa',
    'Iceland': 'Europe',
    'Kazakhstan': 'Asia',
    'Uruguay': 'South America',
    # Add more as needed
}
def get_region(countries):
    country = countries[0] if len(countries) > 0 else ""
    if country not in country_region_map.keys():
        if country != 'UNKNOWN':
            print(country)
    return country_region_map.get(country, 'Others')
print("Separating regions:")
df_data['country_region'] = df_data['countries'].progress_apply(get_region)


# Save the df_data
print("Saved final...")
df_data.to_pickle(df_data_name)

# First author's country
print("Separating first author's country:")
df_data["first_country"] = df_data["countries"].progress_apply(
    lambda x: x[0]
)
df_data.to_pickle(df_data_name)

# First author
print("Separating first author:")
df_data["first_author"] = df_data["authors"].progress_apply(
    lambda x: x[0]
)

# Separate corresponding author and country
def extract_corresponding_author_info(affiliation_string):
    """
    Extracts the name of the corresponding author and country from an affiliation string.

    Args:
        affiliation_string (str): The string of the 'affiliation' column for a document.

    Returns:
        tuple: A tuple containing two lists:
               - corresp_authors (list): Names of the corresponding authors.
               - corresp_countries (list): Countries of the corresponding authors.
    """
    corresp_authors = []
    corresp_countries = []

    affiliation_string = affiliation_string.replace('Peoples R China', 'China').lower()  

    # Split the affiliation string into author entries using ';'
    # Use regex to handle multiple spaces after the semicolon
    author_entries = re.split(r';\s*', affiliation_string)

    for entry in author_entries:
        # Check if the entry contains the expression "(Corresponding Author)"
        if "(corresponding author)" in entry:
            # Split the entry at the point where "(Corresponding Author)" appears
            parts = entry.split("(corresponding author)", 1)
            
            # The first part is the author's name (before the expression)
            author_name = parts[0].strip()
            if author_name: # Ensure that the name is not empty
                corresp_authors.append(author_name)

            # The second part contains the institution's affiliation
            if len(parts) > 1:
                affiliation_info = parts[1].strip()
                
                # To extract the country:
                # The country is the last piece of information in the affiliation.
                # We'll try to get the last part after the comma.
                country = None
                # Split the affiliation information by comma and take the last part
                affiliation_parts = [p.strip() for p in affiliation_info.split(',')]
                
                # Remove empty parts and take the last one
                affiliation_parts = [p for p in affiliation_parts if p]

                if affiliation_parts:
                    country = affiliation_parts[-1] # The last part is the country
                    corresp_countries.append(country)
                else:
                    # If it cannot extract the country, add None or an empty string
                    corresp_countries.append(None) # Or ""
            else:
                corresp_countries.append(None) # Or ""

    return corresp_authors, corresp_countries
df_data[['corresp_author', 'corresp_country']] = df_data['affiliation_'].progress_apply(
    lambda x: pd.Series(extract_corresponding_author_info(x))
)


# Create column with reduced topic names and abbreviations
repls = {
    'oxygen evolution reaction': 'OER',
    'hydrogen evolution reaction': 'HER',
    'electrocatalyst': 'EC',
    'electrocatalysts': 'ECs',
    'hydrogen and oxygen evolution': 'HER and OER',
    'Ruthenium-based': 'Ru-based',
    'Nickel-based': 'Ni-based',
    'iridium-based': 'Ir-based',
    'Co3o4': 'Co₃O₄',
    'ni3s2': 'Ni₃S₂',
    'co9s8': 'Co₉S₈',
    'mos2': 'MoS2',
    'Ni-fe': 'Ni-Fe',
    '(her)': '',
}
def truncate_text(text, max_chars=50):
    text = text.split("_")[1]
    text = text.lower().capitalize()
    for repl in repls:
        text = text.replace(repl, repls[repl])
    if len(text) <= max_chars:
        return text
    # Find the last space before or right after max_chars
    end = text[:max_chars+20].rfind(" ")
    text = text[:end] if end > 0 else text
    return text+"..."

df_data["topic_name_cluster3_reduced"] = df_data["topic_name_cluster3"].progress_apply(truncate_text, max_chars=30)

revistas = {
"international journal of hydrogen energy": "Int. J. Hydrogen Energy",
"journal of materials chemistry a": "J. Mater. Chem. A",
"electrochimica acta": "Electrochim. Acta", 
"acs applied materials \\& interfaces": "ACS Appl. Mater. Interfaces",
"journal of colloid and interface science": "J. Colloid Interface Sci.",
"chemical engineering journal": "Chem. Eng. J.",
"journal of alloys and compounds": "J. Alloys Compd.",
"applied surface science": "Appl. Surf. Sci.",
"acs applied energy materials": "ACS Appl. Energy Mater.",
}

df_data["journal_abrev"] = df_data["journal"].progress_apply(lambda x: revistas[x.lower()] if x.lower() in revistas.keys() else x)


# Separate and count references and citations
# Function to extract DOIs from a reference string
def extract_dois(ref_string):
    if pd.isna(ref_string):
        return []
    
    dois = []
    references = ref_string.split(';')
    
    for ref in references:
        match = re.search(r'DOI\s*(\[[^\]]+\]|10\.\d{4,9}/[^\s\];]+)', ref, flags=re.IGNORECASE)
        if match:
            doi_raw = match.group(1).strip()
            if doi_raw.startswith('['):
                # DOIs inside brackets, separated by comma
                doi_list = re.findall(r'10\.\d{4,9}/[^\s,\];]+', doi_raw)
                if doi_list:
                    doi = doi_list[-1].rstrip('.,;')
                    dois.append(doi.lower())
            else:
                doi = doi_raw.rstrip('.,;')
                dois.append(doi.lower())
    return dois

# Apply function to extract DOIs
df_data['references_doi'] = df_data['references'].progress_apply(extract_dois)

# Standardize 'doi' column (remove spaces and make lowercase)
df_data['doi_clean'] = df_data['doi'].str.strip().str.lower()

# Set of DOIs from the dataframe itself
doi_set = set(df_data['doi_clean'].dropna())

# Calculate internal and external references
df_data['references_int'] = df_data['references_doi'].progress_apply(
    lambda x: sum(doi in doi_set for doi in x)
)
df_data['references_ext'] = df_data['references_doi'].progress_apply(
    lambda x: sum(doi not in doi_set for doi in x)
)


# Count how many times each DOI appears in the references of the entire dataframe
all_references = list(chain.from_iterable(df_data['references_doi']))
doi_counter = Counter(all_references)

# Map to each line how many times the DOI itself appears in the references of the others
df_data['citacoes_int'] = df_data['doi_clean'].map(lambda x: doi_counter.get(x, 0))


# Include the impact factor of each journal
from impact_factor.core import Factor
fa = Factor()
def obtem_fi(row):
    issn, eissn, journal = row['issn'], row['eissn'], row['journal']

    if eissn != 'UNKNOWN':
        chave = eissn
    elif issn != 'UNKNOWN':
        chave = issn   
    elif journal != 'UNKNOWN':
        chave = journal.lower()
    else:
        return None 
    
    ifs = fa.search(chave)

    if len(ifs) == 0:
        return None
    
    return ifs[0]["factor"]


# Apply function to extract DOIs
df_data['journal_if'] = df_data.progress_apply(obtem_fi, axis=1)

print(f"qtd_none = {df_data['journal_if'].isna().sum()}")


# Include the impact factor of each document
df_data["age_years"] = 2025 - df_data["year"] + 0.25  # 0.25 to include the first quarter
df_data["citations_per_year"] = df_data["citations_tot"] / df_data["age_years"]

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_data[["citations_norm", "if_norm"]] = scaler.fit_transform(
    df_data[["citations_per_year", "journal_if"]]
)
df_data["document_relevance_score"] = (
    0.6 * df_data["citations_norm"] +
    0.4 * df_data["if_norm"]
)

# Validate possible discrepancy between authors and countries
if len(df_data["authors"]) != len(df_data["countries"]):
    print("Error: The number of authors does not correspond to the number of countries.")
    a = 1/0
    
# Last author
print("Separating last author:")
df_data["last_author"] = df_data["authors"].progress_apply(
    lambda x: x[-1]
)

# Country of the last author
print("Separating last author's country:")
df_data["last_country"] = df_data["countries"].progress_apply(
    lambda x: x[-1]
)

# Create author:institution dict
def extract_institutions(row):
    # Step 1: Split the 'affiliation_' column by ";"
    raw_list = [item.strip() for item in str(row['affiliation_']).split(';')]
    
    # Step 2: Remove items with "(Corresponding Author)"
    list_inst = [item for item in raw_list if "(Corresponding Author)" not in item]
    
    # Step 3: Create dict with key = value
    inst_dict = {item: item for item in list_inst}
    
    # Step 4: Remove authors' names from the dict's value
    authors = row['authors']
    for k in inst_dict:
        for author in authors:
            pattern = re.compile(re.escape(author), re.IGNORECASE)
            if pattern.search(inst_dict[k]):
                inst_dict[k] = pattern.sub('', inst_dict[k]).strip(",; ").strip()

    # Step 5: Assign each author to the corresponding institution's value
    institutions = {}
    for author in authors:
        institutions[author] = ""
        for k, v in inst_dict.items():
            if author.lower() in k.lower():
                institutions[author] = v
    
    return institutions

# Apply to DataFrame
df_data['institutions'] = df_data.progress_apply(extract_institutions, axis=1)

# Last author's institution
def get_last_institution(row):
    institutions_dict = row['institutions']
    last_author = row['last_author']
    if isinstance(institutions_dict, dict):
        for key in institutions_dict:
            if key.strip().lower() == last_author.strip().lower():
                last_inst = institutions_dict[key].split(",")[0].strip()
                if last_inst == "CSIR":
                    last_inst = "CSIR Cent Electrochem Res Inst CECRI"
                return last_inst
    return None

        
# Apply to dataframe
df_data['last_institution'] = df_data.progress_apply(get_last_institution, axis=1)




# ###############
# Save the df_data
df_data["kw_abstract"] = df_data["kw_abstract"].apply(
    lambda x: json.dumps(x) if isinstance(x, list) else None
)
df_data["kw_abstract_normalized"] = df_data["kw_abstract_normalized"].apply(
    lambda x: json.dumps(x) if isinstance(x, list) else None
)
df_data_name = 'df_data.parquet'
df_data.to_parquet(df_data_name)

## Group 1 Questions
Literature Trends in Electrochemical H2 Production

### Topic name conversion table

In [ ]:
# Execute data processing or visualization steps
df_temp = df_data.drop_duplicates(subset=["topic_name_cluster3_reduced", "topic_name_cluster3"])
df_temp["topic_name_cluster3"] = df_temp["topic_name_cluster3"].apply(lambda x: x.split("_")[1])
df_temp = df_temp[["topic_name_cluster3_reduced", "topic_name_cluster3"]]
df_temp = df_temp.sort_values(by="topic_name_cluster3_reduced")

df_temp.to_latex('table_conversion_names_topics.tex',
    index=False, 
    column_format="l l",
)

### Question S-2.1
What are the topics most frequently discussed in H2 production research?


In [ ]:
# Execute data processing or visualization steps
# ###################################
# Topic relevance index

# Read the data of cluster 3
xls_file = "papers_relevant_cluster3.xlsx"
topics_cluster3 = pd.read_excel(xls_file, sheet_name="Topics")["topics_cluster_3"].to_list()

full_data = bibfile.data.copy()
full_data["citations"] = bibfile.citation
full_data['year'] = full_data['year'].astype('int')

dict_cluster3 = {}
for topic in tqdm(topics_cluster3):
    papers_topic = pd.read_excel(xls_file, sheet_name=topic[:30])

    # Separate data with known and unknown DOI
    known_doi = papers_topic[~papers_topic['doi'].str.contains("UNKNOW", na=False)].copy()
    unknown_doi = papers_topic[papers_topic['doi'].str.contains("UNKNOW", na=False)].copy()

    # Merge using DOI for the known ones
    merged_known = known_doi.merge(
        full_data[['doi', 'citations', 'year', 'document_type']],
        on='doi',
        how='left'
    )

    # Merge using title for the unknown ones
    merged_unknown = unknown_doi.merge(
        full_data[['title', 'citations', 'year', 'document_type']],
        on='title',
        how='left'
    )

    # Join the two sets
    papers_topic = pd.concat([merged_known, merged_unknown], ignore_index=True) 
     
    dict_cluster3[topic] = papers_topic

# #########################################
# Calculate the relevance of the topics: average of the citations of the papers of the topic

# Simulated data example
# dict_cluster3 = {
# "Topic A": pd.DataFrame({"doi": [...], "year": [...], "note": [...]}),
# "Topic B": ...
# }

# Current year (adjustable as needed)
current_year = 2024

# First, we aggregate global statistics by year for z-score calculation. Remove the last incomplete year so it doesn't affect the calculation
all_docs = pd.concat(dict_cluster3.values(), ignore_index=True)
all_docs["document_type"] = all_docs["document_type"].fillna("")
all_docs_wo_review = all_docs[~all_docs['document_type'].str.contains("Review")]
# Identify the last year (ex: 2025)
ultimo_ano = all_docs["year"].max()
# Filter the DataFrame, excluding the last year
docs_filtrados = all_docs[all_docs["year"] < ultimo_ano]
docs_filtrados_wo_review = all_docs_wo_review[all_docs_wo_review["year"] < ultimo_ano]
# Calculate average and standard deviation by year (without the last year)
year_stats = (
    docs_filtrados.groupby("year")["citations"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "year_mean", "std": "year_std"})
    .reset_index()
)
year_stats_wo_review = (
    docs_filtrados_wo_review.groupby("year")["citations"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "year_mean_wo_review", "std": "year_std_wo_review"})
    .reset_index()
)


# Function to calculate the relevance index of a topic
def compute_topic_index(topic_df):

    # #####################################################################
    # Original Relevance Index
    df = topic_df.copy()
    df = df[df["citations"].notna() & df["year"].notna()]
    
    # Calculate citations by year
    df["citations_per_year"] = df["citations"] / (current_year - df["year"] + 1 + 0.25)  # 0.25 to account for only the first 3 months of 2025

    # Merge with year statistics
    df = df.merge(year_stats, on="year", how="left")
    
    # Z-score by year
    df["z_score"] = (df["citations"] - df["year_mean"]) / df["year_std"]
    df["z_score"] = df["z_score"].fillna(0)

    # Identify top 10% by year
    top_percentile_flags = []
    for year in df["year"].unique():
        year_notes = all_docs[all_docs["year"] == year]["citations"]
        if len(year_notes) >= 10:
            threshold = np.percentile(year_notes, 90)
            top_percentile_flags.extend(df[df["year"] == year]["citations"] >= threshold)
        else:
            top_percentile_flags.extend([False] * sum(df["year"] == year))
    df["top_10_percent"] = top_percentile_flags[:len(df)]

    # Calculation of the composite index
    mean_cpy = df["citations_per_year"].mean()
    mean_z = df["z_score"].mean()
    prop_top10 = df["top_10_percent"].mean()
    n_docs = len(df)

    # Composite index (weighted) *** Original ****
    index = (
        0.5 * mean_cpy +
        0.3 * prop_top10 +
        0.2 * np.log1p(n_docs)
    )

    # #####################################################################
    # Relevance Index without review
    df = topic_df.copy()
    df['document_type'] = df['document_type'].fillna("")
    df = df[~df['document_type'].str.contains("Review")]
    df = df[df["citations"].notna() & df["year"].notna()]
    
    # Calculate citations by year
    df["citations_per_year_wo_review"] = df["citations"] / (current_year - df["year"] + 1 + 0.25)  # 0.25 to account for only the first 3 months of 2025

    # Merge with year statistics
    df = df.merge(year_stats_wo_review, on="year", how="left")
    
    # Z-score by year
    df["z_score_wo_review"] = (df["citations"] - df["year_mean_wo_review"]) / df["year_std_wo_review"]
    df["z_score_wo_review"] = df["z_score_wo_review"].fillna(0)

    # Identify top 10% by year
    top_percentile_flags_wo_review = []
    for year in df["year"].unique():
        year_notes_wo_review = all_docs_wo_review[all_docs_wo_review["year"] == year]["citations"]
        if len(year_notes_wo_review) >= 10:
            threshold_wo_review = np.percentile(year_notes_wo_review, 90)
            top_percentile_flags_wo_review.extend(df[df["year"] == year]["citations"] >= threshold_wo_review)
        else:
            top_percentile_flags_wo_review.extend([False] * sum(df["year"] == year))
    df["top_10_percent_wo_review"] = top_percentile_flags_wo_review[:len(df)]

    # Calculation of the composite index
    mean_cpy_wo_review = df["citations_per_year_wo_review"].mean()
    mean_z_wo_review = df["z_score_wo_review"].mean()
    prop_top10_wo_review = df["top_10_percent_wo_review"].mean()
    n_docs_wo_review = len(df)


    # Composite index (weighted) *** without Review ****
    index_wo_review = (
        0.5 * mean_cpy_wo_review +
        0.3 * prop_top10_wo_review +
        0.2 * np.log1p(n_docs_wo_review)
    )



    return {
        "n_documents": n_docs,
        "mean_citations_per_year": mean_cpy,
        "mean_z_score": mean_z,
        "prop_top_10_percent": prop_top10,
        "composite_index": index,

        "n_documents_wo_review": n_docs_wo_review,
        "mean_citations_per_year_wo_review": mean_cpy_wo_review,
        "mean_z_score_wo_review": mean_z_wo_review,
        "prop_top_10_percent_wo_review": prop_top10_wo_review,
        "composite_index_wo_review": index_wo_review,
    }

# Apply to all the topics
topic_indices = {}
for topic, df in dict_cluster3.items():
    topic_indices[topic] = compute_topic_index(df)

# Convert to DataFrame
topic_index_df = pd.DataFrame.from_dict(topic_indices, orient="index").reset_index()
topic_index_df = topic_index_df.rename(columns={"index": "topic"})
topic_index_df = topic_index_df.sort_values(by="composite_index", ascending=False)

topic_index_df["topic"] = topic_index_df["topic"].apply(lambda x: df_data.loc[df_data["topic_name_cluster3"] == x, "topic_name_cluster3_reduced"].values[0] if x in df_data["topic_name_cluster3"].values else x)
# Fix the MoS2 formula so the 2 is subscripted
topic_index_df["topic"] = topic_index_df["topic"].progress_apply(lambda x: x.replace("MoS2", "MoS₂"))

def ordinal(n):
    """
    Converts an integer into an ordinal string, like 1st, 2nd, 3rd, etc.
    """
    if 10 <= n % 100 <= 20:
        suffix = 'th'
    else:
        suffix = {1:'st', 2:'nd', 3:'rd'}.get(n % 10, 'th')
    return f"{n}{suffix}"

# Generate the ranks
topic_index_df["rank_index"] = (
    topic_index_df["composite_index"]
    .rank(ascending=False, method="min")
    .astype(int)
    .map(ordinal)
)

topic_index_df["rank_index_wo_review"] = (
    topic_index_df["composite_index_wo_review"]
    .rank(ascending=False, method="min")
    .astype(int)
    .map(ordinal)
)

topic_index_name = 'topic_index.parquet'
topic_index_df.to_parquet(topic_index_name)

topic_index_df

In [ ]:
# Execute data processing or visualization steps
# Defining columns
col_direita = "composite_index_wo_review"
col_esquerda = "composite_index"
rank_direita = "rank_index_wo_review"
rank_esquerda = "rank_index"


# Sort data by the left column
df_plot = topic_index_df.sort_values(by=col_direita, ascending=False).reset_index(drop=True)

# Generate gradient color palette
n = len(df_plot)
palette_direita = sns.color_palette("viridis", n)
palette_esquerda = sns.color_palette("plasma", n)

# Defining font sizes
fs1 = 50  # Title
fs2 = 30  # Axes and labels
fs3 = 16  # (not used)

# Create figure
fig, ax = plt.subplots(figsize=(18, 24))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Plotting left bars (with reviews)
sns.barplot(
    data=df_plot,
    x=-df_plot[col_esquerda],
    y="topic",
    palette=palette_direita,
    orient='h',
    ax=ax
)

# Plotting right bars (without reviews)
sns.barplot(
    data=df_plot,
    x=df_plot[col_direita],
    y="topic",
    palette=palette_esquerda,
    orient='h',
    ax=ax
)

# Adding the values in the bars
for i, (val_esq, val_dir, rank_esq, rank_dir) in enumerate(zip(df_plot[col_esquerda], df_plot[col_direita], df_plot[rank_esquerda], df_plot[rank_direita])):
    # Left
    ax.text(
        -val_esq - 0.005, i, 
        f" ({rank_esq}) {val_esq:.1f}",
        va='center', ha='right',
        fontsize=fs2, color='black'
    )
    # Right
    ax.text(
        val_dir + 0.005, i, 
        f"{val_dir:.1f} ({rank_dir})",
        va='center', ha='left',
        fontsize=fs2, color='black'
    )

# Adjustment of the Y axis labels (topics)
y_lbls = ax.get_yticklabels()
lbls = ["\n".join(textwrap.wrap(label.get_text(), 80)) for label in y_lbls]
ax.set_yticklabels(lbls)

# Central line in the X axis (zero)
ax.axvline(0, color='black', linewidth=1.5)

ax.set_xlim(-35, 30)

# Adjustment of the X axis ticks (visually removing the negative sign)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([abs(int(x)) if x != 0 else 0 for x in xticks], fontsize=fs2, rotation=90)

# Axes configurations
plt.xlabel("Relevance index", fontsize=fs2)
plt.ylabel("")
plt.yticks(fontsize=fs2)

# Title
plt.title("Topics Ranked by Relevance Index\n", fontsize=fs1, loc='right')

# Adding superior labels
# Take the current axis limits
x_min, x_max = ax.get_xlim()
y = -0.5

# Text to the left
ax.text(
    x_min * 0.4,  # x position (adjustable as needed)
    y,     # slightly above the bars
    "With Review",
    fontsize=fs2,
    ha='center',
    va='bottom'
)

# Text to the right
ax.text(
    x_max * 0.65,  # x position
    y,
    "Without Review",
    fontsize=fs2,
    ha='center',
    va='bottom'
)

# Remove the legend
ax.legend().remove()

plt.tight_layout()
plt.savefig('q3_1_1_topics_ranked_by_relevance_index_violin.pdf', bbox_inches='tight')

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Sort the data by relevance
df_plot = topic_index_df.sort_values(by="composite_index", ascending=False)

fs1 = 50
fs2 = 30
fs3 = 16

fig, ax = plt.subplots(figsize=(18, 24))
fig.patch.set_facecolor('white')   # Figure background
ax.set_facecolor('white')          # Chart area background

sns.barplot(data=df_plot, x="composite_index", y="topic", palette="viridis")

# Add the values in front of each bar
for i, (value, label) in enumerate(zip(df_plot["composite_index"], df_plot["topic"])):
    ax.text(
        value + 0.005,         # x position: slightly ahead of the bar
        i,                     # y position: bar index
        f"{value:.1f}",        # formatted text with 2 decimal places
        va='center',           # vertical alignment
        ha='left',             # horizontal alignment
        fontsize=fs2,
        color='black'
    )

y_lbls = ax.get_yticklabels()
# y_lbls = [replaces(label.get_text()) for label in y_lbls]
lbls = ["\n".join(textwrap.wrap(label.get_text(), 80)) for label in y_lbls]
ax.set_yticklabels(lbls)

plt.xlabel("Relevance index", fontsize=fs2)
plt.ylabel("")
plt.yticks(fontsize=fs2)
plt.xticks(fontsize=fs2, rotation=90)
plt.title("Topics Ranked by Relevance Index", fontsize=fs1, loc='right')
plt.tight_layout()


plt.savefig('q3_1_1/q3_1_1_topics_ranked_by_relevance_index.pdf', bbox_inches='tight') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
df_plot = topic_index_df.sort_values(by="composite_index", ascending=False)
df_plot[["topic", "composite_index"]].to_latex('q3_1_1/q3_1_1_topics_ranked_by_relevance_index.tex',
    index=False, 
    float_format="%.2f",
    column_format="l c",
)

df_plot = topic_index_df.sort_values(by="composite_index_wo_review", ascending=False)
df_plot[["topic", "composite_index_wo_review"]].to_latex('q3_1_1/q3_1_1_topics_ranked_by_relevance_index_wo_review.tex',
    index=False, 
    float_format="%.2f",
    column_format="l c",
)

In [ ]:
# Execute data processing or visualization steps
# Sort the data by relevance
df_plot = topic_index_df.sort_values(by="composite_index_wo_review", ascending=False)

fs1 = 50
fs2 = 30
fs3 = 16

fig, ax = plt.subplots(figsize=(18, 24))
fig.patch.set_facecolor('white')   # Figure background
ax.set_facecolor('white')          # Chart area background

sns.barplot(data=df_plot, x="composite_index_wo_review", y="topic", palette="viridis")

# Add the values in front of each bar
for i, (value, label) in enumerate(zip(df_plot["composite_index_wo_review"], df_plot["topic"])):
    ax.text(
        value + 0.005,         # x position: slightly ahead of the bar
        i,                     # y position: bar index
        f"{value:.1f}",        # formatted text with 2 decimal places
        va='center',           # vertical alignment
        ha='left',             # horizontal alignment
        fontsize=fs2,
        color='black'
    )

y_lbls = ax.get_yticklabels()
# y_lbls = [replaces(label.get_text()) for label in y_lbls]
lbls = ["\n".join(textwrap.wrap(label.get_text(), 80)) for label in y_lbls]
ax.set_yticklabels(lbls)

plt.xlabel("Relevance index", fontsize=fs2)
plt.ylabel("")
plt.yticks(fontsize=fs2)
plt.xticks(fontsize=fs2, rotation=90)
plt.title("Topics Ranked by Relevance Index\n(without 'Review' citations)", fontsize=fs1, loc='right')
plt.tight_layout()


plt.savefig('q3_1_1/q3_1_1_topics_ranked_by_relevance_index_wo_review.pdf', bbox_inches='tight') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Build a dataframe with the quantity of papers published in each topic by year
# List to store the data
records = []

# Iterate over the topics and count documents by year
for topic, df in dict_cluster3.items():
    year_counts = df["year"].value_counts().to_dict()
    for year, count in year_counts.items():
        records.append({"topic": topic, "year": int(year), "document_count": count})

# Create DataFrame
topic_year_df = pd.DataFrame(records)

# Adjust the topic name (take the second part of the string)
# topic_year_df["topic"] = topic_year_df["topic"].str.split("_").str[1]
topic_year_df["topic"] = topic_year_df["topic"].apply(lambda x: df_data.loc[df_data["topic_name_cluster3"] == x, "topic_name_cluster3_reduced"].values[0] if x in df_data["topic_name_cluster3"].values else x)

# Filter only years from 2014 onwards
topic_year_df = topic_year_df[topic_year_df["year"] >= 2014]

# Calculate the total publications by topic
topic_totals = topic_year_df.groupby("topic")["document_count"].sum().reset_index()

# Sort the topics by total publications in descending order
ordered_topics = topic_totals.sort_values(by="document_count", ascending=True)["topic"].tolist()

# Apply sorting in the main DataFrame
topic_year_df["topic"] = pd.Categorical(topic_year_df["topic"], categories=ordered_topics, ordered=True)
topic_year_df = topic_year_df.sort_values(by=["topic", "year"]).reset_index(drop=True)

# Shortened topic name
topic_year_df["topic_short"] = topic_year_df["topic"]


# Create the bubble plot
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor('white')   # Figure background
ax.set_facecolor('white')          # Chart area background

# 4. Normalize sizes for visual bubbles
size_factor = 0.5  # Visual adjustment


highlight_year = 2025
plt.axvspan(highlight_year - 0.5, highlight_year + 0.5, color='lightgray', alpha=0.3, zorder=0)

# 5. Plot each point
for _, row in topic_year_df.iterrows():
    plt.scatter(
        row['year'],
        row['topic_short'],
        s=row['document_count'] * size_factor,
        c = np.log1p(row['document_count']),
        cmap='Pastel1', 
        alpha=0.6,
        # edgecolors='black'
    )

    plt.text(
        row['year'],
        row['topic_short'],
        f"{row['document_count'] if row['document_count'] > 200 else ''}",
        fontsize=11,
        ha='center',
        va='center',
        color = "gray" if row["year"] == 2025 else "black",
    )

# 6. Visual adjustments
plt.xlabel("", fontsize=16)
plt.ylabel("", fontsize=16)
plt.title("Frequency of Documents in Topics by Year (Last 11 Year)", fontsize=20)
plt.grid(True, axis='x', linestyle='--', alpha=0.5)

lbls = topic_year_df["year"]
plt.xticks(rotation=45, ticks = lbls, labels=lbls, fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()

plt.savefig('q3_1_1/q3_1_1_topics_doc_frequency_by_year_bubble.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Table of the plot above
df_pivotado = topic_year_df.pivot_table(
    index='topic',  # rows: keywords
    columns='year',    # columns: years
    values='document_count',   # values: frequency
    aggfunc='sum',    # sum if there are multiple records
    fill_value=0      # fill missing values with 0
)
# Add a "Total" column with the sum per row
df_pivotado['Total'] = df_pivotado.sum(axis=1)

# Sort by the "Total" column in descending order
df_pivotado = df_pivotado.sort_values(by='Total', ascending=False)

df_pivotado.to_latex('q3_1_1/topics_doc_frequency_by_year_bubble.tex', index=True, header=True)

df_pivotado

In [ ]:
# Execute data processing or visualization steps
# Same figure as before, but with a reduced period and number of topics

# Build a dataframe with the quantity of papers published in each topic by year

fs1 = 32
fs2 = 24
fs3 = 16

# List to store the data
records = []

# Iterate over the topics and count documents by year
for topic, df in dict_cluster3.items():
    year_counts = df["year"].value_counts().to_dict()
    for year, count in year_counts.items():
        records.append({"topic": topic, "year": int(year), "document_count": count})

# Create DataFrame
topic_year_df = pd.DataFrame(records)

# Adjust the name of the topic (take the second part of the string)
# topic_year_df["topic"] = topic_year_df["topic"].str.split("_").str[1]
topic_year_df["topic"] = topic_year_df["topic"].apply(lambda x: df_data.loc[df_data["topic_name_cluster3"] == x, "topic_name_cluster3_reduced"].values[0] if x in df_data["topic_name_cluster3"].values else x)

# Filter only years greater than 2019
topic_year_df = topic_year_df[topic_year_df["year"] > 2019]

# Calculate the total of publications by topic (after the year filter)
topic_totals = topic_year_df.groupby("topic")["document_count"].sum().reset_index()

# Selects the top 10 topics
top_topics = topic_totals.sort_values(by="document_count", ascending=False).head(10)["topic"].tolist()

# Filter the DataFrame to contain only the top 10 topics
topic_year_df = topic_year_df[topic_year_df["topic"].isin(top_topics)]

# Sort the topics by total of publications in descending order
ordered_topics = topic_year_df.groupby("topic")["document_count"].sum().sort_values(ascending=True).index.tolist()

# Apply sorting in the main DataFrame
topic_year_df["topic"] = pd.Categorical(topic_year_df["topic"], categories=ordered_topics, ordered=True)
topic_year_df = topic_year_df.sort_values(by=["topic", "year"]).reset_index(drop=True)

# Shortened topic name
topic_year_df["topic_short"] = topic_year_df["topic"].str[:60]+"..."

# Create the bubble plot
fig, ax = plt.subplots(figsize=(14, 12))
fig.patch.set_facecolor('white')   # Figure background
ax.set_facecolor('white')          # Chart area background

# Normalize the sizes for visual bubbles
size_factor = 1.0  # Visual adjustment

highlight_year = 2025
plt.axvspan(highlight_year - 0.5, highlight_year + 0.5, color='lightgray', alpha=0.3, zorder=0)

# Plot each point
for _, row in topic_year_df.iterrows():
    plt.scatter(
        row['year'],
        row['topic'],  #['topic_short'],
        s=row['document_count'] * size_factor,
        c = np.log1p(row['document_count']),
        cmap='Pastel1', 
        alpha=0.6,
    )

    plt.text(
        row['year'],
        row['topic'],
        f"{row['document_count'] if row['document_count'] > 200 else ''}",
        fontsize=fs3,
        ha='center',
        va='center',
        color = "gray" if row["year"] == 2025 else "black",
    )

# Replace spaces by line breaks in the labels of the Y axis
repls = {
    'oxygen evolution reaction': 'OER',
    'hydrogen evolution reaction': 'HER',
    'electrocatalyst': 'EC',
    'electrocatalysts': 'ECs',
    'hydrogen and oxygen evolution': 'HER and OER',
    'Ruthenium-based': 'Ru-based',
    'Nickel-based': 'Ni-based',
    'iridium-based': 'Ir-based',
    'Co3o4': 'Co₃O₄',
    'ni3s2': 'Ni₃S₂',
    'co9s8': 'Co₉S₈',
    'mos2': 'MoS2',
    'Ni-fe': 'Ni-Fe',
    '(her)': '',
}
def replaces(text):
    # Remove extra spaces and replace by line breaks
    text = text.lower().capitalize()
    for repl in repls:
        text = text.replace(repl, repls[repl])
    return text
y_lbls = ax.get_yticklabels()
y_lbls = [replaces(label.get_text()) for label in y_lbls]
lbls = ["\n".join(textwrap.wrap(label, 60)) for label in y_lbls]
ax.set_yticklabels(lbls)

# Visual adjustments
plt.xlabel("", fontsize=fs2)
plt.ylabel("", fontsize=fs2)
plt.title("Frequency of Documents in Top 10 Topics by Year (last 6 year)", fontsize=fs1, loc='right')
plt.grid(True, axis='x', linestyle='--', alpha=0.5)

lbls = sorted(topic_year_df["year"].unique())
plt.xticks(rotation=45, ticks=lbls, labels=lbls, fontsize=fs2)
plt.yticks(fontsize=fs2)
plt.tight_layout()


plt.savefig('q3_1_1/q3_1_1_topics_doc_frequency_by_year_bubble_top10.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# #### here
from matplotlib.gridspec import GridSpec
from sklearn.linear_model import LinearRegression

# ======================
# Settings
# ======================
# Font sizes
fs1 = 30  # chart title
fs2 = 18  # Axes
fs3 = 14  # legend

# ======================
# Data preparation
# ======================
# Filter range of years
df_filtered = df_data[df_data['year'].between(2014, 2024)]

# Create a dictionary: {topic_name: ranking}
rank_dict = dict(
    topic_index_df[['topic', 'rank_index_wo_review']].values
)
rank_dict['Enhanced ECs for hydrogen evolution using MoS2...'] = rank_dict.pop('Enhanced ECs for hydrogen evolution using MoS₂...', None)  # Correct the topic name]


# Count publications by year and topic
yearly_counts = df_filtered.groupby(['year', 'topic_name_cluster3_reduced']).size().reset_index(name='count')
pivot_table = yearly_counts.pivot(index='year', columns='topic_name_cluster3_reduced', values='count').fillna(0)

# ======================
# Classify trends
# ======================
linear_topics = []
flat_topics = []
exp_topics = []

years = pivot_table.index.values.reshape(-1, 1)

for topic in pivot_table.columns:
    counts = pivot_table[topic].values

    if np.count_nonzero(counts) < 5:
        continue  # skip topics with too little data

    # Linear adjustment
    lin_model = LinearRegression()
    lin_model.fit(years, counts)
    pred_linear = lin_model.predict(years)

    # R² of the linear adjustment
    ss_res = np.sum((counts - pred_linear) ** 2)
    ss_tot = np.sum((counts - np.mean(counts)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    # Checks if it looks exponential
    with np.errstate(divide='ignore'):
        log_counts = np.log1p(counts)  # log(1 + x) to avoid log(0)

    exp_model = LinearRegression()
    exp_model.fit(years, log_counts)
    pred_exp = exp_model.predict(years)

    exp_r2 = 1 - np.sum((log_counts - pred_exp) ** 2) / np.sum((log_counts - np.mean(log_counts)) ** 2)

    slope = lin_model.coef_[0]

    if r2 > 0.9 and slope > 2:
        linear_topics.append(topic)
    elif exp_r2 > 0.9 and exp_model.coef_[0] > 0.08:
        exp_topics.append(topic)
    else:
        flat_topics.append(topic)


# ======================
# Function to plot
# ======================
def plot_group(ax, topics, letter, title):
    from itertools import cycle
    import matplotlib.pyplot as plt

    # Color palette and markers
    color_cycle = plt.cm.get_cmap('tab20', len(topics)).colors
    marker_styles = ['o', 's', 'v', '^', '<', '>', 'D', 'P', '*', 'X', 'H']
    color_iterator = cycle(color_cycle)
    marker_iterator = cycle(marker_styles)

    for topic in topics:
        # Obtain ranking of the topic, if not found leave empty
        rank = rank_dict.get(topic, '')
        rank_label = f"({rank}) " if rank else ""
        
        
        # Line break in the name of the topic
        label_wrapped = ' '.join(topic.split(maxsplit=1))

        # Final label with ranking
        label_final = f"{rank_label}{label_wrapped}"

        # Color and marker for each line
        color = next(color_iterator)
        marker = next(marker_iterator)

        ax.plot(
            pivot_table.index,
            pivot_table[topic],
            label=label_final,
            color=color,
            marker=marker,
            linewidth=2,
            markersize=8
        )

    ax.set_title(title, fontsize=fs2, pad=15)

    # Letter in the top left corner
    ax.text(
        0, 1.31, letter,
        transform=ax.transAxes,
        fontsize=fs2+1,
        # fontweight='bold',
        va='top',
        ha='left'
    )

    ax.set_xlabel("", fontsize=fs2)
    ax.set_ylabel("No of Publications", fontsize=fs2)
    ax.tick_params(labelsize=fs2)
    ax.set_xticks(pivot_table.index[::max(len(pivot_table.index)//4,1)])

    # Tilting the labels of the X axis
    ax.tick_params(axis='x', labelrotation=45)

    # Remove grid lines
    ax.grid(False)

    # Black border of the plot area
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.5)

    ax.set_facecolor('white')

    # Legend with white background and reduced width
    leg = ax.legend(
        fontsize=fs3,
        loc='center left',
        bbox_to_anchor=(1.03, 0.5),
        frameon=True,
        fancybox=False,
        framealpha=1,
        borderpad=0.5,
        handlelength=1.5,
        handletextpad=0.5,
        borderaxespad=0.5
    )
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_linewidth(0)


# ======================
# Generate the plots
# ======================
fig = plt.figure(figsize=(10, 15), facecolor='white')
gs = GridSpec(3, 1)

# plt.suptitle("Growth in the Number of Publications by Type", fontsize=fs1, y=1.0)


ax1 = fig.add_subplot(gs[0])
plot_group(ax1, linear_topics, "(a)", "Linear Growth")

ax2 = fig.add_subplot(gs[1])
plot_group(ax2, flat_topics, "(b)", "Plateau")

ax3 = fig.add_subplot(gs[2])
plot_group(ax3, exp_topics, "(c)", "Exponential Growth")
# If you wish to activate logarithmic scale in the last plot:
# ax3.set_yscale('log')

# Separating lines
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
for p in [0.63, 0.33]:
    fig.lines.append(
        plt.Line2D(
            [0.1, 0.9], [p, p],
            transform=fig.transFigure,
            color='black',
            linewidth=1.5
        )
    )


plt.savefig('q3_1_1/q3_1_1_publications_por_ano_por_tipo.pdf', bbox_inches='tight') #, dpi=300)


plt.show()

In [ ]:
# Execute data processing or visualization steps
from io import StringIO

# Function to generate a table of a group
def build_group_table(topics, group_label):
    if not topics:
        return pd.DataFrame()

    df = pivot_table[topics].T  # transpose: topics as rows

    # Add the rank in the name of the topic
    new_index = []
    for topic in df.index:
        rank = rank_dict.get(topic, '')
        rank_label = f"({rank}) " if rank else ""
        new_index.append(f"{rank_label}{topic}")

    df.index = new_index
    df.index.name = 'topic'

    # Add group column
    df.insert(0, 'Group', group_label)

    return df.reset_index()

rank_dict = dict(topic_index_df[['topic', 'rank_index_wo_review']].values)


# Build the tables
df_linear = build_group_table(linear_topics, 'Linear')
df_flat = build_group_table(flat_topics, 'Stable')
df_exp = build_group_table(exp_topics, 'Exponential')

# Concatenate the three tables
df_concat = pd.concat([df_linear, df_flat, df_exp])

# Sort to keep organized (optional)
df_concat = df_concat.set_index(['Group', 'topic'])

# Fill NaN with zero and ensure integers
df_concat = df_concat.fillna(0).astype(int)


# Generate raw LaTeX
# latex_buffer = StringIO()
latex_table = df_concat.to_latex(
    'q3_1_1/q3_1_1_publications_por_ano_por_tipo.tex',
    # buf=latex_buffer,
    multirow=True,
    index=True,
    na_rep="0",
    float_format="%.0f",
    caption="Distribution of publications by topics and trends",
    label="tab:topics_trends",
    bold_rows=False,
    column_format="l" * 2 + "r" * len(df_concat.columns)  # l for group and topic, r for the years
)

# latex_table = latex_buffer.getvalue()

latex_table

In [ ]:
# Execute data processing or visualization steps
# create a dataframe to compare the two rankings
topic_indexes = topic_totals.copy()
topic_indexes.rename({"topico": "topic"})

# Do the merge based on the 'topic' column to bring the 'composite_index' as 'relevance'
topic_indexes = topic_indexes.merge(
    topic_index_df[["topic", "composite_index"]],
    on="topic",
    how="left"
)

fs1 = 26
fs2 = 22
fs3 = 18

# Rename a column to 'relevance'
topic_indexes = topic_indexes.rename(columns={"composite_index": "relevance"})

# create the plot

import matplotlib.cm as cm
import textwrap

# Generate the ranking by quantity of documents
topic_indexes['rank1'] = topic_indexes['document_count'] #.rank(method='first', ascending=False).astype(int)
# Generate the ranking by relevance
topic_indexes['rank2'] = topic_indexes['relevance'] #.rank(method='first', ascending=False).astype(int)

# Calculate the halves of the rankings
rank1_half = (topic_indexes['rank1'].max() / 2) * 1.2
rank2_half = (topic_indexes['rank2'].max() / 2) * 1.2


# Generate a unique color for each topic
topics = topic_indexes['topic'].unique()
colors = cm.get_cmap('tab20', len(topics))
color_dict = {topic: colors(i) for i, topic in enumerate(topics)}

# Configure white background
plt.style.use('default')
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white')
ax.set_facecolor('white')

# Plot scatter plot with colors by topic and larger markers
# for i, row in topic_indexes.iterrows():
# ax.scatter(row['rank1'], row['rank2'], color=color_dict[row['topic']], s=200, label=row['topic'])  # s defines the size of the marker
# Plot conditional points and labels
for i, row in topic_indexes.iterrows():
    x, y = row['rank1'], row['rank2']
    color = color_dict[row['topic']]
    ax.scatter(x, y, color=color, s=100)

    # ################
    # force full name here
    if row['topic'] == 'Recent advances in transition metal ECs for HER...':
        label = "Recent Advances in Transition Metal Electrocatalysts for Hydrogen and Oxygen Evolution"
    if row['topic'] == 'High-performance ECs for hydrogen and OERs in...':
        label = "High-Performance Electrocatalysts for Hydrogen and Oxygen Evolution Reactions in Water Splitting"

    # Display label only if it is above the half of any ranking
    if x > rank1_half :
        label = "\n".join(textwrap.wrap(label, 15))
        # label = row["topic"][:30] + "..."
        # ax.text(x + 0.3, y, label, fontsize=fs3, color="black")
        ax.annotate(
                        label,         # label text
                        (x, y),           # point coordinate (same as scatter)
                        xytext=(-42, -3),  # text displacement: 10 pixels below
                        textcoords='offset points',
                        ha='center',      # horizontally centralized
                        va='top',         # vertically aligned to the top of the text
                        fontsize=fs3,
                        color = "black"
                    )   
    if y > rank2_half:
        label = "\n".join(textwrap.wrap(label, 12))
        # label = row["topic"][:30] + "..."
        # ax.text(x + 0.3, y, label, fontsize=fs3, color="black")
        ax.annotate(
                        label,         # label text
                        (x, y),           # point coordinate (same as scatter)
                        xytext=(0, -10),  # text displacement: 10 pixels below
                        textcoords='offset points',
                        ha='center',      # horizontally centralized
                        va='top',         # vertically aligned to the top of the text
                        fontsize=fs3,
                        color = "black"
                    )   

# Force axes to start from zero
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

# Axes and title
ax.set_xlabel('Document Count', fontsize=fs2)
ax.set_ylabel('Relevance', fontsize=fs2)
ax.set_title('Comparison Between Relevance and\nQuantity of Documents per Topic', fontsize=fs1)
plt.xticks(fontsize=fs2, rotation=45)
plt.yticks(fontsize=fs2)
ax.grid(True)

# Remove legend duplicates
handles_labels = {
    topic: plt.Line2D([], [], marker='o', color='w',
                      markerfacecolor=color_dict[topic], markersize=10, label=topic)
    for topic in topics
}


# # Legend
# lbls = handles_labels.keys()
# lbls = [x[:30]+"..." for x in lbls]
# # plt.legend(handles_labels.values(), lbls,
# #            title="Topics", bbox_to_anchor=(0.5, -0.15), loc='upper center',
# #            ncol=4, fontsize=fs2, frameon=False)

# plt.legend(handles_labels.values(), lbls,
# title="Topics", bbox_to_anchor=(1.05, 1), loc='upper left',
# fontsize=fs3)

plt.tight_layout()

plt.savefig('q3_1_1/q3_1_1_compare_indexes.pdf') #, dpi=300)

plt.show()

### Question S-2.2

How have research priorities shifted over the past decade?


#### Charts and tables with author keywords

In [ ]:
# Execute data processing or visualization steps
# Explode a column of keywords into multiple rows
bibfile.data['keywords'] = bibfile.data['author_keywords'].str.split(';')
kw_year = bibfile.data[["keywords", "year"]].explode('keywords').reset_index(drop=True)
kw_year["keywords"] = kw_year["keywords"].str.strip()  # Remove extra blank spaces
kw_year = kw_year.rename(columns={'keywords': 'keyword'})
kw_year["year"] = kw_year["year"].astype(int)
# kw_year

# Download nltk resources (only once)
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Assuming your DataFrame is df_grouped with column 'keyword'
kw_year['keyword'] = kw_year['keyword'].str.strip().str.lower()  # standardize
kw_year['keyword'] = kw_year['keyword'].apply(lambda x: lemmatizer.lemmatize(x))

# a plural that is not being removed
kw_year['keyword'] = kw_year['keyword'].str.replace('electrocatalysts', 'electrocatalyst')

# remove duplicate spaces
kw_year['keyword'] = kw_year['keyword'].str.replace(r'\s+', ' ', regex=True)

# replace acronyms
kw_year['keyword'] = kw_year['keyword'].str.replace('oxygen evolution reaction (oer)', 'oxygen evolution reaction')
kw_year['keyword'] = kw_year['keyword'].str.replace('oer', 'oxygen evolution reaction')
kw_year['keyword'] = kw_year['keyword'].str.replace('her', 'hydrogen evolution reaction')

# remove word that should not be there
kw_year = kw_year[kw_year['keyword'].str.upper() != 'UNKNOWN']

# take only the last 15 years
kw_year_full = kw_year
kw_year = kw_year[kw_year['year'] >= 2004]

# grouping by year
kw_grouped_full = kw_year_full.groupby(['keyword', 'year']).size().reset_index(name='count') 
kw_grouped = kw_year.groupby(['keyword', 'year']).size().reset_index(name='count') 
kw_grouped

In [ ]:
# Execute data processing or visualization steps
import matplotlib as mpl
mpl.rcParams['font.weight'] = 'normal'
mpl.rcParams['axes.labelweight'] = 'normal'


fs1 = 30
fs2 = 22
fs3 = 14

# 1. Total sum of occurrences per word
top_keywords = (
    kw_grouped.groupby('keyword')['count'].sum()
    .sort_values(ascending=False)
    .head(15)
    .index
)

# 2. Filter only these 15 words
df_top = kw_grouped[kw_grouped['keyword'].isin(top_keywords)]

# Filter the last 11 years
max_year = df_top['year'].max()
df_top = df_top[df_top['year'] >= (max_year - 11)]

df_top['total'] = df_top.groupby('keyword')['count'].transform('sum')
df_top = df_top.reindex(df_top['total'].sort_values( ascending=True).index)
df_top = df_top.drop(columns=['total'])
print(df_top)
# df_top.to_csv('kw_author_year_count.csv')


fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor('white')   # Figure background
ax.set_facecolor('white')          # Chart area background

# 3. Create a figure
# plt.figure(figsize=(16, 8))

# 4. Normalize the sizes for visual bubbles
size_factor = 0.8  # Visual adjustment


highlight_year = 2025
plt.axvspan(highlight_year - 0.5, highlight_year + 0.5, color='lightgray', alpha=0.3, zorder=0)

# 5. Plot each point
for _, row in df_top.iterrows():
    plt.scatter(
        row['year'],
        row['keyword'],
        s=row['count'] * size_factor,
        c = np.log1p(row['count']),
        cmap='Pastel1', 
        alpha=0.6,
        # edgecolors='black'
    )

    # plt.scatter(
    # row['year'],
    # row['keyword'],
    # s=np.log1p(row['count']) * 30,  # log1p(x) = log(1 + x)
    # alpha=0.6,
    # edgecolors='black'
    # )



    # plt.text(
    # row['year'],
    # row['keyword'],
    # f"{row['count'] if row['count'] > 400 else ''}",
    # fontsize=fs3,
    # ha='center',
    # va='center',
    # color = "gray" if row["year"] == 2025 else "black",
    # )

# 6. Visual adjustments
plt.xlabel("Year", fontsize=fs2)
plt.ylabel("Keyword", fontsize=fs2)
plt.title("Top 15 Authors Keywords by Year (Last 11 years)", fontsize=fs1, loc='right')
plt.grid(True, axis='x', linestyle='--', alpha=0.5)

lbls = range(2014, 2026) #df_top["year"].to_list()
plt.xticks(rotation=90, ticks = lbls, fontsize=fs2, fontweight='normal')
plt.yticks(fontsize=fs2)
plt.tight_layout()

plt.savefig('q3_1_2/q3_1_2_top_15_keywords_by_year_bubble.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# table of the plot above
df_pivotado = df_top.pivot_table(
    index='keyword',  # rows: keywords
    columns='year',    # columns: years
    values='count',   # values: frequency
    aggfunc='sum',    # sum if there are multiple records
    fill_value=0      # fill missing values with 0
)
# Add a "Total" column with the sum per row
df_pivotado['Total'] = df_pivotado.sum(axis=1)

# Sort by the "Total" column in descending order
df_pivotado = df_pivotado.sort_values(by='Total', ascending=False)

df_pivotado.to_latex('q3_1_2/top_15_keywords_by_year_bubble.tex', index=True, header=True)

df_pivotado

In [ ]:
# Execute data processing or visualization steps
fs1 = 30
fs2 = 22
fs3 = 14

# Pivot the DataFrame to matrix
pivot = df_top.pivot_table(index='keyword', columns='year', values='count', fill_value=0)
pivot['total'] = pivot.iloc[:, 1:12].sum(axis=1)
pivot = pivot.reindex(pivot['total'].sort_values( ascending=False).index)
pivot.iloc[:, 0:12] = pivot.iloc[:, 0:12].div(pivot['total'], axis=0) * 100
pivot = pivot.drop(columns=['total'])
# print(pivot)
#


plt.figure(figsize=(15, 9))
ax = sns.heatmap(pivot, cmap='YlGnBu') #, annot=True, annot_kws={"size": 11}, fmt='.0f')

# Annotate manually with per-cell control
for i in range(pivot.shape[0]):           # rows (terms)
    for j in range(pivot.shape[1]):       # columns (years)
        value = int(pivot.iloc[i, j])
        color = "gray" if pivot.columns[j] == 2025 else "black"
        ax.text(j + 0.5, i + 0.5, f"{value}", color = color, 
                ha='center', va='center', fontsize=fs3)
        
plt.title("Frequency (%) of Top 15 Authors Keywords by Year", fontsize=fs1, loc='right')
plt.xlabel("Year", fontsize=fs2)
plt.ylabel("Keyword", fontsize=fs2)
plt.xticks(rotation=45,fontsize=fs2)
plt.yticks(fontsize=fs2)


plt.tight_layout()

plt.savefig('q3_1_2/q3_1_2_frequency_top_15_keywords_by_year.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# prepare df_top without year limit
top_keywords_full = (
    kw_grouped_full.groupby('keyword')['count'].sum()
    .sort_values(ascending=False)
    .head(15)
    .index
)

# 2. Filter only these 15 words
df_top_full = kw_grouped_full[kw_grouped_full['keyword'].isin(top_keywords_full)]

df_top_full['total'] = df_top_full.groupby('keyword')['count'].transform('sum')
df_top_full = df_top_full.reindex(df_top_full['total'].sort_values( ascending=True).index)
df_top_full = df_top_full.drop(columns=['total'])


def calculate_metrics(df_top):
    results = []

    for keyword, group in df_top.groupby('keyword'):
        group = group.sort_values('year')

        # Create a list of years proportional to the frequency
        repeated_years = np.repeat(group['year'], group['count'])

        # Calculate Q1, Q3 and IQR
        q1 = np.percentile(repeated_years, 25)
        q3 = np.percentile(repeated_years, 75)
        iqr = q3 - q1

        # limits calculated by the IQR rule
        lower_bound = max(int(np.floor(q1 - 1.5 * iqr)), repeated_years.min())
        upper_bound = min(int(np.ceil(q3 + 1.5 * iqr)), repeated_years.max())

        # Identify peak year
        idx_peak = group['count'].idxmax()
        peak_year = group.loc[idx_peak, 'year']
        frequency = group.loc[idx_peak, 'count']

        results.append({
            'keyword': keyword,
            'start_year': lower_bound,
            'end_year': upper_bound,
            'peak_year': peak_year,
            'frequency': frequency
        })

    return pd.DataFrame(results)

summary_df = calculate_metrics(df_top_full)
summary_df = summary_df.sort_values(by='frequency', ascending=True)

# Plot
# plt.figure(figsize=(12, 10))
fig, ax = plt.subplots(figsize=(12, 10))
ax.yaxis.grid(False)
sns.set_theme(style="whitegrid")

# Horizontal lines (period of use of the term)
for i, row in summary_df.iterrows():
    plt.plot([row['start_year'], row['end_year']], [row['keyword'], row['keyword']], color='gray', linewidth=4)

# Bubbles
plt.scatter(summary_df['peak_year'], summary_df['keyword'], s=summary_df['frequency'], alpha=0.6, color='steelblue', edgecolors='k')

# Labels
plt.xlabel("Year",fontsize=16)
plt.ylabel("Keyword",fontsize=16)
plt.title("Trend Topics (Authors Keywords)", fontsize=20)

lbls = df_top["year"].to_list()
plt.xticks(rotation=45,  fontsize=16)
plt.yticks(fontsize=16)
plt.grid(True, axis='x')

# Manual legend for frequency
import matplotlib.patches as mpatches
# for size in [2000, 4000, 6000, 8000]:
# plt.scatter([], [], s=size/2, label=str(size), color='steelblue', alpha=0.6, edgecolors='k')
# plt.legend(title='Term frequency', loc

In [ ]:
# Execute data processing or visualization steps
keywords = summary_df.sort_values(by='frequency', ascending=False)["keyword"].to_list()

years = list(range(min(df_top["year"]), max(df_top["year"])))

# Plot
fig, ax = plt.subplots(nrows=len(keywords), ncols=1, figsize=(10, 0.8 * len(keywords)),sharex=True, sharey=True)  # nrows=2 and ncols=2 to create a 2x2 grid

sns.set_theme(style="whitegrid")

# Horizontal lines (period of use of the term)
for i,kw in enumerate(keywords):
    kw_data = df_top[df_top["keyword"] == kw]
    sns.lineplot(x = 'year', y = 'count', data = kw_data, ax = ax[i]) #, color='gray', linewidth=4)

    ax[i].set_ylabel(kw, rotation=0, labelpad=30, ha='right', va='center')

    ax[i].set_yticklabels("")
    ax[i].grid(axis='y', visible=False) 

    if i != len(keywords) - 1:
        ax[i].set_xlabel("")    # Remove label from the X axis (except on the last one)
        ax[i].tick_params(labelbottom=False)  # Hides the X axis ticks
    else:
        ax[i].set_xticks(years)  # Sets the ticks for all years
        ax[i].set_xlabel("Year") # Adds X axis label (optional)

    # Bubbles
    point = (summary_df[summary_df["keyword"] == kw]["peak_year"], summary_df[summary_df["keyword"] == kw]["frequency"])
    ax[i].scatter(point[0], point[1], s=point[1]*0.5, alpha=0.6, color='steelblue', edgecolors='k')

    # Obtain the current upper limit of the Y axis and double it
    y_min, y_max = ax[i].get_ylim()
    ax[i].set_ylim(y_min * 1.15, y_max * 1.05)

plt.xticks(rotation=45)
plt.suptitle("Trend Topics", fontsize=20)
plt.tight_layout()

plt.savefig('q3_1_2/q3_1_2_frequency_trend_topics_v2.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# attempt to do the same previous plot with the ridge plot of seaborn

df_top["year"] = df_top["year"].astype(int)

# First, calculate the total count by keyword
keyword_totals = df_top.groupby("keyword")["count"].sum().reset_index()
keyword_totals = keyword_totals.sort_values(by="count", ascending=False)

# Create a dictionary to map the order of the keywords
keyword_order = {k: i for i, k in enumerate(keyword_totals["keyword"])}

# Add a column with the order of the keywords
df_top["keyword_order"] = df_top["keyword"].map(keyword_order)

# Sort by order of keyword and by ascending year
df_top = df_top.sort_values(by=["keyword_order", "year"]).drop(columns="keyword_order").reset_index(drop=True)



# Style settings
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Color palette
pal = sns.cubehelix_palette(len(df_top["keyword"].unique()), rot=-.25, light=.7)

# Create the FacetGrid with a line for each keyword
g = sns.FacetGrid(df_top, row="keyword", hue="keyword", aspect=18, height=.6, palette=pal)

# Draw an evolution line of 'count' throughout the years
g.map(sns.lineplot, "year", "count", linewidth=1.5)
g.map(sns.lineplot, "year", "count", color="b", linewidth=2)

# Base line (Y axis=0)
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)

# Function to place the name of the keyword in each plot
def label(x, y, label, color):
    ax = plt.gca()
    ax.text(0, .4, label, fontweight="bold", color=color, fontsize = 16,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, "year", "count")


unique_years = sorted(df_top["year"].unique())

for ax in g.axes.flatten():
    ax.set_xticks(unique_years)
    ax.set_xticklabels(unique_years, rotation=45, ha='right', fontsize=16)

# Adjustment of the subplots
g.figure.subplots_adjust(hspace=-.25)

g.figure.suptitle(
    "Trend Topics (Authors Keywords) - Alternative View",
    fontsize=20,            # Font size
    # fontweight='bold',      # Bold (optional)
    y=0.98                  # Vertical adjustment of the title (above the chart)
)

# Visual cleaning
g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

plt.tight_layout()
plt.savefig('q3_1_2/q3_1_2_frequency_trend_topics_av.pdf') #, dpi=300)


plt.show()

#### Charts and tables with keywords extracted from abstracts

In [ ]:
# Execute data processing or visualization steps
# normalize acronyms in the abstracts
# cannot lemmatize or remove stopwords because Yake already does this at the right time
# acronym replacement is necessary because YAKE does not handle acronyms and terms, considering both as two distinct occurrences

corpora = bibfile.data[["title", 'abstract', "year"]]

# remove duplicate spaces
corpora["abstract"] = corpora["abstract"].str.replace(r'\s+', ' ', regex=True)

# remove acronyms
siglas  = {'oer': 'Oxygen Evolution Reaction',
            'her': 'Hydrogen Evolution Reaction',
            'orr': 'Oxygen Reduction Reaction'}


def replace_acronym_by_term(texto):

    texto_modificado = texto

    for sigla, termo in siglas.items():

        # Regex to detect common patterns of the acronym with the term
        padroes_combinados = [
            rf"\(?{sigla}\)?\s*[-–:]\s*{re.escape(termo)}",   # (OER) - Oxygen Evolution Reaction
            rf"{re.escape(termo)}\s*\(?{sigla}\)?",           # Oxygen Evolution Reaction (OER)
            rf"\(?{sigla}\)?\s*{re.escape(termo)}",           # OER Oxygen Evolution Reaction
            rf"{re.escape(termo)}\s*\(?{sigla}\)?",           # Oxygen Evolution Reaction OER
        ]
        # Step 1: remove the acronym when it appears together with the term
        for padrao in padroes_combinados:
            texto_modificado = re.sub(padrao, termo, texto_modificado, flags=re.IGNORECASE)

        # Step 2: replace the isolated acronym with the term (if it still exists)
        def replace_isolated_acronym(m):
            return termo

        texto_modificado = re.sub(rf"\b{sigla}\b", replace_isolated_acronym, texto_modificado, flags=re.IGNORECASE)

    # Clean duplicate spaces left by removals
    texto_modificado = re.sub(r'\s{2,}', ' ', texto_modificado).strip()

    return texto_modificado    

corpora["abstract"] = corpora["abstract"].progress_apply(replace_acronym_by_term)

corpora.to_pickle("corpora.pkl")
# corpora = pd.read_pickle("corpora.pkl")

corpora

##### Keyword extraction with YAKE (low keyword quality)

In [ ]:
# Execute data processing or visualization steps
# Get the keywords of each paper

# import yake

# kw_extractor = yake.KeywordExtractor()

# kws = corpora.progress_apply(kw_extractor.extract_keywords)

# kws_final = bibfile.data.copy()
# kws_final["yake_kws"] = kws
# kws_final.to_pickle("keyword_by_yake.pkl")

kws_final = pd.read_pickle("keyword_by_yake.pkl")

In [ ]:
# Execute data processing or visualization steps
# # min-max score
# def normalize_yake_min_max(row):
# scores = [score for _, score in row]
# if len(scores) == 0:
# return row
# min_score = min(scores)
# max_score = max(scores)
    
# # avoids division by zero
# if max_score == min_score:
# return [(kw, 0.0) for kw, _ in row]
    
# scores_norm = [(kw, (score - min_score) / (max_score - min_score)) for kw, score in row]

# # check if there is no score outside the 0, 1 range
# for _, score in scores_norm:
# if score < 0 or score > 1:
# print(f"Error in scores of {scores}")
# print(f"    Normalized scores: {scores_norm}")
# print(f"-----------------------------")
# break

# return scores_norm


# # Z-score
# def normalize_yake_z_score(row):
# scores = [score for _, score in row]
# if len(scores) <= 1:
# return row
# mean_score = np.mean(scores)
# std_score = np.std(scores)
    
# if std_score == 0:
# return row

# #scores_norm = [(kw, (score - min_score) / (max_score - min_score)) for kw, score in row]
# scores_norm = [(kw, (score - mean_score) / std_score) for kw, score in row]

# return scores_norm


# # Apply normalization line by line
# #kws_final['yake_kws_norm'] = kws_final['yake_kws'].apply(normalize_yake_min_max)
# kws_final['yake_kws_norm'] = kws_final['yake_kws'].apply(normalize_yake_z_score)

# kws_final.to_pickle("keyword_by_yake_norm.pkl")

kws_final = pd.read_pickle("keyword_by_yake_norm.pkl")

kws_final

In [ ]:
# Execute data processing or visualization steps
# Create a dataframe with the average scores of keywords by year

# Step 1: Expand each tuple into separate rows with the corresponding year
records = []

for _, row in kws_final.iterrows():
    year = row['year']
    for kw, score in row['yake_kws_norm']:
        records.append({'year': year, 'keyword': kw, 'score': score})


# Step 2: Create a new DataFrame with the extracted data
kws_expanded = pd.DataFrame(records)

# Step 3: Group by year and keyword and calculate the average of the scores
# kws_year = kws_expanded.groupby(['year', 'keyword'], as_index=False)['score'].mean()

# # Optional: rename a column
# kws_year.rename(columns={'score': 'mean_score'}, inplace=True)

# print(min(kws_year["mean_score"]))
# print(max(kws_year["mean_score"]))
# print("---------------")
# kws_year

# Group by year and keyword, taking the average of the scores
kws_year = kws_expanded.groupby(['year', 'keyword'], as_index=False)['score'].mean()

# Calculate the first quartile by year
quartil_1_por_ano = kws_year.groupby('year')['score'].quantile(0.25).reset_index()
quartil_1_por_ano.columns = ['year', 'q1']

# Join with the original DataFrame to filter only scores <= first quartile
kws_year_q1 = kws_year.merge(quartil_1_por_ano, on='year')
kws_year_q1 = kws_year_q1[kws_year_q1['score'] <= kws_year_q1['q1']].drop(columns='q1')

kws_year_q1.rename(columns={'score': 'mean_score'}, inplace=True)

# Result: only words with average score in the first quartile (best)
kws_year_q1

In [ ]:
# Execute data processing or visualization steps
# Top 5 keywords of the last 5 years

# Obtain the 5 most recent years
last_5_years = sorted(kws_year_q1['year'].unique())[-5:]

# Filter the data to the last 5 years and remove keywords with score 0
recent_data = kws_year_q1[
    (kws_year_q1['year'].isin(last_5_years)) 
]

# For each year, get the 5 keywords with the lowest score (excluding 0 scores)
for year in sorted(last_5_years):
    print(f"\nTop 5 keywords with lowest scores in {year}:")
    top_keywords = recent_data[recent_data['year'] == year].nsmallest(5, 'mean_score')
    print(top_keywords[['keyword', 'mean_score']].to_string(index=False))

In [ ]:
# Execute data processing or visualization steps
# count frequency of the keywords

# ######################
# unparallelized version. in the next cell there is a parallelized version, which is more viable

import string
# 1. Unique and normalized (lowercase) keywords
keywords = kws_year_q1["keyword"].dropna().str.lower().unique()
keyword_set = set(keywords)
sorted_kws = sorted(keyword_set, key=lambda x: -len(x))

# 2. Remove punctuation
translator = str.maketrans('', '', string.punctuation)

# 3. Initialize count structure by year
year_keyword_counts = defaultdict(Counter)

# 4. Iterate over rows of the dataframe with tqdm
for _, row in tqdm(kws_final[["abstract", "year"]].dropna().iterrows(), total=kws_final.shape[0], desc="Counting by year"):
    year = row["year"]
    abstract = row["abstract"]
    
    # Pre-processing: lowercase, remove punctuation, tokenize
    clean_text = abstract.lower().translate(translator)
    
    # Add to counts by year (only keywords)
    for kw in sorted_kws:
        pattern = rf'\b{re.escape(kw)}\b'
        matches = re.findall(pattern, clean_text, flags=re.IGNORECASE)
        year_keyword_counts[year][kw] += len(matches)
        # Remove the already counted words from the text, replacing them with spaces
        clean_text_text = re.sub(pattern, ' ', clean_text, flags=re.IGNORECASE)

# 5. Transform the result into DataFrame
records = []
for year, counter in year_keyword_counts.items():
    for kw, freq in counter.items():
        records.append({"year": year, "keyword": kw, "frequency": freq})

kw_freq_by_year_df = pd.DataFrame(records)

# 6. Filter for frequencies > 0 and sort
kw_freq_by_year_df = kw_freq_by_year_df[kw_freq_by_year_df["frequency"] > 0]
kw_freq_by_year_df = kw_freq_by_year_df.sort_values(by=["year", "frequency"], ascending=[True, False])

# Display
print(kw_freq_by_year_df.head(10))

In [ ]:
# Execute data processing or visualization steps
# word frequency count

# ###################
# even with multiprocessing it was very slow 452 min and did not finish

from multiprocessing import Pool, cpu_count

# --------- Pre-processing ---------
translator = str.maketrans('', '', string.punctuation)

# List of keywords sorted by descending length
keywords = kws_year["keyword"].dropna().str.lower().unique()
sorted_kws = sorted(set(keywords), key=lambda x: -len(x))  # to avoid overlapping

# Count function for a part of the DataFrame
def count_keywords_in_chunk(chunk):
    local_counts = defaultdict(Counter)

    for _, row in chunk.iterrows():
        year = row["year"]
        abstract = row["abstract"]
        if pd.isna(abstract) or pd.isna(year):
            continue

        clean_text = abstract.lower().translate(translator)
        already_matched_spans = []

        for kw in sorted_kws:
            pattern = re.compile(rf'\b{re.escape(kw)}\b', flags=re.IGNORECASE)

            for match in pattern.finditer(clean_text):
                span = match.span()
                # Checks if the span does not overlap any already registered
                if not any(start < span[1] and end > span[0] for start, end in already_matched_spans):
                    local_counts[year][kw] += 1
                    already_matched_spans.append(span)

    return local_counts

# --------- Parallelization ---------
def parallel_count(df, n_processes=None):
    if n_processes is None:
        n_processes = max(cpu_count() - 1, 1)

    chunks = np.array_split(df, n_processes)

    with Pool(processes=n_processes) as pool:
        results = list(tqdm(pool.imap(count_keywords_in_chunk, chunks), total=len(chunks), desc="Processing in parallel"))

    # Combine results
    combined_counts = defaultdict(Counter)
    for partial in results:
        for year, counter in partial.items():
            combined_counts[year].update(counter)

    return combined_counts

# --------- Execute ---------
import numpy as np  # make sure to import numpy

counts = parallel_count(kws_final[["abstract", "year"]].dropna())

# Convert to DataFrame
records = []
for year, counter in counts.items():
    for kw, freq in counter.items():
        records.append({"year": year, "keyword": kw, "frequency": freq})

kw_freq_by_year_df = pd.DataFrame(records)
kw_freq_by_year_df = kw_freq_by_year_df[kw_freq_by_year_df["frequency"] > 0]
kw_freq_by_year_df = kw_freq_by_year_df.sort_values(by=["year", "frequency"], ascending=[True, False])

# Display result
print(kw_freq_by_year_df.head(10))

##### Keyword extraction with KeyBERT (very generic keywords)

In [ ]:
# Execute data processing or visualization steps
import pandas as pd
from keybert import KeyBERT
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
from sentence_transformers import SentenceTransformer
import re

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load the model with GPU support
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
sbert_model = sbert_model.to(device)

# Initialize the KeyBERT model with SBERT model loaded on the GPU
kw_model = KeyBERT(model=sbert_model)

# List to store the extracted data
records = set()  # use set to avoid duplicates (keyword, year)

# Split test sample
_, kws_teste = train_test_split(bibfile.data, test_size=0.1, random_state=42)
print(f"Number of abstracts in the test set: {len(kws_teste)}")

# Group texts by year and concatenate abstracts
year_text_dict = (
    kws_teste
    .dropna(subset=['year', 'abstract'])  # remove rows with null values
    .groupby('year')['abstract']
    .apply(lambda texts: ' '.join(texts).lower())
    .to_dict()
)

# Iterate over the DataFrame to extract keywords
for _, row in tqdm(kws_teste[["abstract", "year"]].dropna().iterrows(), total=len(kws_teste), desc="Extracting keywords with KeyBERT"):
    text = row["abstract"]
    year = row["year"]
    
    # Extract keywords
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 3),
        stop_words='english',
        top_n=10
    )

    for kw, _ in keywords:
        keyword = kw.lower()
        records.add((year, keyword))  # avoids duplicates

# Convert to DataFrame
keywords_df = pd.DataFrame(list(records), columns=["year", "keyword"])
keywords_df["count"] = 0

# Sort keywords by year and descending length (number of words)
keywords_df["n_words"] = keywords_df["keyword"].str.split().str.len()
keywords_df = keywords_df.sort_values(by=["year", "n_words"], ascending=[True, False])

# Count with progressive removal of text by year
for year in tqdm(keywords_df["year"].unique(), desc="Processing by year"):
    if year not in year_text_dict:
        continue

    text = year_text_dict[year]

    indices = keywords_df[keywords_df["year"] == year].index
    for idx in indices:
        keyword = keywords_df.at[idx, "keyword"]
        pattern = rf'\b{re.escape(keyword)}\b'
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        count = len(matches)
        keywords_df.at[idx, "count"] = count
        # Removal from text to avoid duplicate counting
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

# Clean auxiliary column
del keywords_df["n_words"]

# Sort and display
keywords_df = keywords_df.sort_values(by=["year", "count"], ascending=[True, False])
print(keywords_df.head(10))

In [ ]:
# Execute data processing or visualization steps
# top 5 keywords of the last 5 years

# Obtain the 5 most recent years
last_5_years = sorted(keywords_df['year'].unique())[-5:]

# Filter the data for the last 5 years and remove keywords with score 0
recent_data = keywords_df[
    (keywords_df['year'].isin(last_5_years)) 
]

# For each year, get the 5 keywords with the highest score (excluding 0 scores)
for year in sorted(last_5_years):
    print(f"\nTop 5 keywords with highest count in {year}:")
    top_keywords = recent_data[recent_data['year'] == year].nlargest(5, 'count')
    print(top_keywords[['keyword', 'count']].to_string(index=False))

##### Keyword extraction with TextRank

In [ ]:
# Execute data processing or visualization steps
# #!python -m spacy download en_core_web_sm
 
# import pandas as pd
# import spacy
# import pytextrank
# from collections import defaultdict, Counter
# from sklearn.model_selection import train_test_split
# from nltk.corpus import stopwords
# from tqdm import tqdm
# import torch
# from sentence_transformers import SentenceTransformer, util
# from sklearn.cluster import AgglomerativeClustering
# from sklearn.cluster import FeatureAgglomeration
# from sklearn.metrics import pairwise_distances

# # # Load spaCy + pyTextRank model
# # nlp = spacy.load("en_core_web_sm")
# # nlp.add_pipe("textrank")

# # # Initialize structure to store the results
# # year_kw_counter = defaultdict(Counter)
# # records = []

# # # _, kws_teste = train_test_split(corpora, test_size=0.05, random_state=42)
# # # print(f"Number of abstracts in the test set: {len(kws_teste)}")
# # kws_teste = corpora.copy()

# # # Iterate over abstracts
# # for _, row in tqdm(kws_teste[["abstract", "year"]].dropna().iterrows(), total=len(kws_teste), desc="Extracting keywords"):
# #     text = row["abstract"]
# #     year = row["year"]
    
# #     doc = nlp(text)
# #     keywords = [(phrase.text.lower(), phrase.count) for phrase in doc._.phrases[:10]]  # top 10 keywords
    
# #     for kw, qtd in keywords:
# #         kw_clean = kw.strip()
# #         if kw_clean:
# #             year_kw_counter[year][kw_clean] += qtd

# # # Convert the dictionary to DataFrame
# # records = []
# # for year, kw_counts in year_kw_counter.items():
# #     for kw, freq in kw_counts.items():
# #         records.append({"year": year, "keyword": kw, "frequency": freq})

# # kw_textrank_df = pd.DataFrame(records)

# # kw_textrank_df.to_pickle("kw_textrank_df.pkl")

# kw_textrank_df = pd.read_pickle("kw_textrank_df.pkl")

# # Group by similarity using sentencetransformers
# # Load model with GPU support
# device = "cpu"  #"cuda" if torch.cuda.is_available() else "cpu"
# model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# # Similarity threshold
# similarity_threshold = 0.85

# # Function to process each group by year
# def process_year_group(year, group_df):
# keywords = group_df["keyword"].tolist()
# frequencies = group_df["frequency"].tolist()

# # Embeddings
# embeddings = model.encode(keywords, convert_to_tensor=True, device=device)

# # Similarity matrix
# similarity_matrix = util.pytorch_cos_sim(embeddings, embeddings).cpu().numpy()
# distance_matrix = 1 - similarity_matrix

# # Hierarchical clustering
# clustering = AgglomerativeClustering(
# n_clusters=None,
# metric='precomputed',
# linkage='average',
# distance_threshold=1 - similarity_threshold
# )
# labels = clustering.fit_predict(distance_matrix)

# # Grouping
# cluster_map = defaultdict(list)
# freq_map = defaultdict(int)
# for kw, label, freq in zip(keywords, labels, frequencies):
# cluster_map[label].append(kw)
# freq_map[label] += freq

# # Result with shortest expression as representative
# result = []
# for label, kw_list in cluster_map.items():
# representative = min(kw_list, key=len)
# result.append({"year": year, "keyword": representative, "frequency": freq_map[label]})
# return result

# # Apply the function to all years
# results = []
# for year, group in tqdm(kw_textrank_df.groupby("year"), desc="Grouping by year"):
# results.extend(process_year_group(year, group))

# # Final result
# kw_textrank_df = pd.DataFrame(results)
# kw_textrank_df = kw_textrank_df.sort_values(by=["year", "frequency"], ascending=[True, False])

# # Replace the incorrect variant with the correct one
# kw_textrank_df["keyword"] = kw_textrank_df["keyword"].replace(
# {"oxygen evolving reaction": "oxygen evolution reaction"}
# )
# kw_textrank_df["keyword"] = kw_textrank_df["keyword"].replace(
# {"hydrogen evolution process": "hydrogen evolution reaction"}
# )
# kw_textrank_df["keyword"] = kw_textrank_df["keyword"].replace(
# {"hydrogen evolving reaction": "hydrogen evolution reaction"}
# )

# # Regroup by summing the frequencies
# kw_textrank_df = (
# kw_textrank_df.groupby(["keyword", "year"], as_index=False)["frequency"]
# .sum()
# .sort_values(by=["year", "frequency"], ascending=[True, False])
# )

# kw_textrank_df.to_pickle("keyword_by_textrank.pkl")

kw_textrank_df = pd.read_pickle("keyword_by_textrank.pkl")


kw_year_full = kw_textrank_df.copy()
kw_year_full["year"] = kw_year_full["year"].astype(int)
kw_year = kw_year_full[kw_year_full['year'] >= 2004]

kw_grouped_full = (
    kw_year_full.groupby(["keyword", "year"], as_index=False)["frequency"]
    .sum()
    .rename(columns={"frequency": "count"})
)

kw_grouped = (
    kw_year.groupby(["keyword", "year"], as_index=False)["frequency"]
    .sum()
    .rename(columns={"frequency": "count"})
)

kw_grouped

In [ ]:
# Execute data processing or visualization steps
# top 5 keywords of the last 5 years

# Obtain the 5 most recent years
last_5_years = sorted(kw_year_full['year'].unique())[-5:]

# Filter the data for the last 5 years and remove keywords with score 0
recent_data = kw_year_full[
    (kw_year_full['year'].isin(last_5_years)) 
]

# For each year, get the 5 keywords with the lowest score (excluding 0 scores)
for year in sorted(last_5_years):
    print(f"\nTop 5 keywords in {year}:")
    top_keywords = recent_data[recent_data['year'] == year].nlargest(5, 'frequency')
    print(top_keywords[['keyword', 'frequency']].to_string(index=False))

In [ ]:
# Execute data processing or visualization steps
# 1. Total sum of occurrences per word limiting to 10
top_keywords = (
    kw_grouped.groupby('keyword')['count'].sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

# 2. Filter only these 10 words
df_top = kw_grouped[kw_grouped['keyword'].isin(top_keywords)]

df_top['total'] = df_top.groupby('keyword')['count'].transform('sum')
df_top = df_top.reindex(df_top['total'].sort_values( ascending=True).index)
df_top = df_top.drop(columns=['total'])


print(df_top)



fig, ax = plt.subplots(figsize=(16, 9))
fig.patch.set_facecolor('white')   # Figure background
ax.set_facecolor('white')          # Chart area background

# 4. Normalize the sizes for visual bubbles
size_factor = 0.3  # Visual adjustment


highlight_year = 2025
plt.axvspan(highlight_year - 0.5, highlight_year + 0.5, color='lightgray', alpha=0.3, zorder=0)

# 5. Plot each point
for _, row in df_top.iterrows():
    plt.scatter(
        row['year'],
        row['keyword'],
        s=row['count'] * size_factor,
        c = np.log1p(row['count']),
        cmap='Pastel1', 
        alpha=0.6,
        # edgecolors='black'
    )

    # plt.scatter(
    # row['year'],
    # row['keyword'],
    # s=np.log1p(row['count']) * 30,  # log1p(x) = log(1 + x)
    # alpha=0.6,
    # edgecolors='black'
    # )



    plt.text(
        row['year'],
        row['keyword'],
        f"{row['count'] if row['count'] > 1000 else ''}",
        fontsize=11,
        ha='center',
        va='center',
        color = "gray" if row["year"] == 2025 else "black",
    )

# 6. Visual adjustments
plt.xlabel("Year", fontsize=16)
plt.ylabel("Keyword", fontsize=16)
plt.title("Top 10 Abstracts Keywords by Year (Last 11 years)", fontsize=20)
plt.grid(True, axis='x', linestyle='--', alpha=0.5)

lbls = df_top["year"].to_list()
plt.xticks(rotation=45, ticks = lbls, labels=lbls, fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()

plt.savefig('q3_1_2/top_10_kw_abstracts_by_year_bubble.pdf')

plt.show()

In [ ]:
# Execute data processing or visualization steps
# table of the plot above
df_pivotado = df_top.pivot_table(
    index='keyword',  # rows: keywords
    columns='year',    # columns: years
    values='count',   # values: frequency
    aggfunc='sum',    # sum if there are multiple records
    fill_value=0      # fill missing values with 0
)
# Add a "Total" column with the sum per row
df_pivotado['Total'] = df_pivotado.sum(axis=1)

# Sort by the "Total" column in descending order
df_pivotado = df_pivotado.sort_values(by='Total', ascending=False)

df_pivotado.to_latex('q3_1_2/top_10_kw_abstracts_by_year_bubble.tex', index=True, header=True)

df_pivotado

In [ ]:
# Execute data processing or visualization steps
# Pivot the DataFrame to matrix
pivot = df_top.pivot_table(index='keyword', columns='year', values='count', fill_value=0)
pivot['total'] = pivot.iloc[:, 1:22].sum(axis=1)
pivot = pivot.reindex(pivot['total'].sort_values( ascending=False).index)
pivot.iloc[:, 1:22] = pivot.iloc[:, 1:22].div(pivot['total'], axis=0) * 100
pivot = pivot.drop(columns=['total'])
# print(pivot)
#


plt.figure(figsize=(14, 7))
ax = sns.heatmap(pivot, cmap='YlGnBu') #, annot=True, annot_kws={"size": 11}, fmt='.0f')

# Annotate manually with per-cell control
for i in range(pivot.shape[0]):           # rows (terms)
    for j in range(pivot.shape[1]):       # columns (years)
        value = int(pivot.iloc[i, j])
        color = "gray" if pivot.columns[j] == 2025 else "black"
        ax.text(j + 0.5, i + 0.5, f"{value}", color = color, 
                ha='center', va='center', fontsize=11)
        
plt.title("Frequency (%) of Top 15 Abstracts Keywords by Year", fontsize=20)
plt.xlabel("Year", fontsize=16)
plt.ylabel("Keyword", fontsize=16)
plt.xticks(rotation=45,fontsize=16)
plt.yticks(fontsize=16)


plt.tight_layout()

plt.savefig('q3_1_2/q3_1_2_frequency_top_15_kw_abstracts_by_year.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# prepare a df_top without year limit
top_keywords_full = (
    kw_grouped_full.groupby('keyword')['count'].sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

# 2. Filter only these 15 words
df_top_full = kw_grouped_full[kw_grouped_full['keyword'].isin(top_keywords_full)]

df_top_full['total'] = df_top_full.groupby('keyword')['count'].transform('sum')
df_top_full = df_top_full.reindex(df_top_full['total'].sort_values( ascending=True).index)
df_top_full = df_top_full.drop(columns=['total'])


def calculate_metrics(df_top):
    results = []

    for keyword, group in df_top.groupby('keyword'):
        group = group.sort_values('year')

        # Create a list of years proportional to frequency
        repeated_years = np.repeat(group['year'], group['count'])

        # Calculate Q1, Q3 and IQR
        q1 = np.percentile(repeated_years, 25)
        q3 = np.percentile(repeated_years, 75)
        iqr = q3 - q1

        # limits calculated by the IQR rule
        lower_bound = max(int(np.floor(q1 - 1.5 * iqr)), repeated_years.min())
        upper_bound = min(int(np.ceil(q3 + 1.5 * iqr)), repeated_years.max())

        # Identify peak year
        idx_peak = group['count'].idxmax()
        peak_year = group.loc[idx_peak, 'year']
        frequency = group.loc[idx_peak, 'count']

        results.append({
            'keyword': keyword,
            'start_year': lower_bound,
            'end_year': upper_bound,
            'peak_year': peak_year,
            'frequency': frequency
        })

    return pd.DataFrame(results)

summary_df = calculate_metrics(df_top_full)
summary_df = summary_df.sort_values(by='frequency', ascending=True)

# Plot
# plt.figure(figsize=(12, 10))
fig, ax = plt.subplots(figsize=(12, 9))
ax.yaxis.grid(False)
sns.set_theme(style="whitegrid")

# Horizontal lines (period of use of the term)
for i, row in summary_df.iterrows():
    plt.plot([row['start_year'], row['end_year']], [row['keyword'], row['keyword']], color='gray', linewidth=4)

# Bubbles
plt.scatter(summary_df['peak_year'], summary_df['keyword'], s=summary_df['frequency']*0.3, alpha=0.6, color='steelblue', edgecolors='k')

# Labels
plt.xlabel("Year",fontsize=16)
plt.ylabel("Keyword",fontsize=16)
plt.title("Trend Topics (Abstracts Keywords)", fontsize=20)

lbls = range(min(summary_df["start_year"]), max(summary_df["end_year"]))
plt.xticks(rotation=45, ticks = lbls, labels=lbls, fontsize=16)

# plt.xticks(rotation=45,  fontsize=16)

plt.yticks(fontsize=16)
plt.grid(True, axis='x')

# Manual legend for frequency
import matplotlib.patches as mpatches
# for size in [2000, 4000, 6000, 8000]:
# plt.scatter([], [], s=size/2, label=str(size), color='steelblue', alpha=0.6, edgecolors='k')
# plt.legend(title='Term frequency', loc='upper right', scatterpoints=1, labelspacing=1.5)

plt.tight_layout()

plt.savefig('q3_1_2/q3_1_2_frequency_trend_topics_kw_abstracts.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
keywords = summary_df.sort_values(by='frequency', ascending=False)["keyword"].to_list()

years = list(range(min(df_top["year"]), max(df_top["year"])))

# Plot
fig, ax = plt.subplots(nrows=len(keywords), ncols=1, figsize=(10, 0.8 * len(keywords)),sharex=True, sharey=True)  # nrows=2 and ncols=2 to create a 2x2 grid

sns.set_theme(style="whitegrid")

# Horizontal lines (period of use of the term)
for i,kw in enumerate(keywords):
    kw_data = df_top[df_top["keyword"] == kw]
    sns.lineplot(x = 'year', y = 'count', data = kw_data, ax = ax[i]) #, color='gray', linewidth=4)

    ax[i].set_ylabel(kw, rotation=0, labelpad=30, ha='right', va='center')

    ax[i].set_yticklabels("")
    ax[i].grid(axis='y', visible=False) 

    if i != len(keywords) - 1:
        ax[i].set_xlabel("")    # Remove label from the X axis (except on the last one)
        ax[i].tick_params(labelbottom=False)  # Hides the X axis ticks
    else:
        ax[i].set_xticks(years)  # Sets the ticks for all the years
        ax[i].set_xlabel("Year") # Adds X axis label (optional)

    # Bubbles
    point = (summary_df[summary_df["keyword"] == kw]["peak_year"], summary_df[summary_df["keyword"] == kw]["frequency"])
    ax[i].scatter(point[0], point[1], s=point[1]*0.1, alpha=0.6, color='steelblue', edgecolors='k')

    # Obtain the current upper limit of the Y axis and double it
    y_min, y_max = ax[i].get_ylim()
    ax[i].set_ylim(y_min * 1.15, y_max * 1.05)

plt.xticks(rotation=45)
plt.suptitle("Trend Topics", fontsize=20)
plt.tight_layout()

plt.savefig('q3_1_2/q3_1_2_frequency_trend_topics_kw_abstracts_v2.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# attempt to do the same previous plot with the ridge plot of seaborn

df_top["year"] = df_top["year"].astype(int)
df_top = df_top.sort_values(by=["year", "count"], ascending=[True, True])

# Style settings
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Color palette
pal = sns.cubehelix_palette(len(df_top["keyword"].unique()), rot=-.25, light=.7)

# Create the FacetGrid with a line for each keyword
g = sns.FacetGrid(df_top, row="keyword", hue="keyword", aspect=18, height=.6, palette=pal)

# Draw an evolution line of 'count' throughout the years
g.map(sns.lineplot, "year", "count", linewidth=1.5)
g.map(sns.lineplot, "year", "count", color="b", linewidth=2)

# Base line (Y axis=0)
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)

# Function to place the name of the keyword in each plot
def label(x, y, label, color):
    ax = plt.gca()
    ax.text(0, .4, label, fontweight="bold", color=color, fontsize = 16,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, "year", "count")


unique_years = sorted(df_top["year"].unique())

for ax in g.axes.flatten():
    ax.set_xticks(unique_years)
    ax.set_xticklabels(unique_years, rotation=45, ha='right', fontsize=16)

# Adjustment of the subplots
g.figure.subplots_adjust(hspace=-.25)

g.figure.suptitle(
    "Trend Topics (Abstracts Keywords) - Alternative View",
    fontsize=20,            # Font size
    # fontweight='bold',      # Bold (optional)
    y=0.98                  # Vertical adjustment of the title (above the chart)
)

# Visual cleaning
g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

plt.tight_layout()
plt.savefig('q3_1_2/q3_1_2_frequency_trend_topics_kw_abstracts_av.pdf')


plt.show()

### Question S-2.3
What are the emerging trends and knowledge gaps?


In [ ]:
# Execute data processing or visualization steps
# #######################
# papers in only a topic

import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # <-- ESSENTIAL

import pandas as pd
from sentence_transformers import SentenceTransformer, util
from tqdm.contrib.concurrent import process_map
import torch

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}")

# Load light model compatible with 4GB GPU
# Recommended models:
# all-MiniLM-L6-v2 -> good and fast
# all-mpnet-base-v2 -> better semantic quality
# paraphrase-MiniLM-L12-v2 -> Intermediate between MiniLM and MPNet
# multi-qa-MiniLM-L6-cos-v1 -> Optimized for questions/answers and search
#
# Model                      Size      Semantic quality        GPU requirements           Notes
# all-MiniLM-L6-v2           ~80MB       Good (fast)             Very light                 Standard light
# paraphrase-MiniLM-L12-v2	 ~120MB      Better	                 Light	                    Upgrade light
# multi-qa-MiniLM-L6-cos-v1	 ~100MB	     Good for QA/search	     Light	                    Specialized
# all-mpnet-base-v2	         ~420MB 	 Excellent	             Medium (adjust batch size)	More precise
model = SentenceTransformer("all-mpnet-base-v2", device=device)


# ======================
# 1. Topics and descriptions
# ======================
topics_old = {
"Catalyst efficiency and cost": "Limited availability and high cost of platinum-group metals (PGMs) for HER and HOR. Need for stable, efficient, and low-cost alternatives (e.g., non-precious metal catalysts).",
"Catalyst stability and durability": "Catalyst degradation under acidic or alkaline conditions, especially during OER. Long-term operation leads to structural and chemical changes in materials.",
"Slow reaction kinetics": "OER, in particular, suffers from sluggish kinetics due to multi-electron transfer steps. HOR in alkaline media is much slower than in acidic media.",
"Electrode and membrane degradation": "Membrane materials (e.g., Nafion) degrade over time, affecting efficiency. Electrodes suffer from corrosion and fouling.", 
"Mass transport limitations": "Gas bubble formation in electrolysis can hinder effective mass transfer. In fuel cells, water and gas management is critical for efficiency.",
"Operating environment constraints": "Alkaline and acidic environments each have trade-offs in terms of materials compatibility and reaction rates.",
"Scalability": "Difficulty in scaling up lab-scale systems to industrial-level electrolysers or fuel cells.", 
"Energy efficiency": "Need for minimizing overpotentials and improving round-trip efficiency in energy storage systems.", 
"System integration": "Challenges in integrating electrolysis and fuel cells with intermittent renewable energy sources (e.g., solar, wind).",
"Hydrogen storage and transport": "High costs and safety concerns associated with hydrogen compression, liquefaction, or chemical storage."
}

topics = {
"Catalyst efficiency": "Active catalysts for HER and OER. Need to be stable, efficient, present low overpotentials, and low charge transfer resistance.",

"Non-noble materials": "Limited availability and high cost of platinum-group metals (PGMs). Non-noble transition metals, low-cost alternatives, and  non-precious metal catalysts.",

"Catalyst stability and durability": "Catalyst degradation under acidic or alkaline conditions”,  “Long-term operation leads to structural and chemical changes in materials”,  “Electrodes suffer from leaching and poisoning",

"Membrane degradation": "Membrane materials (e.g., Nafion, Sustainion, Fumasep, Zirfon) degrade over time, affecting efficiency. Losses of membrane permeselectivity”, “Low chemical stability due to attack by  hydroxide ion or proton ion.”, “Low ion exchange capacity.” “The increase of swelling ratio and water uptake over time indicates membrane degradation", 

"Mass transport limitations": "Gas bubble formation in electrolysis can hinder effective mass transfer. The use of flow system to improve the mass transport. Alternatives to faster eliminate gas bubbles from electrolyzer stack.",  

"Operating environment constraints": "Alkaline and acidic environments each have trade-offs in terms of materials compatibility and reaction rates. Alternatives to working under high-pressure operating conditions. Complementary reaction replacement. Organic molecules oxidation assisted hydrogen production. Inorganic molecules oxidation assisted hydrogen production. Urea Oxidation Reaction. Hydrazine Oxidation Reaction. Glycerol Oxidation Reaction", 

# "Scalability": "Explores the technical and economic challenges and strategies involved in scaling up electrolysis systems from laboratory prototypes to industrial-scale hydrogen production. ",

"System integration": "Addresses the challenges of integrating electrolysis technologies with intermittent renewable energy sources such as solar and wind. The use of electrolysis system integrated with the electrical grid to operate at time-variable price.",

"Hydrogen storage and transport": "High costs and safety concerns associated with hydrogen compression, liquefaction, or chemical storage."
}

# Generate topic embeddings
topic_names = list(topics.keys())
topic_descriptions = list(topics.values())
topic_embeddings = model.encode(topic_descriptions, convert_to_tensor=True)
with open("topic_embeddings.pkl", "wb") as f:
    pickle.dump(topic_embeddings, f)

topics2_old = {
"Computational and\nsimulation studies": "use theoretical models and computational tools to predict and analyze the properties and behavior of monochalcogenides, enabling the exploration of their electronic, optical, and mechanical characteristics without physical experiments.",
"Experimental studies": "involve the practical investigation and characterization of monochalcogenides through laboratory experiments to determine their physical, chemical, and mechanical properties using various analytical techniques.",
"Integrated experimental and\ncomputational studies": "combine experimental data with computational models to provide a comprehensive understanding of monochalcogenides, validate theoretical predictions, and enhance the interpretation of experimental results.",
"Proof-of-concept and\nprototype demonstrations": "focus on developing and testing preliminary versions of devices or systems using monochalcogenides to demonstrate their feasibility and potential applications in real-world scenarios.",
"Review and perspective articles": "provide comprehensive overviews of the current state of research on monochalcogenides, summarizing recent findings, identifying trends, and offering expert insights into future research directions and opportunities.",
}

topics2 ={
    "Computational and\nsimulation studies": "use theoretical models and computational tools to predict and analyze the properties and mechanical characteristics without physical experiments",

    "Experimental studies": "involve the practical investigation and characterization through laboratory experiments to determine their physical, chemical, and mechanical properties using various analytical techniques",

    # "Integrated experimental and\ncomputational studies": "Combine empirical data from laboratory experiments with theoretical frameworks and computational models in the same study.",
    # "Proof-of-concept and\nprototype demonstrations": "Present preliminary research focused on validating the feasibility of novel concepts, systems, or devices. ",
    
    "Review and\nperspective articles": "Systematically synthesize and critically evaluate **pre-existing scientific literature**. This topic exclusively comprises **literature surveys, systematic reviews, and meta-analyses** that consolidate current knowledge, identify research trends and gaps, and propose future directions **derived solely from the analysis of previously published findings, without introducing any novel empirical data, experimental results, or primary research contributions.**",


    "Policy and Regulatory\nStudies": "Analyze governmental policies, regulatory frameworks, international agreements, and strategic roadmaps. This includes assessments of legal, environmental, and safety regulations, as well as socio-economic policy impacts."
}


# Generate embeddings for topics2
topic_names2 = list(topics2.keys())
topic_descriptions2 = list(topics2.values())
topic_embeddings2 = model.encode(topic_descriptions2, convert_to_tensor=True)
with open("topic_embeddings2.pkl", "wb") as f:
    pickle.dump(topic_embeddings2, f)


# ======================
# 4. Texts to be classified
# ======================
# Replace with your real data
texts = bibfile.data["abstract"].to_list()


# Generate embeddings of the texts (in batch for better performance)
# text_embeddings = model.encode(texts, convert_to_tensor=True, batch_size=32, show_progress_bar=True)
# with open("text_embeddings.pkl", "wb") as f:
# pickle.dump(text_embeddings, f)
with open("text_embeddings.pkl", "rb") as f:
    text_embeddings = pickle.load(f)

# Classify texts
predicted_topics = []
predicted_topics2 = []
for text_embedding in tqdm(text_embeddings, desc="Classifying texts"):
    similarity_scores = util.cos_sim(text_embedding, topic_embeddings)
    best_topic_idx = similarity_scores.argmax().item()

    predicted_topics.append(topic_names[best_topic_idx])

    similarity_scores2 = util.cos_sim(text_embedding, topic_embeddings2)
    best_topic_idx2 = similarity_scores2.argmax().item()
    predicted_topics2.append(topic_names2[best_topic_idx2])



# Final result in DataFrame
df_results = pd.DataFrame({
    "text": texts,
    "predicted_topic_1": predicted_topics,
    "predicted_topic_2": predicted_topics2
})

# print(df_results)

In [ ]:
# Execute data processing or visualization steps
import openai
import pandas as pd
from tqdm import tqdm
import random
# Settings
# key of phd project
api_key = "sk-..."  # Replace with your key

# quantity of items to separate from each topic
amostra = 500

# Define the model and configure the API
MODEL_NAME = "gpt-4o-mini"
client = openai.OpenAI(api_key=api_key)  # Replace with your key

# Fixed prompt introduction
INTRO = (
    "You are a chemistry and materials science major studying H2 electrolysis. "
    "Below is the name and description of a research topic, followed by an abstract of a paper. "
    "Your job is to evaluate whether the paper is correctly classified under that topic.\n\n"
)

# Function to create the prompt
def create_prompt(topic_name, topic_desc, abstract):
    return (
        INTRO +
        f"### Topic:\n{topic_name}\n{topic_desc}\n\n"
        f"### Abstract:\n{abstract}\n\n"
        "Answer with one of the following options only:\n"
        "(Yes) if the classification is correct.\n"
        "(No) if it is incorrect.\n"
        "(Partially) if it is partially related but not the best match.\n"
        "Give a brief one-sentence justification."
    )

# Function to validate with the model
def validate_classification(topic_name, topic_desc, abstract):
    prompt = create_prompt(topic_name, topic_desc, abstract)
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=150
        )
        answer = response.choices[0].message.content.strip()

        if answer.startswith("(Yes)"):
            return "Yes", answer.replace("(Yes)", "").strip()
        elif answer.startswith("(No)"):
            return "No", answer.replace("(No)", "").strip()
        elif answer.startswith("(Partially)"):
            return "Partially", answer.replace("(Partially)", "").strip()
        else:
            return "Undefined", answer
    except Exception as e:
        return "Error", str(e)

# Function to select up to 100 samples per topic
def sample_100_per_topic(df, topic_col):
    samples = []
    for topic in df[topic_col].dropna().unique():
        subset = df[df[topic_col] == topic]
        sample = subset.sample(n=min(amostra, len(subset)), random_state=42)
        samples.append(sample)
    return pd.concat(samples).drop_duplicates(subset=["text"])

# Data sampling
sampled_df1 = sample_100_per_topic(df_results, "predicted_topic_1")
sampled_df2 = sample_100_per_topic(df_results, "predicted_topic_2")
sampled_df = pd.concat([sampled_df1, sampled_df2]).drop_duplicates(subset=["text"]).reset_index(drop=True)

# Initialize result columns
sampled_df["validacao1"] = ""
sampled_df["justificativa1"] = ""
sampled_df["validacao2"] = ""
sampled_df["justificativa2"] = ""

# Validation loop with progress bar
for i, row in tqdm(sampled_df.iterrows(), total=len(sampled_df)):
    abstract = row["text"]

    # Validation of topic 1
    topic1 = row["predicted_topic_1"]
    desc1 = topics.get(topic1, "No description available.")
    val1, just1 = validate_classification(topic1, desc1, abstract)
    sampled_df.at[i, "validacao1"] = val1
    sampled_df.at[i, "justificativa1"] = just1

    # Validation of topic 2
    topic2 = row["predicted_topic_2"]
    desc2 = topics2.get(topic2, "No description available.")
    val2, just2 = validate_classification(topic2, desc2, abstract)
    sampled_df.at[i, "validacao2"] = val2
    sampled_df.at[i, "justificativa2"] = just2

# Final result
validated_df = sampled_df[[
    "text", "predicted_topic_1", "validacao1", "justificativa1",
    "predicted_topic_2", "validacao2", "justificativa2"
]]

print("Predict topic 1")
x = validated_df[validated_df["validacao1"] == "Yes"]
sim1 = x["predicted_topic_1"].value_counts()
print(sim1)
tot1 = validated_df["predicted_topic_1"].value_counts()
print()
print(tot1)
print("\n\n\nPredict topic 2")
x = validated_df[validated_df["validacao2"] == "Yes"]
sim2 = x["predicted_topic_2"].value_counts()
print(sim2)
tot2 = validated_df["predicted_topic_2"].value_counts()
print()
print(tot2)

# Optional: save to CSV file
validated_df.to_csv("q3_1_3/validation_llm.csv", index=False)

In [ ]:
# Execute data processing or visualization steps
# ###################################
# papers in more than one topic

import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # <-- ESSENTIAL

import pandas as pd
from sentence_transformers import SentenceTransformer, util
from tqdm.contrib.concurrent import process_map
import torch

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}")

# Load light model compatible with 4GB GPU
# Recommended models:
# all-MiniLM-L6-v2 -> good and fast
# all-mpnet-base-v2 -> better semantic quality
# paraphrase-MiniLM-L12-v2 -> Intermediate between MiniLM and MPNet
# multi-qa-MiniLM-L6-cos-v1 -> Optimized for questions/answers and search
#
# Model                      Size      Semantic quality        GPU requirements           Notes
# all-MiniLM-L6-v2           ~80MB       Good (fast)             Very light                 Standard light
# paraphrase-MiniLM-L12-v2	 ~120MB      Better	                 Light	                    Upgrade light
# multi-qa-MiniLM-L6-cos-v1	 ~100MB	     Good for QA/search	     Light	                    Specialized
# all-mpnet-base-v2	         ~420MB 	 Excellent	             Medium (adjust batch size)	More precise
model = SentenceTransformer("all-mpnet-base-v2", device=device)


# ======================
# 1. Topics and descriptions
# ======================
topics_old = {
"Catalyst efficiency and cost": "Limited availability and high cost of platinum-group metals (PGMs) for HER and HOR. Need for stable, efficient, and low-cost alternatives (e.g., non-precious metal catalysts).",
"Catalyst stability and durability": "Catalyst degradation under acidic or alkaline conditions, especially during OER. Long-term operation leads to structural and chemical changes in materials.",
"Slow reaction kinetics": "OER, in particular, suffers from sluggish kinetics due to multi-electron transfer steps. HOR in alkaline media is much slower than in acidic media.",
"Electrode and membrane degradation": "Membrane materials (e.g., Nafion) degrade over time, affecting efficiency. Electrodes suffer from corrosion and fouling.", 
"Mass transport limitations": "Gas bubble formation in electrolysis can hinder effective mass transfer. In fuel cells, water and gas management is critical for efficiency.",
"Operating environment constraints": "Alkaline and acidic environments each have trade-offs in terms of materials compatibility and reaction rates.",
"Scalability": "Difficulty in scaling up lab-scale systems to industrial-level electrolysers or fuel cells.", 
"Energy efficiency": "Need for minimizing overpotentials and improving round-trip efficiency in energy storage systems.", 
"System integration": "Challenges in integrating electrolysis and fuel cells with intermittent renewable energy sources (e.g., solar, wind).",
"Hydrogen storage and transport": "High costs and safety concerns associated with hydrogen compression, liquefaction, or chemical storage."
}

topics = {
"Catalyst efficiency": "Active catalysts for HER and OER. Need to be stable, efficient, present low overpotentials, and low charge transfer resistance.",

"Non-noble materials": "Limited availability and high cost of platinum-group metals (PGMs). Non-noble transition metals, low-cost alternatives, and  non-precious metal catalysts.",

"Catalyst stability and durability": "Catalyst degradation under acidic or alkaline conditions”,  “Long-term operation leads to structural and chemical changes in materials”,  “Electrodes suffer from leaching and poisoning",

"Membrane degradation": "Membrane materials (e.g., Nafion, Sustainion, Fumasep, Zirfon) degrade over time, affecting efficiency. Losses of membrane permeselectivity. Low chemical stability due to attack by  hydroxide ion or proton ion. Low ion exchange capacity. The increase of swelling ratio and water uptake over time indicates membrane degradation", 

"Mass transport limitations": "Gas bubble formation in electrolysis can hinder effective mass transfer. The use of flow system to improve the mass transport. Alternatives to faster eliminate gas bubbles from electrolyzer stack.",  

"Operating environment constraints": "Alkaline and acidic environments each have trade-offs in terms of materials compatibility and reaction rates. Alternatives to working under high-pressure operating conditions. Complementary reaction replacement. Organic molecules oxidation assisted hydrogen production. Inorganic molecules oxidation assisted hydrogen production. Urea Oxidation Reaction. Hydrazine Oxidation Reaction. Glycerol Oxidation Reaction", 

# "Scalability": "Explores the technical and economic challenges and strategies involved in scaling up electrolysis systems from laboratory prototypes to industrial-scale hydrogen production. ",

"System integration": "Addresses the challenges of integrating electrolysis technologies with intermittent renewable energy sources such as solar and wind. The use of electrolysis system integrated with the electrical grid to operate at time-variable price.",

"Hydrogen storage and transport": "High costs and safety concerns associated with hydrogen compression, liquefaction, or chemical storage."
}

# Generate embeddings of the topics
topic_names = list(topics.keys())
topic_descriptions = list(topics.values())
topic_embeddings = model.encode(topic_descriptions, convert_to_tensor=True)
with open("topic_embeddings.pkl", "wb") as f:
    pickle.dump(topic_embeddings, f)

topics2_old = {
"Computational and\nsimulation studies": "use theoretical models and computational tools to predict and analyze the properties and behavior of monochalcogenides, enabling the exploration of their electronic, optical, and mechanical characteristics without physical experiments.",
"Experimental studies": "involve the practical investigation and characterization of monochalcogenides through laboratory experiments to determine their physical, chemical, and mechanical properties using various analytical techniques.",
"Integrated experimental and\ncomputational studies": "combine experimental data with computational models to provide a comprehensive understanding of monochalcogenides, validate theoretical predictions, and enhance the interpretation of experimental results.",
"Proof-of-concept and\nprototype demonstrations": "focus on developing and testing preliminary versions of devices or systems using monochalcogenides to demonstrate their feasibility and potential applications in real-world scenarios.",
"Review and perspective articles": "provide comprehensive overviews of the current state of research on monochalcogenides, summarizing recent findings, identifying trends, and offering expert insights into future research directions and opportunities.",
}

topics2 ={
    "Computational and\nsimulation studies": "use theoretical models and computational tools to predict and analyze the properties and mechanical characteristics without physical experiments",

    "Experimental studies": "involve the practical investigation and characterization through laboratory experiments to determine their physical, chemical, and mechanical properties using various analytical techniques",

    # "Integrated experimental and\ncomputational studies": "Combine empirical data from laboratory experiments with theoretical frameworks and computational models.",
    # "Proof-of-concept and\nprototype demonstrations": "Present preliminary research focused on validating the feasibility of novel concepts, systems, or devices. ",
    
    "Review and\nperspective articles": "Systematically synthesize and critically evaluate **pre-existing scientific literature**. This topic exclusively comprises **literature surveys, systematic reviews, and meta-analyses** that consolidate current knowledge, identify research trends and gaps, and propose future directions **derived solely from the analysis of previously published findings, without introducing any novel empirical data, experimental results, or primary research contributions.**",


    "Policy and Regulatory\nStudies": "Analyze governmental policies, regulatory frameworks, international agreements, and strategic roadmaps. This includes assessments of legal, environmental, and safety regulations, as well as socio-economic policy impacts."
}


# Generate embeddings of topics2
topic_names2 = list(topics2.keys())
topic_descriptions2 = list(topics2.values())
topic_embeddings2 = model.encode(topic_descriptions2, convert_to_tensor=True)
with open("topic_embeddings2.pkl", "wb") as f:
    pickle.dump(topic_embeddings2, f)


# ======================
# 4. Texts to be classified
# ======================
# Replace with your real data
texts = bibfile.data["abstract"].to_list()


# Generate embeddings of the texts (in batch for better performance)
# text_embeddings = model.encode(texts, convert_to_tensor=True, batch_size=32, show_progress_bar=True)
# with open("text_embeddings.pkl", "wb") as f:
# pickle.dump(text_embeddings, f)
with open("text_embeddings.pkl", "rb") as f:
    text_embeddings = pickle.load(f)

# # Classify texts
# predicted_topics = []
# predicted_topics2 = []
# for text_embedding in tqdm(text_embeddings, desc="Classifying texts"):
# similarity_scores = util.cos_sim(text_embedding, topic_embeddings)
# best_topic_idx = similarity_scores.argmax().item()

# predicted_topics.append(topic_names[best_topic_idx])

# similarity_scores2 = util.cos_sim(text_embedding, topic_embeddings2)
# best_topic_idx2 = similarity_scores2.argmax().item()
# predicted_topics2.append(topic_names2[best_topic_idx2])

# Define the maximum tolerance (relative difference to the most similar topic)
delta1 = 0.003  # fine-tuning based on your data
delta2 = 0.001 # fine-tuning based on your data

# Classification based on the relative difference
predicted_multi_topics1 = []
predicted_multi_topics2 = []
for text_embedding in tqdm(text_embeddings, desc="Multi-label classification (relative delta)"):
    similarity_scores = util.cos_sim(text_embedding, topic_embeddings)[0]
    max_score = similarity_scores.max().item()
    selected_topic_idxs = (similarity_scores >= (max_score - delta1)).nonzero(as_tuple=True)[0]
    topics_for_text = [topic_names[i] for i in selected_topic_idxs]
    predicted_multi_topics1.append(topics_for_text)

    similarity_scores2 = util.cos_sim(text_embedding, topic_embeddings2)[0]
    max_score2 = similarity_scores2.max().item()
    selected_topic_idxs2 = (similarity_scores2 >= (max_score2 - delta2)).nonzero(as_tuple=True)[0]
    topics_for_text2 = [topic_names2[i] for i in selected_topic_idxs2]
    predicted_multi_topics2.append(topics_for_text2)



# Final result in DataFrame
df_results = pd.DataFrame({
    "text": texts,
    "predicted_topic_1": predicted_multi_topics1,
    "predicted_topic_2": predicted_multi_topics2
})

# print(df_results)

df_temp = df_results.copy()
df_temp["qtd1"] = df_temp["predicted_topic_1"].apply(lambda x: len(x))
df_temp["qtd2"] = df_temp["predicted_topic_2"].apply(lambda x: len(x))
print("predicted_topic_1")
print(df_temp["qtd1"].value_counts())
print("predicted_topic_2")
print(df_temp["qtd2"].value_counts())

In [ ]:
# Execute data processing or visualization steps
fs1 = 28
fs2 = 20
fs3 = 18

# Group the data to count combinations of topics
grouped = (
    df_results.groupby(["predicted_topic_1", "predicted_topic_2"])
    .size()
    .reset_index(name="count")
)

# Create a figure and axes
plt.figure(figsize=(13, 10))
plt.gca().set_facecolor("white")  # chart area background
plt.gcf().patch.set_facecolor("white")  # figure background

# Scatter plot (bubbles) with salmon color
scatter = sns.scatterplot(
    data=grouped,
    x="predicted_topic_1",
    y="predicted_topic_2",
    size="count",
    color="salmon",
    sizes=(100, 3000),
    legend=False  # Remove the legend
)

# Add text inside the circles for counts > 0
for _, row in grouped.iterrows():
    if row["count"] > 0: #100:
        plt.text(
            row["predicted_topic_1"],
            row["predicted_topic_2"],
            str(row["count"]),
            color="black",
            ha="center",
            va="center",
            fontsize=fs3,
            # fontweight="bold"
        )

# Get the unique values of the y axis (topics)
y_labels = grouped["predicted_topic_2"].unique()
# Define an additional margin (e.g., 1 visual unit)
plt.ylim(len(y_labels)-0.5, -0.5)

# Visual improvements
plt.xticks(rotation=45, ha="right", fontsize=fs2)
plt.yticks(fontsize=fs2)
plt.xlabel("")
plt.ylabel("")
plt.title("Overlap Between Types of Studies and Publications", fontsize=fs1, loc="right")
plt.grid(True)
plt.tight_layout()

plt.savefig('q3_1_3/q3_1_3_overlap_between_topics.pdf') #, dpi=300)

plt.show()

pivot = grouped.pivot_table(
    index="predicted_topic_2",
    columns="predicted_topic_1",
    values="count",
    fill_value=0  # put 0 where there are no combinations
)

print(pivot)

In [ ]:
# Execute data processing or visualization steps
# ##########################################
# by document type
from upsetplot import from_memberships, UpSet
import matplotlib.pyplot as plt
import pandas as pd

# Ensure that the columns are lists (in case they come as strings)
df_results["predicted_topic_1"] = df_results["predicted_topic_1"].apply(lambda x: x if isinstance(x, list) else [x])
df_results["predicted_topic_2"] = df_results["predicted_topic_2"].apply(lambda x: x if isinstance(x, list) else [x])

# Unique list of topics from predicted_topic_1
unique_topic_2 = sorted(set(t for sublist in df_results["predicted_topic_2"] for t in sublist))

# Generate an UpSet plot for each topic in predicted_topic_1
for topic in unique_topic_2:
    # Filter the articles that have the current topic in predicted_topic_1
    subset_df = df_results[df_results["predicted_topic_2"].apply(lambda topics: topic in topics)].copy()

    if subset_df.empty:
        continue  # Skip if there is no data

    # Remove duplicates in the list of topics (in case it is repeated in both lists)
    subset_df["all_topics_1"] = subset_df["predicted_topic_1"].apply(lambda x: list(set(x)))

    # Create the data for the UpSetPlot
    upset_data = from_memberships(subset_df["all_topics_1"], data=subset_df["text"])

    # Plot
    plt.figure(figsize=(10, 6))
    UpSet(upset_data, subset_size='count', show_counts=True).plot()
    plt.suptitle(f"Topic in predicted_topic_2: {topic}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f'q3_1_3/q3_1_3_test_{topic.replace(" ", "_").lower()}.pdf') 
    plt.show()

In [ ]:
# Execute data processing or visualization steps
# ############################################
# by study type

from upsetplot import from_memberships, UpSet
import matplotlib.pyplot as plt
import pandas as pd

# Ensure that the columns are lists (in case they come as strings)
df_results["predicted_topic_1"] = df_results["predicted_topic_1"].apply(lambda x: x if isinstance(x, list) else [x])
df_results["predicted_topic_2"] = df_results["predicted_topic_2"].apply(lambda x: x if isinstance(x, list) else [x])

# Unique list of topics from predicted_topic_1
unique_topic_1 = sorted(set(t for sublist in df_results["predicted_topic_1"] for t in sublist))

# Generate an UpSet plot for each topic in predicted_topic_1
for topic in unique_topic_1:
    # Filter the articles that have the current topic in predicted_topic_1
    subset_df = df_results[df_results["predicted_topic_1"].apply(lambda topics: topic in topics)].copy()

    if subset_df.empty:
        continue  # Skip if there is no data

    # Remove duplicates in the list of topics (in case it is repeated in both lists)
    subset_df["all_topics_2"] = subset_df["predicted_topic_2"].apply(lambda x: list(set(x)))

    # Create the data for the UpSetPlot
    upset_data = from_memberships(subset_df["all_topics_2"], data=subset_df["text"])

    # Plot
    plt.figure(figsize=(10, 6))
    UpSet(upset_data, subset_size='count', show_counts=True).plot()
    plt.suptitle(f"Topic in predicted_topic_1: {topic}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Execute data processing or visualization steps
from itertools import product
from matplotlib.ticker import MaxNLocator
from collections import Counter
from itertools import product

# Font sizes
fs1 = 24  # Title
fs2 = 14  # Axes, facet titles, legend
fs3 = 12  # Internal labels of the bars

# # Ensure that the columns are lists
# df_results["predicted_topic_1"] = df_results["predicted_topic_1"].apply(lambda x: x if isinstance(x, list) else [x])
# df_results["predicted_topic_2"] = df_results["predicted_topic_2"].apply(lambda x: x if isinstance(x, list) else [x])

# Unique list of topics
topics_1 = sorted(set(t for sublist in df_results["predicted_topic_1"] for t in sublist))
topics_2 = sorted(set(t for sublist in df_results["predicted_topic_2"] for t in sublist))

# Create a data matrix
from itertools import product
grid_data = []
for t1, t2 in product(topics_1, topics_2):

    # #############################################
    # out regarding the X axis
    # "In Both": documents that have t1 in topic_1 AND t2 in topic_2
    in_both = df_results[
        df_results["predicted_topic_1"].apply(lambda x: t1 in x) &
        df_results["predicted_topic_2"].apply(lambda x: t2 in x)
    ]
    count_both = len(in_both)

    # "Out": documents that DO NOT have t1 in topic_1 but have t2 in topic_2
    out_only = df_results[
        df_results["predicted_topic_1"].apply(lambda x: t1 not in x) &
        df_results["predicted_topic_2"].apply(lambda x: t2 in x)
    ]
    count_out = len(out_only)


    grid_data.append({
        "topic_1": t1,
        "topic_2": t2,
        "in_both": count_both,
        "out_both": count_out
    })


df_grid = pd.DataFrame(grid_data)

# Colors of the dark bars
colors = {"In Both": "#003f5c", "Out": "#7a5195"}

# Plotting function
def barplot_in_cell(data, color, **kwargs):
    ax = plt.gca()
    values = [data["out_both"].values[0], data["in_both"].values[0]]
    labels = ["Out", "In Both"]
    bars = ax.barh(labels, values, height=0.6, color=[colors[l] for l in labels])   # 0.6 bar thickness
    # ax.set_xlim(0, max(10, max(values) * 1.4 + 5))   #1.15 label position in the bar
    ax.set_facecolor("white")
    ax.set_yticklabels([])
    ax.tick_params(axis='x', labelsize=fs2)
    ax.tick_params(axis='y', length=0)

    for bar, val in zip(bars, values):
        if val < 10000:
            ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                    f"{val}", va='center', ha='left', fontsize=fs3, color='black')
        else:
            ax.text(bar.get_width() * 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{val}", va='center', ha='center', fontsize=fs3, color='white')

# Invert col/row
g = sns.FacetGrid(df_grid, row="topic_1", col="topic_2", margin_titles=False, despine=True, height=2)
g.map_dataframe(barplot_in_cell)
g.set_titles("")
# g.set_axis_labels("Document types", "Study types", fontsize=fs2)
g.figure.set_size_inches(14, 16)

# External labels with 45 degrees rotation
for ax, label in zip(g.axes[-1], topics_2):  # x axis
    ax.set_xlabel(label, fontsize=fs2, rotation=45, labelpad=5, ha='right')
for ax_row, label in zip(g.axes[:, 0], topics_1):  # y axis
    ax_row.set_ylabel(label, fontsize=fs2, rotation=45, labelpad=5, ha='right', va='center')

# Layout adjustments
plt.subplots_adjust(hspace=0.3, wspace=0.3)  # height and width of each chart

# Global legend, moved further down
handles = [plt.Rectangle((0, 0), 1, 1, color=colors["In Both"]),
           plt.Rectangle((0, 0), 1, 1, color=colors["Out"])]
labels = ["In Both", "Other document type"]
g.figure.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=2, fontsize=fs2)

# Title closer to the plot
plt.suptitle("Study Types vs. Document Types\n(and other document types)", fontsize=fs1, y=0.98)

# final space
plt.tight_layout(rect=[0, 0.005, 1, 0.96])

g.figure.text(0.5, 0.01, "Document types", ha='center', fontsize=fs2)
g.figure.text(0.01, 0.5, "Study types", va='center', rotation='vertical', fontsize=fs2)


plt.savefig('q3_1_3/q3_1_3_study_types_vs_document_types.pdf')

plt.show()

In [ ]:
# Execute data processing or visualization steps
from itertools import product
from matplotlib.ticker import MaxNLocator
from collections import Counter
from itertools import product

# Font sizes
fs1 = 24  # Title
fs2 = 14  # Axes, facet titles, legend
fs3 = 12  # Internal bar labels

# Ensure that columns are lists
df_results["predicted_topic_1"] = df_results["predicted_topic_1"].apply(lambda x: x if isinstance(x, list) else [x])
df_results["predicted_topic_2"] = df_results["predicted_topic_2"].apply(lambda x: x if isinstance(x, list) else [x])

# Unique list of topics
topics_1 = sorted(set(t for sublist in df_results["predicted_topic_1"] for t in sublist))
topics_2 = sorted(set(t for sublist in df_results["predicted_topic_2"] for t in sublist))

# Create a data matrix
grid_data = []
for t1, t2 in product(topics_1, topics_2):
    # "In Both": documents that have t1 in topic_1 AND t2 in topic_2
    in_both = df_results[
        df_results["predicted_topic_1"].apply(lambda x: t1 in x) &
        df_results["predicted_topic_2"].apply(lambda x: t2 in x)
    ]
    count_both = len(in_both)

    # "Out": documents that HAVE t1 in topic_1, but DO NOT have t2 in topic_2 (i.e., outside the Y axis)
    out_only = df_results[
        df_results["predicted_topic_1"].apply(lambda x: t1 in x) &
        df_results["predicted_topic_2"].apply(lambda x: t2 not in x)
    ]
    count_out = len(out_only)

    grid_data.append({
        "topic_1": t1,
        "topic_2": t2,
        "in_both": count_both,
        "out_both": count_out
    })

df_grid = pd.DataFrame(grid_data)

# Define the global maximum limit of the X axis with margin
max_val = df_grid[["in_both", "out_both"]].values.max()
xlim_max = max(10, max_val * 1.2)  # 20% margin

# Bar colors
colors = {"In Both": "#003f5c", "Out": "#7a5195"}

# Plotting function for each cell
def barplot_in_cell(data, xlim_max, **kwargs):
    ax = plt.gca()
    values = [data["out_both"].values[0], data["in_both"].values[0]]
    labels = ["Out", "In Both"]
    bars = ax.barh(labels, values, height=0.6, color=[colors[l] for l in labels])
    # ax.set_xlim(0, xlim_max)
    ax.set_facecolor("white")
    ax.set_yticklabels([])
    ax.tick_params(axis='x', labelsize=fs2)
    ax.tick_params(axis='y', length=0)

    for bar, val in zip(bars, values):
        if val < 10000:
            ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                    f"{val}", va='center', ha='left', fontsize=fs3, color='black')
        else:
            ax.text(bar.get_width() * 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{val}", va='center', ha='center', fontsize=fs3, color='white')

# Create subplot grid
g = sns.FacetGrid(df_grid, row="topic_1", col="topic_2", margin_titles=False, despine=True, height=2)
g.map_dataframe(barplot_in_cell, xlim_max=xlim_max)
g.set_titles("")
g.figure.set_size_inches(14, 16)

# External labels
for ax, label in zip(g.axes[-1], topics_2):  # X axis
    ax.set_xlabel(label, fontsize=fs2, rotation=45, labelpad=5, ha='right')
for ax_row, label in zip(g.axes[:, 0], topics_1):  # Y axis
    ax_row.set_ylabel(label, fontsize=fs2, rotation=45, labelpad=5, ha='right', va='center')

# Layout adjustments
plt.subplots_adjust(hspace=0.3, wspace=0.3)

# Global legend
handles = [plt.Rectangle((0, 0), 1, 1, color=colors["In Both"]),
           plt.Rectangle((0, 0), 1, 1, color=colors["Out"])]
labels = ["In Both", "Other study type"]
g.figure.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=2, fontsize=fs2)

# Title
plt.suptitle("Study Types vs. Document Types\n(and other study types)", fontsize=fs1, y=0.98)

# final spacing
plt.tight_layout(rect=[0, 0.005, 1, 0.96])

# Titles of main axes
g.figure.text(0.5, 0.01, "Document types", ha='center', fontsize=fs2)
g.figure.text(0.01, 0.5, "Study types", va='center', rotation='vertical', fontsize=fs2)

plt.savefig('q3_1_3/q3_1_3_teste4.pdf', bbox_inches='tight') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
from itertools import product
from matplotlib.ticker import MaxNLocator
from collections import Counter

# Ensure that columns are lists
df_results["predicted_topic_1"] = df_results["predicted_topic_1"].apply(lambda x: x if isinstance(x, list) else [x])
df_results["predicted_topic_2"] = df_results["predicted_topic_2"].apply(lambda x: x if isinstance(x, list) else [x])

# Unique list of topics
topics_1 = sorted(set(t for sublist in df_results["predicted_topic_1"] for t in sublist))
topics_2 = sorted(set(t for sublist in df_results["predicted_topic_2"] for t in sublist))

# Prepare grid with the data
grid_data = []
for t1, t2 in product(topics_1, topics_2):
    # Articles that have t2 in predicted_topic_2
    subset = df_results[df_results["predicted_topic_2"].apply(lambda lst: t2 in lst)]
    
    # Count topics in predicted_topic_1 within this subset
    all_t1s = [t for row in subset["predicted_topic_1"] for t in row]
    count_topics = Counter(all_t1s)
    
    # Cell data (include "In Both" explicitly and the others)
    cell_data = {"topic_1": t1, "topic_2": t2}
    for k, v in count_topics.items():
        label = "In Both" if k == t1 else k
        cell_data[label] = v
    grid_data.append(cell_data)

# Transform to DataFrame and fill missing values with 0
df_grid = pd.DataFrame(grid_data).fillna(0)

# Count columns (excludes topic_1 and topic_2)
count_cols = df_grid.columns.difference(["topic_1", "topic_2"])

# Color palette
palette = {"In Both": "#1f77b4"}
other_topics = [c for c in count_cols if c != "In Both"]
palette.update({t: "#d3d3d3" for t in other_topics})  # Gray for others

# Horizontal bar plotting function by cell
def cell_barplot(data, color=None, **kwargs):
    ax = plt.gca()
    row = data.iloc[0]
    values = []
    labels = []

    for col in count_cols:
        if row[col] > 0:
            labels.append(col)
            values.append(row[col])

    bars = ax.barh(labels, values, height=0.35, color=[palette[l] for l in labels])
    ax.set_xlim(0, max(values) * 1.2 + 1)
    ax.set_facecolor("white")
    ax.set_yticklabels([])
    ax.tick_params(axis='x', labelsize=6)
    ax.tick_params(axis='y', length=0)

    for bar, val, lbl in zip(bars, values, labels):
        ax.text(bar.get_width() * 0.5, bar.get_y() + bar.get_height() / 2,
                f"{val}", va='center', ha='center',
                fontsize=6, color='white' if bar.get_width() > 1 else 'black')

# FacetGrid
g = sns.FacetGrid(df_grid, row="topic_2", col="topic_1", margin_titles=True, despine=True)
g.map_dataframe(cell_barplot)
g.set_titles(col_template="{col_name}", row_template="{row_name}", size=7)
g.set_axis_labels("", "")

# External labels
for ax, label in zip(g.axes[-1], topics_1):  # x axis
    ax.set_xlabel(label, fontsize=6, rotation=90, labelpad=5)
for ax_row, label in zip(g.axes[:, 0], topics_2):  # y axis
    ax_row.set_ylabel(label, fontsize=6, labelpad=3, rotation=0, ha='right', va='center')

# Layout adjustment
plt.subplots_adjust(hspace=0.6, wspace=0.6)

# Global legend
unique_labels = ["In Both"] + sorted(other_topics)
handles = [plt.Rectangle((0, 0), 1, 1, color=palette[l]) for l in unique_labels]
g.fig.legend(handles, unique_labels, loc='lower center', ncol=5, fontsize=8)
plt.suptitle("Article Distribution by Topic Intersection", fontsize=14, y=1.03)
plt.tight_layout(rect=[0, 0.08, 1, 0.97])

plt.savefig('q3_1_3/q3_1_3_teste1.pdf') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Ensure that columns are lists
df_results["predicted_topic_1"] = df_results["predicted_topic_1"].apply(lambda x: x if isinstance(x, list) else [x])
df_results["predicted_topic_2"] = df_results["predicted_topic_2"].apply(lambda x: x if isinstance(x, list) else [x])

# Unique list of topics
topics_1 = sorted(set(t for sublist in df_results["predicted_topic_1"] for t in sublist))
topics_2 = sorted(set(t for sublist in df_results["predicted_topic_2"] for t in sublist))

# Initialize count matrix
heatmap_data = pd.DataFrame(0, index=topics_2, columns=topics_1)

# Count how many times each combination occurs
for _, row in df_results.iterrows():
    for t2 in row["predicted_topic_2"]:
        for t1 in row["predicted_topic_1"]:
            if t2 in topics_2 and t1 in topics_1:
                heatmap_data.at[t2, t1] += 1

# Plot
plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    cbar_kws={"label": "Number of articles"},
    linewidths=0.5,
    linecolor="gray"
)

plt.title("Topic Intersection Heatmap (Predicted_topic_2 vs Predicted_topic_1)", fontsize=14)
plt.xlabel("topics - predicted_topic_1")
plt.ylabel("topics - predicted_topic_2")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig('q3_1_3/q3_1_3_teste2.pdf') #, dpi=300)

plt.show()

## Group 2 Questions
Geographical and Research Group Characterization

### Question S-3.1
Which countries and regions contribute the most significantly to electrochemical research H2?


In [ ]:
# Execute data processing or visualization steps
# #############################################################################################
# first version with author, region, country

# ==============================
# 2. Expand DataFrame (aut, ctr, lan are lists)
# ==============================

# Example: each row can have several authors and countries
records = []
for _, row in df_data.iterrows():
    for aut, ctr in zip(row['aut'], row['ctr']):
        records.append({'author': aut, 'country': ctr})

df_expanded = pd.DataFrame(records)

# ==============================
# 3. Select the 10 most frequent authors
# ==============================

top_authors = df_expanded['author'].value_counts().head(10).index.tolist()
df_top = df_expanded[df_expanded['author'].isin(top_authors)].copy()

# ==============================
# 4. Add regions
# ==============================

df_top['region'] = df_top['country'].apply(get_region)

# ==============================
# 5. Build labels and indexes
# ==============================

authors = sorted(df_top['author'].unique())
regions = sorted(df_top['region'].unique())
countries = sorted(df_top['country'].unique())
labels = authors + regions + countries

label_index = {label: i for i, label in enumerate(labels)}

# ==============================
# 6. Create links to the Sankey (author → region → country)
# ==============================

# author → region
link1 = df_top.groupby(['author', 'region']).size().reset_index(name='value')
link1['source'] = link1['author'].map(label_index)
link1['target'] = link1['region'].map(label_index)

# region → country
link2 = df_top.groupby(['region', 'country']).size().reset_index(name='value')
link2['source'] = link2['region'].map(label_index)
link2['target'] = link2['country'].map(label_index)

# Join the links
links = pd.concat([
    link1[['source', 'target', 'value']],
    link2[['source', 'target', 'value']]
])

# ==============================
# 7. Create Sankey Diagram
# ==============================

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=links['source'],
        target=links['target'],
        value=links['value']
    )
)])

fig.update_layout(
    title_text="Top 10 Authors → Regions → Countries",
    font_size=24,
    width=1000,     # width in pixels
    height=600,      # height in pixels
    font=dict(
        size=18,
        color="black"
    )
)

fig.write_image('q3_2_1/q3_2_1_sankey_author_region_country.pdf')

fig.show()

In [ ]:
# Execute data processing or visualization steps
# #############################################################################################
# second version with only region and country

# 1. Filter the 6 most frequent regions
top_regions = df_data[df_data['country_region'] != "Others"]['country_region'].value_counts().nlargest(6).index
df_filtered = df_data[df_data['country_region'].isin(top_regions)]

# Group and count the countries by region
counts = df_filtered.groupby(['country_region', 'first_country']).size().reset_index(name='count')

# For each region, select the 5 most frequent countries
top_countries_by_region = (
    counts
    .sort_values(['country_region', 'count'], ascending=[True, False])
    .groupby('country_region')
    .head(5)
)

# # Transform to DataFrame
# top_countries_by_region = top_countries_by_region.reset_index(name='count')

# 3. Create the nodes and the links for the Sankey
# Create list of nodes
regions = top_countries_by_region['country_region'].unique().tolist()
countries = top_countries_by_region['first_country'].unique().tolist()
labels = regions + countries

# Map names to indexes
label_to_index = {label: idx for idx, label in enumerate(labels)}

# Create the connections (source, target, value)
sources = top_countries_by_region['country_region'].map(label_to_index)
targets = top_countries_by_region['first_country'].map(label_to_index)
values = top_countries_by_region['count']


# ==============================
# 7. Create Sankey Diagram
# ==============================

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=30,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(
    title_text="Top 6 Regions → Top 5 Countries (referring to the first author)",
    font_size=24,
    width=1000,     # width in pixels
    height=1000,      # height in pixels
    font=dict(
        size=20,
        color="black"
    )
)

fig.write_image('q3_2_1/q3_2_1_sankey_region_country.pdf')

fig.show()

In [ ]:
# Execute data processing or visualization steps
# #############################################################################################
# third version with quantity in front of the names

# 1. Obtain and sort the 6 most frequent regions
top_regions = df_data[df_data['country_region'] != "Others"]['country_region'].value_counts().nlargest(6).index

# 2. Filter the DataFrame to contain only these regions
df_filtered = df_data[df_data['country_region'].isin(top_regions)].copy()

# 3. Convert the 'country_region' column to a sorted categorical type.
# This ensures that the regions appear in descending order in the plot.
df_filtered['country_region'] = pd.Categorical(df_filtered['country_region'], categories=top_regions, ordered=True)

# 4. Group, count and select the 5 most frequent countries for each region
top_countries_by_region = (
    df_filtered.groupby(['country_region', 'first_country'], observed=True)
    .size()
    .reset_index(name='count')
    .sort_values(['country_region', 'count'], ascending=[True, False]) # 'country_region' will respect the categorical order
    .groupby('country_region', observed=True)
    .head(5)
)

# 5. Calculate the totals for each region (sum of the counts of its filtered countries)
region_totals = top_countries_by_region.groupby('country_region', observed=True)['count'].sum()

# 6. Create the labels for the nodes of the diagram, already including the counts
# Region labels: "Region Name (Total)"
region_labels = [f"{region} ({count})" for region, count in region_totals.items()]

# Country labels: "Country Name (Count)"
country_labels = [f"{row['first_country']} ({row['count']})" for idx, row in top_countries_by_region.iterrows()]

# Final list of all labels (nodes)
labels = region_labels + country_labels

# 7. Map the labels to numerical indexes, which will be used to create the links
label_to_index = {label: i for i, label in enumerate(labels)}

# 8. Create the 'source', 'target', and 'value' lists for the links
# Maps the origin regions to their respective indexes
sources = top_countries_by_region['country_region'].apply(
    lambda region: label_to_index[f"{region} ({region_totals[region]})"]
)

# Maps the destination countries to their respective indexes
targets = top_countries_by_region.apply(
    lambda row: label_to_index[f"{row['first_country']} ({row['count']})"], axis=1
)

# The values of the links are the counts of each country
values = top_countries_by_region['count']

# ==============================
# 9. Create a Sankey Diagram figure
# ==============================

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=30,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,  # Uses the new labels with counts
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(
    title_text="Top 6 Regions → Top 5 Countries (referring to the first author)",
    font_size=24,
    width=1000,
    height=1000,
    font=dict(
        size=20,
        color="black"
    )
)

fig.write_image('q3_2_1/q3_2_1_sankey_region_country.pdf')

fig.show()

In [ ]:
# Execute data processing or visualization steps
fs1 = 28
fs2 = 22
fs3 = 20

# Filter data: remove "UNKNOWN" countries
df_filtered = df_data[df_data['first_country'] != "UNKNOWN"]

# Select top 10 authors by number of publications
top_authors = df_filtered['first_author'].value_counts().nlargest(10).index
df_top = df_filtered[df_filtered['first_author'].isin(top_authors)]

# Group data by author and country
summary = df_top.groupby(['first_author', 'first_country']).agg(
    publications=('first_author', 'count'),
    citations=('citations_tot', 'sum')
).reset_index()

# Only authors with 5 or more publications will have a visible label
summary['label'] = summary.apply(
    lambda row: row['first_author'] if row['publications'] >= 5 else '', axis=1
)

# Calculate sizeref
max_citations = summary['citations'].max()
sizeref = 2. * max_citations / (40. ** 2)
sizeref = sizeref * (2/3)

# Determine text position
summary['textposition'] = 'top center'
summary.loc[summary['first_author'].str.lower().str.strip() == 'liu, wei', 'textposition'] = 'bottom center'

# Create figure
fig = go.Figure()

# Create a trace by country
for country in summary['first_country'].unique():
    country_data = summary[summary['first_country'] == country]
    
    fig.add_trace(go.Scatter(
        x=country_data['publications'],
        y=country_data['citations'],
        mode='markers+text',
        name=country,
        text=country_data['label'],
        textposition=country_data['textposition'],
        textfont=dict(size=fs2),
        marker=dict(
            size=country_data['citations'] / sizeref,
            sizemode='area',
            sizeref=1,
            line=dict(width=1, color='DarkSlateGrey')
        ),
        hovertext=country_data['first_author'],
        hoverinfo='text',
        showlegend=True
    ))

# Layout
fig.update_layout(
    width=1300,
    height=900,
    plot_bgcolor='white',
    paper_bgcolor='white',
    title="Top 10 Authors by Number of Publications",
    title_font=dict(size=fs1),
    xaxis=dict(
        title='# Publications',
        title_font=dict(size=fs2),
        tickfont=dict(size=fs2),
        showline=True,
        linecolor='black',
        gridcolor='lightgray',
        gridwidth=1,
        griddash='dot'
    ),
    yaxis=dict(
        title='# Citations',
        title_font=dict(size=fs2),
        tickfont=dict(size=fs2),
        showline=True,
        linecolor='black',
        gridcolor='lightgray',
        gridwidth=1,
        griddash='dot'
    ),
    legend_title_text='Countries',
    legend_title_font=dict(size=fs2),
    legend=dict(font=dict(size=fs2))
)

# Save image
fig.write_image('q3_2_1/q3_2_1_authors_publication_citations_countries.pdf')

fig.show()

In [ ]:
# Execute data processing or visualization steps
df_data["countries"]

In [ ]:
# Execute data processing or visualization steps
# =============================
# Auxiliary function for consecutive pairs
# =============================
def pairwise(iterable):
    a, b = tee(iterable)
    next(b, None)
    return zip(a, b)

# =============================
# Select the top 5 countries
# =============================
df_data['countries'] = df_data['countries'].apply(
    lambda lst: ["USA" if c == "United States of America" else c for c in lst]
)

country_counter = Counter()
for ctr_list in df_data['countries']:
    country_counter.update(set(ctr_list))  

top5_countries = [country for country, _ in country_counter.most_common(10)]

# =============================
# Generate flows with a maximum of 3 unique countries per doc
# =============================
country_links = []

for _, row in df_data.iterrows():
    countries = [c for c in row['countries'] if c in top5_countries]

    # Remove duplicates while keeping order
    seen = set()
    unique_countries = []
    for c in countries:
        if c not in seen:
            seen.add(c)
            unique_countries.append(c)

    if len(unique_countries) >= 2:
        limited_countries = unique_countries[:3]  # Limits to at most 3 countries
        labeled_countries = [f"{c} ({i+1})" for i, c in enumerate(limited_countries)]
        for source, target in pairwise(labeled_countries):
            country_links.append((source, target))

# =============================
# Create DataFrame with the flows
# =============================
df_links = pd.DataFrame(country_links, columns=['source', 'target'])
df_links['value'] = 1
df_links = df_links.groupby(['source', 'target']).agg({'value': 'sum'}).reset_index()

# =============================
# Create nodes and indexes
# =============================
nodes = sorted(set(df_links['source']) | set(df_links['target']))
node_map = {label: i for i, label in enumerate(nodes)}


df_links['source_idx'] = df_links['source'].map(node_map)
df_links['target_idx'] = df_links['target'].map(node_map)


# Extract base country name
base_countries = sorted(set(n.split(" (")[0] for n in nodes))
base_countries = top5_countries
x_positions = []
y_positions = []


flow_totals = {}

for _, row in df_links.iterrows():
    src_country = row['source'].split(" (")[0]
    tgt_country = row['target'].split(" (")[0]
    value = row['value']
    
    flow_totals[src_country] = flow_totals.get(src_country, 0) + value
    flow_totals[tgt_country] = flow_totals.get(tgt_country, 0) + value

total_flow = sum(flow_totals.values())

# proportional height (scale 0–1)
heights = {c: flow_totals[c] / total_flow for c in base_countries}
x_positions = []
y_positions = []

current_y = 0.02   # top margin
gap = 0.01         # Space between nodes

y_map = {}

for country in base_countries:
    y_map[country] = current_y
    current_y += heights[country] + gap


n_countries = len(base_countries)

# reduces the usable area to avoid collision
top_margin = 0.05
bottom_margin = 0.05
usable_space = 1 - top_margin - bottom_margin

spacing = usable_space / n_countries

for n in nodes:
    country = n.split(" (")[0]
    level = n.split("(")[1][0]

    y = y_map[country]

    if level == "1":
        x = 0.05
    elif level == "2":
        x = 0.5
    else:
        x = 1.0   # keep below 0.8

    x_positions.append(x)
    y_positions.append(y)

# =============================
# Sankey Diagram
# =============================
fig = go.Figure(data=[go.Sankey(
    arrangement="fixed",
    node=dict(
        pad=40,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=nodes,
        x=x_positions,
        y=y_positions
    ),
    link=dict(
        source=df_links['source_idx'].tolist(),
        target=df_links['target_idx'].tolist(),
        value=df_links['value'].tolist()
    )
)])

fig.update_layout(
    title_text="Collaborations between the 10 most productive countries<br>(up to 3 levels)",
    title_x=0.5,  # center
    title_y=0.95, # vertical title position
    font_size=32,
    width=1200,
    height=1200,
    margin=dict(t=250),  # 👈 Space above the chart
    font=dict(
        size=32,
        color="black"
    )
)


fig.write_image('q3_2_1/q3_2_1_sankey_country_country.pdf')


fig.show()

### Question S-3.2
How has the global distribution of research evolved over the years?


In [ ]:
# Execute data processing or visualization steps
import plotly.express as px

# #################################################################################################################
#
# CREATING THE PLOT
#
# #################################################################################################################

# ==============================
# 1. Extract the first country of each article
# ==============================
df_data['first_country'] = df_data['ctr'].apply(lambda x: x[0] if isinstance(x, list) and x else None)

# ==============================
# 2. Count publications by country and year
# ==============================
df_yearly = df_data.groupby(['year', 'first_country']).size().reset_index(name='count')

# ==============================
# 3. Calculate cumulative total (last frame)
# ==============================
df_total = df_yearly.groupby('first_country')['count'].sum().reset_index()
df_total['year'] = 'Total'  # Adds a fake column for the "Total" frame

# ==============================
# 4. Join with original data
# ==============================
df_all = pd.concat([df_yearly, df_total], ignore_index=True)

# ==============================
# 5. Obtain the maximum value to keep fixed scale
# ==============================
max_count = df_yearly['count'].max()

# ==============================
# 6. Create the Choropleth with Plotly Express
# ==============================
fig = px.choropleth(
    df_all,
    locations='first_country',
    locationmode='country names',
    color='count',
    hover_name='first_country',
    animation_frame='year',
    color_continuous_scale='YlOrRd',
    range_color=(0, max_count),
    title='Annual scientific production by country'
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title="No. of publications"
    ),
    width=1000,
    height=600
)

fig.show()

# #################################################################################################################
#
# SAVING SOME FRAMES AS PDF
#
# #################################################################################################################

# Make sure the Year column is an integer
df_yearly['year'] = df_yearly['year'].astype(int)

anos_para_salvar = [1991, 2000, 2010, 2024]

# Define the global maximum value for the color scale
max_count = df_yearly['count'].max()

for ano in anos_para_salvar:
    df_ano = df_yearly[df_yearly['year'] == ano]
    
    if df_ano.empty:
        print(f"Year {ano} not found in the DataFrame. Skipping...")
        continue

    fig_ano = px.choropleth(
        df_ano,
        locations='first_country',
        locationmode='country names',
        color='count',
        hover_name='first_country',
        color_continuous_scale='YlOrRd',
        range_color=(0, max_count),
        title=f'Publications by country - {ano}',
    )

    fig_ano.update_layout(
        geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
        coloraxis_colorbar=dict(title="No of publications"),
        width=1000,
        height=600,
        font_size=24,
        font=dict(
            size=18,
            color="black"
        )        
    )

    # Save as PNG (requires kaleido installed)
    fig_ano.write_image(f"q3_2_2/q3_2_2_publications_{ano}.pdf")


# #################################################################################################################
#
# SAVING AS GIF
#
# #################################################################################################################

import imageio.v2 as imageio  # use v2 for compatibility

# =============================
# 1. Ensure that 'Year' is an integer
# =============================
df_yearly['year'] = df_yearly['year'].astype(int)

# =============================
# 2. Create temporary directory for images
# =============================
output_dir = "q3_2_2/map_frames"
os.makedirs(output_dir, exist_ok=True)

# =============================
# 3. Obtain sorted years + add 'Total' frame
# =============================
anos = sorted(df_yearly['year'].unique())
max_count = df_yearly['count'].max()

# Add the cumulative frame
df_total = df_yearly.groupby('first_country')['count'].sum().reset_index()
df_total['year'] = 'Total'
df_all = pd.concat([df_yearly, df_total], ignore_index=True)

# =============================
# 4. Generate map images
# =============================
image_files = []

for ano in list(anos) + ['Total']:
    df_ano = df_all[df_all['year'] == ano]

    fig = px.choropleth(
        df_ano,
        locations='first_country',
        locationmode='country names',
        color='count',
        hover_name='first_country',
        color_continuous_scale='YlOrRd',
        range_color=(0, max_count),
        title=f'Publications by country - {ano}'
    )

    fig.update_layout(
        geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
        coloraxis_colorbar=dict(title="No. of publications"),
        width=1000,
        height=600
    )

    image_path = os.path.join(output_dir, f"frame_{ano}.png")
    fig.write_image(image_path)
    image_files.append(image_path)
    # print(f"Saved: {image_path}")

# =============================
# 5. Create the GIF
# =============================
gif_path = "q3_2_2/publications_by_country.gif"
with imageio.get_writer(gif_path, mode='I', duration=1.5) as writer:
    for file in image_files:
        image = imageio.imread(file)
        writer.append_data(image)

In [ ]:
# Execute data processing or visualization steps
# maps by decades


# Define periods and create dictionary of intervals
periods = {
    "1993-2004": (1993, 2004),
    "2005-2014": (2005, 2014),
    "2015-2024": (2015, 2024),
    "1993-2024": (1993, 2024)
}

# Count by country in each period
country_counts = {}

for label, (start, end) in periods.items():
    df_period = df_data[(df_data['year'] >= start) & (df_data['year'] <= end)]

    # ###################################################################################
    # # filter for study based on normalized impact factor
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['journal_if'] >= df_period['journal_if'].quantile(threshold_percent)]
    # ###################################################################################

    # ###################################################################################
    # # filter for study based on article relevance index
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['citations_tot'] >= df_period['citations_tot'].quantile(threshold_percent)]
    # ###################################################################################


    # ###################################################################################
    # # filter for study based on JNIF, which indicates the papers that received citations above the expected (Impact Factor)
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['JNIF'] >= df_period['JNIF'].quantile(threshold_percent)]
    # ###################################################################################

    # ###################################################################################
    # # filter for study based on score_log = w1*log(citations+1) + w2*log(jif+1)
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['score_log'] >= df_period['score_log'].quantile(threshold_percent)]
    # ###################################################################################

    # ###################################################################################
    # # filter for study based on score_sqrt = sqrt(citations * JIF)
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['score_sqrt'] >= df_period['score_sqrt'].quantile(threshold_percent)]
    # ###################################################################################


    counts = df_period['first_country'].value_counts().reset_index()
    counts.columns = ['country', 'publications']
    country_counts[label] = counts

# Find maximum value to standardize color scale
max_value = max(df['publications'].max() for df in country_counts.values())

# Generate plots separately
for label, df_counts in country_counts.items():
    fig = px.choropleth(
        df_counts,
        locations='country',
        locationmode='country names',
        color=np.log1p(df_counts['publications']), #'publications',
        color_continuous_scale='YlOrRd',
        range_color=(0, np.log1p(max_value)), #(0, max_value),
        labels={'publications': 'publications'},
        title=label
    )

    fig.update_layout(
        title=dict(
            text=label,
            x=0.5,
            xanchor='center',
            font=dict(size=20)
        ),
        coloraxis_colorbar=dict(
            title='publications (log)',
            title_font=dict(size=20),
            tickfont=dict(size=20)
        ),
        margin=dict(l=0, r=0, t=50, b=0)
    )

    # Save as image
    my_path = "q3_2_2/"
    fig.write_image(f"{my_path}publication_map_log_{label.replace('-', '_')}.jpg", width=900, height=500)

    # Show interactively (optional)
    fig.show()

# Generate a LaTeX table of the data
# Initialize set with countries of the top 10 of each period
top_countries = set()

# Identify the top 10 countries by period
for period, df in country_counts.items():
    df_clean = df[df['country'] != 'UNKNOWN']
    top10 = df_clean.nlargest(20, 'publications')['country']
    top_countries.update(top10)

# Create consolidated DataFrame with the relevant countries
consolidated_df = pd.DataFrame({'country': sorted(top_countries)})

for period, df in country_counts.items():
    df_clean = df[df['country'] != 'UNKNOWN']
    df_period = df_clean[df_clean['country'].isin(top_countries)]

    df_grouped = df_period.groupby('country', as_index=False)['publications'].sum()
    df_grouped.rename(columns={'publications': period}, inplace=True)

    consolidated_df = consolidated_df.merge(df_grouped, on='country', how='left')

# Fill missing values with zero and convert to integers
consolidated_df.fillna(0, inplace=True)
consolidated_df = consolidated_df.set_index('country').astype(int)

# Sort by the values of the period 1993-2024, if exists
if '1993-2024' in consolidated_df.columns:
    consolidated_df = consolidated_df.sort_values(by='1993-2024', ascending=False)

print(consolidated_df[["1993-2024"]].head(10))

# Generate LaTeX
latex_table = consolidated_df.to_latex(
    index=True,
    caption='Top 10 countries by period in number of publications on H₂ electrochemistry',
    label='tab:top10_publications_country',
    bold_rows=True,
    column_format='l' + 'r' * len(consolidated_df.columns)
)

tex_file = "q3_2_2/q3_2_2_publications_by_decade_log.tex"
with open(tex_file, "w") as f:
    f.write(latex_table)

In [ ]:
# Execute data processing or visualization steps
# ######################################
# scatter plot to evaluate the impact factor distribution and the quantity of citations

import matplotlib.pyplot as plt

df_period = df_data[(df_data['year'] >= 1993) & (df_data['year'] <= 2024)]

plt.figure(figsize=(8,6))
plt.scatter(df_period['journal_if'], df_period['citations_tot'])

plt.xlabel('Journal Impact Factor (journal_if)')
plt.ylabel('Total Citations (citations_tot)')
plt.title('Scatter Plot: Journal IF vs Total Citations')

plt.grid(True)
plt.show()


# ######################################
# create column with categories based on IF and citations

# 1. Calculate the thresholds (Medians)
# We use medians because they are robust to outliers (Power Law distribution)
citation_threshold = df_data['citations_per_year'].median()  #.quantile(.75) #
if_threshold = df_data['journal_if'].median()   #.quantile(.75) # 

# 2. Define a function to categorize each row
def get_quadrant(row):
    high_cit = row['citations_per_year'] >= citation_threshold
    high_if = row['journal_if'] >= if_threshold
   
    if high_if and high_cit:
        return "Star (High IF, High Cit)"
    elif not high_if and high_cit:
        return "Hidden Gem (Low IF, High Cit)"
    elif high_if and not high_cit:
        return "Coattail (High IF, Low Cit)"
    else:
        return "Long Tail (Low IF, Low Cit)"

# 3. Apply the function
df_data['category_group'] = df_data.apply(get_quadrant, axis=1)

# Optional: View the counts to see how balanced your groups are
print("\n\n category_group")
print(df_data['category_group'].value_counts())

# ######################################
# create column with the JNIF = citations of paper / expected citations (JIF)
df_data['JNIF'] = df_data.apply(
    lambda row: row['citations_tot'] / row['journal_if'] 
    if row['journal_if'] > 0 else None,
    axis=1
)
print("\n\n JNIF")
print(df_data['JNIF'].value_counts())

# ######################################
# create column with score_log = w1*log(citations+1) + w2*log(jif+1).
w1 = 0.7  # citations weight
w2 = 0.3  # JIF weight
df_data['score_log'] = (
    w1 * np.log(df_data['citations_tot'] + 1) +
    w2 * np.log(df_data['journal_if'] + 1)
)
print("\n\n score_log")
print(df_data['score_log'].value_counts())

# ######################################
# create column with score_sqrt = sqrt(cittaions * JIF)
df_data['score_sqrt'] = np.sqrt(df_data['citations_tot'].fillna(0) * df_data['journal_if'].fillna(0))
print("\n\n score_sqrt")
print(df_data['score_sqrt'].value_counts())

df_data.to_parquet('df_data.parquet', index=False)
df_data.to_csv('df_data.csv', index=False)

In [ ]:
# Execute data processing or visualization steps
# ########################################################
# generate the chart by countries from two cells ago, but with a count by categories
# ########################################################

categories = df_data['category_group'].unique()

periods = {
    "1993-2004": (1993, 2004),
    "2005-2014": (2005, 2014),
    "2015-2024": (2015, 2024),
    "1993-2024": (1993, 2024)
}

MIN_PUBLICATIONS = 10
TOP_N = 5000000

df_top_n = []

for label, (start, end) in periods.items():

    if label != "1993-2024":
        continue

    df_period = df_data[(df_data['year'] >= start) & (df_data['year'] <= end)]

    # ###################################################################################
    # # filter for study based on JNIF, which indicates the papers that received citations above the expected (Impact Factor)
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['JNIF'] >= df_period['JNIF'].quantile(threshold_percent)]
    # ###################################################################################

    # ###################################################################################
    # # filter for study based on score_log = w1*log(citations+1) + w2*log(jif+1)
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['score_log'] >= df_period['score_log'].quantile(threshold_percent)]
    # ###################################################################################

    # ###################################################################################
    # # filter for study based on score_sqrt = sqrt(citations * JIF)
    # # threshold_percent can be 0.25 (1st quartile), 0.5 (median), etc.
    # threshold_percent = 0.75
    # df_period = df_period[df_period['score_sqrt'] >= df_period['score_sqrt'].quantile(threshold_percent)]
    # ###################################################################################


    # [new] 1. Calculate the total publications of each country in this period (denominator)
    # This needs to be done outside the categories loop to get the global total of the country
    country_totals = df_period.groupby('first_country').size().reset_index(name='total_period')

    # [FILTER] Keep only countries with minimum relevant production
    country_totals = country_totals[country_totals['total_period'] >= MIN_PUBLICATIONS]

    country_totals = country_totals.sort_values(by='total_period', ascending=False).head(TOP_N)
    
    for cat in categories:

        df_cat = df_period[df_period['category_group'] == cat]

        counts = df_cat.groupby('first_country').size().reset_index(name='publications')

        if counts.empty:
            continue

        # [new] 2. Join the category counts with the country total
        counts = counts.merge(country_totals, on='first_country', how='left')

        # [new] 3. Calculate percentage
        counts['percentage'] = (counts['publications'] / counts['total_period']) * 100

        if label == "1993-2024":
            # print(counts.sort_values(by='publications', ascending=False).head(10))
            # Displays sorted by percentage now, if preferred
            top10 = counts.sort_values(by='percentage', ascending=False).head(100000)
            # add column indicating the category
            top10['category'] = cat
            # accumulate in the vector
            df_top_n.append(top10)
            print(top10)

        # max_value = counts['publications'].max()
        # [ADJUSTMENT] Define the maximum value for a color scale based on the percentage
        max_value = counts['percentage'].max()

        fig = px.choropleth(
            counts,
            locations='first_country',
            locationmode='country names',
            # color=np.log1p(counts['publications']),
            # [ADJUSTMENT] Use percentage column for a color
            # Removed np.log1p because percentage is linear and easier to read directly
            color='percentage',
            color_continuous_scale='YlOrRd',
            # range_color=(0, np.log1p(max_value)),
            range_color=(0, 100), # max_value),
            # title=f"{label} — {cat}",
            # labels={'publications': 'Publications'}
            # [ADJUSTMENT] Update hover label
            title=f"{label} — {cat} (%)",
            hover_data=['publications', 'total_period'], # Optional: show absolute values on hover
            labels={'percentage': 'Percentage (%)', 'publications': 'publications in Cat.', 'total_period': 'Total country'}            
        )

        fig.update_layout(
            title=dict(text=f"{label} — {cat}", x=0.5, font=dict(size=20)),
            margin=dict(l=0, r=0, t=40, b=0)
        )

        path = "q3_2_2/"
        fig.write_image(f"{path}map_{cat.replace(' ', '_')}_{label.replace('-', '_')}.jpg",
                        width=900, height=500)

        fig.show()

    df_top_n = pd.concat(df_top_n, ignore_index=True)

In [ ]:
# Execute data processing or visualization steps
df_top_n

In [ ]:
# Execute data processing or visualization steps
import plot_likert

plot_likert.__internal__.BAR_LABEL_FORMAT = "%.1f"

top10_countries = df_top_n.sort_values(by='total_period', ascending=False)
top10_countries = top10_countries["first_country"].unique()
top10_countries = top10_countries[:10]

df_filtrado = df_top_n[df_top_n["first_country"] != "UNKNOWN"]
df_filtrado = df_filtrado[df_filtrado["first_country"].isin(top10_countries)]

# 1. Pivot the dataframe
df_likert = (
    df_filtrado
    .pivot_table(
        index='first_country',
        columns='category',
        values='percentage',
        aggfunc='sum',        # uses sum in case there are duplicates
        fill_value=0          # if no value exists, fill with zero
    )
)


df_likert["sort_value"] = df_likert["Hidden Gem (Low IF, High Cit)"] + df_likert["Star (High IF, High Cit)"]
df_likert = df_likert.sort_values(by="sort_value", ascending=False)
df_likert = df_likert.drop(columns=["sort_value"])

categorias_ordenadas = [ "Long Tail (Low IF, Low Cit)", "Coattail (High IF, Low Cit)", "Hidden Gem (Low IF, High Cit)","Star (High IF, High Cit)"]
df_likert = df_likert.reindex(columns=categorias_ordenadas, fill_value=0)

print(df_likert)

fig = plot_likert.plot_counts(df_likert,
                        scale = categorias_ordenadas,
                        compute_percentages=True,
                        bar_labels=True,
                        bar_labels_color="snow",
                        figsize=(15,15),
                        colors=['#ffffff00',
 'firebrick',
 'lightcoral',
 'cornflowerblue',
 'darkblue'])


fig.set_xlabel("Percentage of papers in each category")

# --- Visual adjustments ---
ax = fig.axes   # gets the first (and only) axis of the figure

# White background
ax.set_facecolor("white")
fig.patch.set_facecolor("white")

# Remove grid lines
ax.grid(False)

# Black borders (spines)
for spine in ax.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1.5)

# Axis titles
ax.set_xlabel(ax.get_xlabel(), fontsize=20)
ax.set_ylabel(ax.get_ylabel(), fontsize=20)

# Axis labels (ticks)
ax.tick_params(axis='x', labelsize=20, rotation=45)
ax.tick_params(axis='y', labelsize=20)

# Legend font size
leg = ax.get_legend()
if leg is not None:
    for text in leg.get_texts():
        text.set_fontsize(20)   # legend label font
    leg.set_title(leg.get_title().get_text(), prop={"size": 20})  # legend title font

# Adjust bar label font
for txt in ax.texts:
    txt.set_fontsize(20) 

# -------------------------------------------------
# SAVE FIGURE
# -------------------------------------------------
path = "q3_2_2/"

plt.savefig(
    f"{path}likert_1993_2024_top10_categories_countries.jpg",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# ##########################################################################
# identifying the elite publication countries
# ##########################################################################

df_period = df_data[(df_data['year'] >= 1993) & (df_data['year'] <= 2024)]
df_period = df_period[df_period['first_country'] != "UNKNOWN"]

# -----------------------------------------------------
# Function to calculate % in the top quartile by metric
# -----------------------------------------------------
def calcular_percentual_top(df, column, pct=0.75):
    threshold = df[column].quantile(pct)
    df_top = df[df[column] >= threshold]

    total = df.groupby("first_country").size()
    top = df_top.groupby("first_country").size()

    df_out = pd.DataFrame({
        "total_publications": total,
        f"top_{column}": top
    }).fillna(0)

    df_out[f"perc_{column}"] = df_out[f"top_{column}"] / df_out["total_publications"]
    return df_out


# -----------------------------------------------------
# 1️⃣ Separate calculation for each metric
# -----------------------------------------------------
df_jnif = calcular_percentual_top(df_period, "JNIF")
df_if = calcular_percentual_top(df_period, "journal_if")
df_cit = calcular_percentual_top(df_period, "citations_tot")

# -----------------------------------------------------
# 2️⃣ Consolidate everything into a single dataframe
# -----------------------------------------------------
df_all = df_jnif.join(df_if[['top_journal_if','perc_journal_if']], how="outer")
df_all = df_all.join(df_cit[['top_citations_tot','perc_citations_tot']], how="outer")

# Ensure zeros where there is no data
df_all = df_all.fillna(0)

# -----------------------------------------------------
# 3️⃣ Create flag columns (Elite = ≥25% in the top)
# And apply a minimum filter of 10 publications
# -----------------------------------------------------
df_all["elite_jnif"] = df_all["perc_JNIF"] >= 0.25
df_all["elite_if"] = df_all["perc_journal_if"] >= 0.25
df_all["elite_cit"] = df_all["perc_citations_tot"] >= 0.25

df_all = df_all[df_all["total_publications"] >= 10]

# Sorting by combined percentages (optional)
df_all = df_all.sort_values(["elite_jnif","elite_if","elite_cit","perc_JNIF"], 
                            ascending=False)

df_all

In [ ]:
# Execute data processing or visualization steps
# #################################################################################
# the plots from the previous cell, but in bars, with each bar being a country
# #################################################################################

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# -------------------------------------------------
# DEFINE TOP 10 COUNTRIES BY OVERALL TOTAL
# -------------------------------------------------

# List of existing categories
categories = df_top_n["category"].unique()

# # Final set of countries (avoids duplicates)
# selected_countries = set()

# # For each category, get the top10 countries
# for cat in categories:
# df_cat = df_top_n[df_top_n["category"] == cat]
# top10 = (
# df_cat.groupby("first_country")["percentage"]
# .sum()
# .sort_values(ascending=False)
# .head(5)
# .index.tolist()
# )
# selected_countries.update(top10)

# # Transform back into a list and sort (optional)
# top10_countries = sorted(selected_countries)

top10_countries = df_top_n.sort_values(by='total_period', ascending=False)
top10_countries = top10_countries["first_country"].unique()
top10_countries = top10_countries[:10]

# Distinct palette
# base_cmap = cm.get_cmap("tab10")
base_cmap = cm.get_cmap("Pastel2")

# Fixed list of categories (for colors)
categories = sorted(df_top_n["category"].unique())
n_categories = len(categories)

# Define a dictionary: category → color
category_colors = {
    cat: base_cmap(i % 10)
    for i, cat in enumerate(categories)
}

# -------------------------------------------------
# FIGURE
# -------------------------------------------------
fig, axes = plt.subplots(
    len(top10_countries), 1,
    figsize=(14, 1.5 * len(top10_countries) + 2),
    sharex=True
)

if len(top10_countries) == 1:
    axes = [axes]

# Store handles/labels for a global legend
global_handles = []
global_labels = []

# -------------------------------------------------
# PLOT EACH COUNTRY
# -------------------------------------------------
for ax, country in zip(axes, top10_countries):

    df_ct = df_top_n[df_top_n["first_country"] == country].copy()
    df_ct = df_ct.sort_values("percentage", ascending=False)

    left = 0

    for _, row in df_ct.iterrows():

        cat = row["category"]
        val = row["percentage"]
        color = category_colors[cat]

        h = ax.barh(country, val, left=left, color=color, label=cat)

        # accumulate handles only 1 time per category
        if cat not in global_labels:
            global_handles.append(h)
            global_labels.append(cat)

        ax.text(
            left + val/2,
            country,
            f"{val:.1f}%",
            va="center",
            ha="center",
            fontsize=10,
            rotation=90
        )

        left += val

    ax.grid(axis="x", linestyle="--", alpha=0.4)

# -------------------------------------------------
# GLOBAL LEGEND
# -------------------------------------------------
fig.legend(
    global_handles,
    global_labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=5,
    title="Categories",
    fontsize=10,
    title_fontsize=11
)

# -------------------------------------------------
# TITLES
# -------------------------------------------------
fig.suptitle("Category Contribution per Country (Top 10 Countries)", fontsize=15)
fig.supxlabel("Percentage (%)")

plt.tight_layout(rect=[0, 0.05, 1, 0.95])

# -------------------------------------------------
# SAVE FIGURE
# -------------------------------------------------
path = "q3_2_2/"

plt.savefig(
    f"{path}panel_1993_2024_top10_countries_global_legend.jpg",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# Execute data processing or visualization steps
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------
# 1) Pivot the dataframe to Country × Category format
# -------------------------------------------------
df_pivot = df_top_n.pivot_table(
    index="first_country",
    columns="category",
    values="percentage",
    aggfunc="sum",   # in case there are multiple rows per country-category
    fill_value=0
)

# -------------------------------------------------
# 2) Sort countries by the total (of the row)
# -------------------------------------------------
df_pivot["total"] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values("total", ascending=False)
df_pivot = df_pivot.drop(columns="total")

# -------------------------------------------------
# 3) Heatmap Settings
# -------------------------------------------------
plt.figure(figsize=(14, max(6, 0.4 * len(df_pivot))))

sns.set_theme(style="white")

ax = sns.heatmap(
    df_pivot,
    annot=True,              # writes values in cells
    fmt=".1f",               # XX.X format
    cmap="YlGnBu",           # discrete and scientific palette
    linewidths=.5,           # thin lines between cells
    cbar_kws={"label": "Percentage (%)"},
)

# -------------------------------------------------
# 4) Titles and adjustments
# -------------------------------------------------
plt.title("Country Contribution per Category — Heatmap", fontsize=15)
plt.xlabel("Category")
plt.ylabel("Country")

plt.tight_layout()

# -------------------------------------------------
# 5) Save image (optional)
# -------------------------------------------------
path = "q3_2_2/"
plt.savefig(
    f"{path}heatmap_countries_categories.jpg",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# #################################################################################
# the plots from the previous cell, but in bars, with each bar being a category
# #################################################################################

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Assuming df_top_n has:
# - first_country
# - percentage
# - category

categories = sorted(df_top_n["category"].unique())

# More distinct palette (tab10 is excellent)
base_cmap = cm.get_cmap("tab10")

n_cats = len(categories)

fig, axes = plt.subplots(
    n_cats, 1,
    figsize=(14, 1.5 * n_cats + 2),
    sharex=True
)

if n_cats == 1:
    axes = [axes]

# -------------------------------------------------
# PLOT SUBPLOTS
# -------------------------------------------------
for ax, cat in zip(axes, categories):

    df_cat = df_top_n[df_top_n["category"] == cat].copy()
    df_cat = df_cat.sort_values("percentage", ascending=False)

    countries_sorted = df_cat["first_country"].tolist()
    values_sorted = df_cat["percentage"].tolist()

    # Number of countries at the top of this category
    n_countries_cat = len(countries_sorted)

    # Define colors ONLY for this category
    colors = [base_cmap(i % 10) for i in range(n_countries_cat)]

    left = 0
    handles = []
    labels = []

    for (country, val, color) in zip(countries_sorted, values_sorted, colors):

        h = ax.barh(cat, val, left=left, color=color, label=country)
        handles.append(h)
        labels.append(country)

        ax.text(
            left + val/2,
            cat,
            f"{val:.1f}%",
            va="center",
            ha="center",
            fontsize=11,
            rotation=90
        )

        left += val

    # Legend FOR THIS SUBPLOT
    ax.legend(
        handles, labels,
        bbox_to_anchor=(1.02, 0.5),
        loc="center left",
        title=f"{cat} — Countries",
        ncol=2,
    )

    ax.grid(axis="x", linestyle="--", alpha=0.4)

# -------------------------------------------------
# General Titles
# -------------------------------------------------
fig.suptitle("Country Contribution per Category", fontsize=15)
fig.supxlabel("Percentage (%)")

plt.tight_layout(rect=[0, 0, 0.88, 0.97])

# -------------------------------------------------
# Save figure without losing legends
# -------------------------------------------------
path = "q3_2_2/"
plt.savefig(
    f"{path}panel_1993_2024_individual_legends.jpg",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

### Question S-3.3
What research institutions and universities are leading in this field?

In [ ]:
# Execute data processing or visualization steps
# ##############################################################################################
# plots by journal


# Font sizes
fs1 = 30
fs2 = 24
fs3 = 22

# Select the top 10 journals
top_journal_counts = df_data['journal_abrev'].value_counts().nlargest(10)
top_journals = top_journal_counts.index

# Create layout with 8 columns
fig = plt.figure(figsize=(24, 30))
gs = fig.add_gridspec(5, 8, width_ratios=[1]*8, height_ratios=[1.2, 1, 1, 1, 0.4])

# === General plot of the journals ===
ax_top = fig.add_subplot(gs[0, :])
wedges, texts, autotexts = ax_top.pie(
    top_journal_counts.values,
    labels=None,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': fs3},
    pctdistance=1.05
)
for text in autotexts:
    text.set_path_effects([
        path_effects.Stroke(linewidth=2, foreground='white'),
        path_effects.Normal()
    ])
ax_top.set_title("Topics by journal (top 10 journals, top 5 topics)", fontsize=fs1,  pad=40)
ax_top.legend(
    wedges,
    top_journal_counts.index,
    loc='center left',
    ncol=1,
    bbox_to_anchor=(1.15, 0.5),
    fontsize=fs3
)


# === Color palette and unique topics ===
color_palette = cm.get_cmap('tab20', 20)
topic_label_set = set()
journal_topic_labels = []

for journal in top_journals:
    df_journal = df_data[df_data['journal_abrev'] == journal]
    topic_counts = df_journal['topic_name_cluster3_reduced'].value_counts().nlargest(5)
    full_labels = [label for label in topic_counts.index]
    journal_topic_labels.append((topic_counts.index, full_labels))
    topic_label_set.update(full_labels)

sorted_labels = sorted(topic_label_set)
label_to_color = {label: color_palette(i) for i, label in enumerate(sorted_labels)}

# === Positions of the 10 plots ===
# subplot_positions = (
# [(1, 0), (1, 2), (1, 4), (1, 6)] +  # Row with 4 plots
# [(2, 1), (2, 3), (2, 5)] +          # Row with 3
# [(3, 1), (3, 3), (3, 5)]            # Row with 3
# )
subplot_positions = (
    [(1, 0), (1, 2), (1, 4), (1, 6)] +  # Row with 4 charts
    [(2, 0), (2, 3), (2, 6)] +          # Row with 3
    [(3, 0), (3, 3), (3, 6)]            # Row with 3
)

# === Plots of the journals ===
for i, (journal, (topic_keys, labels)) in enumerate(zip(top_journals, journal_topic_labels)):
    row, col = subplot_positions[i]
    ax = fig.add_subplot(gs[row, col:col+2])

    df_journal = df_data[df_data['journal_abrev'] == journal]
    topic_counts = df_journal['topic_name_cluster3_reduced'].value_counts().loc[topic_keys]

    wedges, texts, autotexts = ax.pie(
        topic_counts.values,
        labels=None,
        colors=[label_to_color[label] for label in topic_keys],
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': fs3},
        pctdistance=1.05
    )

    ax.set_xlim(-0.5, 1.0)
    for text in autotexts:
        v = 0.85
        text.set_path_effects([
            path_effects.Stroke(linewidth=2, foreground=(v, v, v)),
            path_effects.Normal()
        ])

    jrn = journal.replace('INTERNATIONAL', 'INT.').replace("JOURNAL", "J.").replace("\&", "&")
    ax.set_title(f"{jrn}\n({len(df_journal)} docs)", fontsize=fs2, loc="center")

    # Move the plots of the first row closer (row 1)
    if row == 1:
        pos = ax.get_position()
        # Reduces the width and centers it more
        ax.set_position([
            pos.x0 + 0.01,  # moves slightly to the right
            pos.y0,         # same height
            pos.width - 0.01,  # slightly narrower
            pos.height
        ])

# === Single legend (row 4) ===
legend_ax = fig.add_subplot(gs[4, :])
legend_ax.axis('off')

patches = [
    plt.Line2D([0], [0], marker='o', color='w',
               label=label, markersize=30,
               markerfacecolor=color)
    for label, color in label_to_color.items()
]

legend = legend_ax.legend(
    handles=patches,
    loc='center',
    ncol=1,  # One column for easier reading
    fontsize=fs2,
    frameon=True
)
legend.get_frame().set_facecolor('white')

# General spacing adjustment
fig.subplots_adjust(hspace=0.8, wspace=1.0)
plt.tight_layout()



plt.savefig('q3_2_3/q3_2_3_top_journal_topic.pdf', bbox_inches='tight') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# ##############################################################################################
# table by journal

import pandas as pd
from tabulate import tabulate

# Ensure columns are strings
df_data['journal_abrev'] = df_data['journal_abrev'].astype(str)
# df_data['topic_name'] = df_data['topic_name'].astype(str)

# Top 10 journals by number of documents
top_journals = df_data['journal_abrev'].value_counts().nlargest(10)

# List to store the rows of the table
table_rows = []

# Generate the data for the table
for journal in top_journals.index:
    # jrn = journal.replace('INTERNATIONAL', 'INT.').replace("JOURNAL", "J.").replace("\&", "&") will have to be changed here
    df_journal = df_data[df_data['journal_abrev'] == journal]
    journal_total = len(df_journal)

    # Add the journal row (with total)
    table_rows.append([journal, journal_total, '', ''])

    # Top 5 topics of the journal
    top_topics = df_journal['topic_name_cluster3_reduced'].value_counts().nlargest(5)
    for topic, count in top_topics.items():
        table_rows.append(['', '', topic, count])

# Create DataFrame for the table
df_table = pd.DataFrame(table_rows, columns=["Journal (Top 10)", "Total Docs", "Topics (Top 5)", "Total Docs"])

# Export as LaTeX using tabulate
latex_table = tabulate(df_table, headers='keys', tablefmt='LaTeX', showindex=False)

with open("q3_2_3/q3_2_3_top_journal_topic_table.tex", "w") as f:
    f.write(latex_table)

# Display or save
print(latex_table)

In [ ]:
# Execute data processing or visualization steps
# alternative view 1
import matplotlib.patches as mpatches


fs1 = 36
fs2 = 28
fs2_5 = 24
fs3 = 26

# Step 1: count publications by journal and select the top 10
top_journals = (
    df_data['journal_abrev']
    .value_counts()
    .nlargest(10)
    .index
)

df_top_journals = df_data[df_data['journal_abrev'].isin(top_journals)]
# df_top_journals["journal_abrev"] = df_top_journals["journal_abrev"].apply(lambda x: x.replace('INTERNATIONAL', 'INT.').replace("JOURNAL", "J.").replace("\&", "&")) will have to be changed here

# Step 2: count top 5 topics by journal
grouped = (
    df_top_journals
    .groupby(['journal_abrev', 'topic_name_cluster3_reduced'])
    .size()
    .reset_index(name='count')
)

# Select the top 5 topics per journal
def get_top_topics_per_journal(df):
    return (
        df.sort_values(['journal_abrev', 'count'], ascending=[True, False])
          .groupby('journal_abrev')
          .head(5)
    )

top_topics = get_top_topics_per_journal(grouped)

# Pivot to wide format (journals x topics)
pivot_df = (
    top_topics
    .pivot_table(index='journal_abrev', columns='topic_name_cluster3_reduced', values='count', fill_value=0)
)

# Sort by the total sum
pivot_df = pivot_df.loc[pivot_df.sum(axis=1).sort_values(ascending=True).index]

# Prepare simplified labels for a legend
def simplify_label(label):
    parts = str(label).split('_')
    new_label = parts[1] if len(parts) > 1 else label
    new_label = re.sub(r'(\D)(\d+)', lambda m: f"{m.group(1)}$_{m.group(2)}$", new_label)
    return new_label

# Plot with white background
fig, ax = plt.subplots(figsize=(26, 28), facecolor='white')
plot = pivot_df.plot(
    kind='barh',
    stacked=True,
    colormap='tab20',
    ax=ax,
    width=0.3  # decreases the bar height, increasing space between them (default 0.8)    
)


# White background also for the axis
ax.set_facecolor('white')

# Title and labels
ax.set_title('Topics by journal (top 10 journals, top 5 topics)', fontsize=fs1)
ax.set_xlabel('# Documents', fontsize=fs2)
ax.set_ylabel('', fontsize=fs2)

# Font of the axes labels
ax.tick_params(axis='x', labelsize=fs2)
ax.tick_params(axis='y', labelsize=fs2)

# Axes style
ax.spines['bottom'].set_linestyle('-')
ax.spines['left'].set_linestyle('-')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Vertical dotted grid
ax.grid(axis='x', linestyle=':', linewidth=1.2)
ax.grid(axis='y', visible=False)

# Legend below the plot
handles, labels = ax.get_legend_handles_labels()
new_labels = [simplify_label(label) for label in labels]
ax.legend(
    handles, new_labels,
    title='',
    fontsize=fs2,
    title_fontsize=fs2,
    # for centered legend at the bottom
    loc='lower center',
    bbox_to_anchor=(0.3, -0.35),
    # # for legend on the right, need to change the width of the image
    # loc='center left',
    # bbox_to_anchor=(1.01, 0.5),
    # for legend on the bottom left
    facecolor='white',
    ncol=2
)

# Adjust space to reduce the height of the plot and free up space for the legend
fig.subplots_adjust(top=0.31, bottom=0.1, left=0.01, right=0.99)
plt.tight_layout()



# Add labels with a percentage alternating between above and below the bars, per row
row_totals = pivot_df.sum(axis=1)

# Dictionary to control alternation per row (journal)
toggle_per_row = {}

for container in plot.containers:
    for rect in container:
        width = rect.get_width()
        if width > 1:  # avoids very small values
            y_center = rect.get_y() + rect.get_height() / 2
            y_index = int(y_center)
            journal_name = plot.get_yticklabels()[y_index].get_text()
            total = row_totals[journal_name]
            percent = (width / total) * 100

            # Initialize alternator for the journal, if it doesn't exist
            if journal_name not in toggle_per_row:
                toggle_per_row[journal_name] = True
            
            x = rect.get_x() + width / 2
            
            if toggle_per_row[journal_name]:
                y = rect.get_y() + rect.get_height() + 0.05  # above the bar
                va = 'bottom'
            else:
                y = rect.get_y() - 0.05  # below the bar
                va = 'top'

            ax.text(
                x, y,
                f'{percent:.0f}%',
                ha='center',
                va=va,
                fontsize=fs3,
                color='black'
            )
            
            # Alternates for the next bar in that row
            toggle_per_row[journal_name] = not toggle_per_row[journal_name]

  

plt.savefig('q3_2_3/q3_2_3_top_journal_topic_v2.pdf', bbox_inches='tight') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Alternative 2

from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image

mypath = 'q3_2_3/'

# Font
fs1, fs2, fs2_5, fs3 = 70, 46, 46, 46
fs_legenda = 42
largura = 30
altura = 28

# Data processing
top_journals = (
    df_data['journal_abrev']
    .value_counts()
    .nlargest(10)
    .index
)

df_top_journals = df_data[df_data['journal_abrev'].isin(top_journals)].copy()
# df_top_journals["journal_abrev"] = df_top_journals["journal_abrev"].apply(
# lambda x: x.replace('INTERNATIONAL', 'INT.').replace("JOURNAL", "J.").replace("&", "&")
# )

grouped = (
    df_top_journals
    .groupby(['journal_abrev', 'topic_name_cluster3_reduced'])
    .size()
    .reset_index(name='count')
)

def get_top_topics_per_journal(df):
    return (
        df.sort_values(['journal_abrev', 'count'], ascending=[True, False])
          .groupby('journal_abrev')
          .head(5)
    )

top_topics = get_top_topics_per_journal(grouped)

pivot_df = (
    top_topics
    .pivot_table(index='journal_abrev', columns='topic_name_cluster3_reduced', values='count', fill_value=0)
)
pivot_df = pivot_df.loc[pivot_df.sum(axis=1).sort_values(ascending=True).index]

def simplify_label(label):
    parts = str(label).split('_')
    new_label = parts[1] if len(parts) > 1 else label
    new_label = re.sub(r'(\D)(\d+)', lambda m: f"{m.group(1)}$_{m.group(2)}$", new_label)
    return new_label

row_totals = pivot_df.sum(axis=1)

import tempfile
import matplotlib.patches as mpatches

def plot_single_journal(journal_name, data_series, colors_dict):
    """
    Creates a bar chart with the top 5 topics for a single journal
    and saves it to a temporary image file.
    """
    # Create the figure
    fig, ax = plt.subplots(figsize=(largura, altura/10), facecolor='white')
    
    # Filter the topics with count > 0 for this journal
    data = data_series[data_series > 0].copy()
    
    # Needs to follow the original order of the topics (from largest to smallest)
    # The original order was defined in `top_topics`
    journal_top_topics = top_topics[top_topics['journal_abrev'] == journal_name]
    ordered_topics = journal_top_topics['topic_name_cluster3_reduced'].tolist()
    ordered_counts = journal_top_topics['count'].tolist()
    
    # Calculate percentages
    total = sum(ordered_counts)
    percentages = [(count / total) * 100 for count in ordered_counts]
    
    # Build stacked bar
    left = 0
    for i, (topic, count, pct) in enumerate(zip(ordered_topics, ordered_counts, percentages)):
        color = colors_dict[topic]
        ax.barh(0, count, left=left, height=0.5, color=color, edgecolor='none')
        
        # Determine the label position (above or below, alternating)
        y_pos = 0.35 if i % 2 == 0 else -0.35
        va = 'bottom' if i % 2 == 0 else 'top'
        
        # Add the percentage label
        ax.text(
            left + count / 2, y_pos, 
            f'{pct:.0f}%', 
            ha='center', va=va, 
            fontsize=fs3, color='black'
        )
        
        left += count
        
    # Axis settings
    ax.set_yticks([0])
    ax.set_yticklabels([journal_name], fontsize=fs2)
    ax.tick_params(axis='x', labelsize=fs2)
    ax.set_xlabel('# Documents' if journal_name == pivot_df.index[0] else '', fontsize=fs2_5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Define x limits based on the total sum to align properly across all charts
    ax.set_xlim(0, pivot_df.sum(axis=1).max() * 1.05)
    ax.grid(axis='x', linestyle=':', linewidth=1.2)
    
    plt.tight_layout()
    
    # Save to a temporary file
    temp_file = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    plt.savefig(temp_file.name, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    return temp_file.name

# 1. Create a dictionary of colors for all unique topics
all_topics = pivot_df.columns.tolist()
cmap = plt.get_cmap('tab20')
colors_dict = {topic: cmap(i % 20) for i, topic in enumerate(all_topics)}

# 2. Generate charts for each journal
temp_images = []
for journal_name in pivot_df.index:
    img_path = plot_single_journal(journal_name, pivot_df.loc[journal_name], colors_dict)
    temp_images.append(img_path)

# 3. Create legend image separately
fig_leg, ax_leg = plt.subplots(figsize=(largura, 4), facecolor='white')
ax_leg.axis('off')

# Extract unique labels from the data for the legend
handles = [mpatches.Patch(color=colors_dict[t], label=simplify_label(t)) for t in all_topics]
ax_leg.legend(handles=handles, loc='center', ncol=2, fontsize=fs_legenda, frameon=False)

temp_leg = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
plt.savefig(temp_leg.name, dpi=300, bbox_inches='tight')
plt.close(fig_leg)

# 4. Combine all images vertically
images = [Image.open(img) for img in reversed(temp_images)] # reversed to have the largest at the top
leg_img = Image.open(temp_leg.name)

# Calculate final dimensions
total_height = sum(img.height for img in images) + leg_img.height
max_width = max(img.width for img in images + [leg_img])

final_image = Image.new('RGB', (max_width, total_height), 'white')

# Paste images
y_offset = 0
for img in images:
    final_image.paste(img, (0, y_offset))
    y_offset += img.height

final_image.paste(leg_img, (0, y_offset))

# Save the final result
final_image.save(f'{mypath}q3_2_3_top_journal_topic_v3.pdf', resolution=300)

print(f"File saved in {mypath}q3_2_3_top_journal_topic_v3.pdf")

# Clean temporary files
import os
for f in temp_images + [temp_leg.name]:
    os.remove(f)

# plt.show()

In [ ]:
# Execute data processing or visualization steps
# ##############################################################################################
# plots by institution

import matplotlib.patheffects as path_effects

# Size of the fonts
fs1 = 22
fs2 = 18
fs3 = 16

# Ensure that the data are in the correct format
df_data['affiliation'] = df_data['affiliation'].astype(str)
df_data['topic_name'] = df_data['topic_name'].astype(str)
df_data["frst_inst"] = df_data["affiliation"].apply(
    lambda x: x.split(";")[0].strip().replace("\&", "&") if isinstance(x, str) and x.strip() != "" else ""
)

# Select the top 10 journals
top_journal_counts = df_data['frst_inst'].value_counts().nlargest(10)
top_journals = top_journal_counts.index

# Create grid of subplots: 6 rows (1 for general pie + 5 for 2x5 topics)
fig = plt.figure(figsize=(16, 23))
gs = fig.add_gridspec(6, 2,  height_ratios=[1.2, 1, 1, 1, 1, 1])

# === Top row: Journals pie ===
ax_top = fig.add_subplot(gs[0, :])  # Occupies the two columns
wedges, texts, autotexts = ax_top.pie(
    top_journal_counts.values,
    labels=None,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': fs3},
    pctdistance=1.05  # percentage distance to the outside of the pie
)
# Apply white outline in the percentage texts
for text in autotexts:
    text.set_path_effects([
        path_effects.Stroke(linewidth=2, foreground='white'),
        path_effects.Normal()
    ])
ax_top.set_title("Topics by Institution (top 10 Institutions, top 5 topics)", fontsize=fs1)
ax_top.legend(
    wedges,
    top_journal_counts.index,
    title="",
    # loc='upper center',
    # bbox_to_anchor=(0.5, -0.1),
    # ncol=3,
    loc='center left',
    ncol=1, 
    bbox_to_anchor=(1, 0.5),
    fontsize=fs3
)

# === Plots by journal (5 rows x 2 columns) ===
for i, journal in enumerate(top_journals):
    row = (i // 2) + 1  # Starts from row 1
    col = i % 2
    ax = fig.add_subplot(gs[row, col])

    # Journal data
    df_journal = df_data[df_data['frst_inst'] == journal]
    topic_counts = df_journal['topic_name'].value_counts().nlargest(5)
    labels = [label.split("_")[1][:30]+"..." for label in topic_counts.index]

    wedges, texts, autotexts= ax.pie(
        topic_counts.values,
        labels=None,
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': fs3},
        pctdistance=1.05  # percentage distance to the outside of the pie
    )
    ax.set_xlim(-1.5, 2)  # shifts the pie horizontally to the left
    # Apply white outline in the percentage texts
    for text in autotexts:
        v = 0.85
        text.set_path_effects([
            path_effects.Stroke(linewidth=2, foreground=(v, v, v)),
            path_effects.Normal()
        ])

    ax.legend(
        wedges,
        labels,
        title="",
        loc='center left',
        ncol=1, 
        bbox_to_anchor=(0.75, 0.5),
        fontsize=fs3
    )
    jrn = journal.replace('INTERNATIONAL', 'INT.').replace("JOURNAL", "J.").replace("\&", "&")
    ax.set_title(f"{jrn}\n({len(df_journal)} docs)", fontsize=fs2, loc = "right")

fig.subplots_adjust(hspace=0.35)
plt.tight_layout()



plt.savefig('q3_2_3/q3_2_3_top_institutions_topic.pdf', bbox_inches='tight') #, dpi=300)

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Alternative 2 of the plot by institution

from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image

mypath = 'q3_2_3/'

# Font
fs1, fs2, fs2_5, fs3 = 70, 44, 44, 36
fs_legenda = 40
largura = 30
altura = 28

# Data processing
top_journals = (
    df_data['first_institution']
    .value_counts()
    .nlargest(10)
    .index
)

df_top_journals = df_data[df_data['first_institution'].isin(top_journals)].copy()
df_top_journals["first_institution"] = df_top_journals["first_institution"].apply(
    lambda x: x.replace('Institute', 'Inst.').replace("(IIT System)", "").replace("&", "&").replace(" - China", "")
)

grouped = (
    df_top_journals
    .groupby(['first_institution', 'topic_name_cluster3_reduced'])
    .size()
    .reset_index(name='count')
)

def get_top_topics_per_journal(df):
    return (
        df.sort_values(['first_institution', 'count'], ascending=[True, False])
          .groupby('first_institution')
          .head(5)
    )

top_topics = get_top_topics_per_journal(grouped)

pivot_df = (
    top_topics
    .pivot_table(index='first_institution', columns='topic_name_cluster3_reduced', values='count', fill_value=0)
)
pivot_df = pivot_df.loc[pivot_df.sum(axis=1).sort_values(ascending=True).index]

def simplify_label(label):
    parts = str(label).split('_')
    new_label = parts[1] if len(parts) > 1 else label
    new_label = re.sub(r'(\D)(\d+)', lambda m: f"{m.group(1)}$_{m.group(2)}$", new_label)
    return new_label

# ========== Step 1: plot without legend ==========
fig1, ax1 = plt.subplots(figsize=(largura, altura), facecolor='white')
plot = pivot_df.plot(
    kind='barh', stacked=True, colormap='tab20', ax=ax1, width=0.3
)

ax1.set_facecolor('white')
ax1.set_title('Topics by Institution (top 10 institutions, top 5 topics)', fontsize=fs1,loc='right')
ax1.set_xlabel('No of Documents', fontsize=fs2)
ax1.set_ylabel('', fontsize=fs2)
ax1.tick_params(axis='x', labelsize=fs2)
ax1.tick_params(axis='y', labelsize=fs2)
ax1.spines['bottom'].set_linestyle('-')
ax1.spines['left'].set_linestyle('-')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='x', linestyle=':', linewidth=1.2)
ax1.grid(axis='y', visible=False)

# Remove the legend
ax1.get_legend().remove()

# Add labels of percentage
row_totals = pivot_df.sum(axis=1)
toggle_per_row = {}

for container in plot.containers:
    for rect in container:
        width = rect.get_width()
        if width > 1:
            y_center = rect.get_y() + rect.get_height() / 2
            y_index = int(y_center)
            journal_name = plot.get_yticklabels()[y_index].get_text()
            total = row_totals[journal_name]
            percent = (width / total) * 100
            if journal_name not in toggle_per_row:
                toggle_per_row[journal_name] = True
            x = rect.get_x() + width / 2
            y = rect.get_y() + rect.get_height() + 0.05 if toggle_per_row[journal_name] else rect.get_y() - 0.05
            va = 'bottom' if toggle_per_row[journal_name] else 'top'
            ax1.text(x, y, f'{percent:.0f}%', ha='center', va=va, fontsize=fs3, color='black')
            toggle_per_row[journal_name] = not toggle_per_row[journal_name]

fig1.tight_layout()
fig1.savefig(f"{mypath}q3_2_3_top_institutions_topic_v3_img1.jpg", bbox_inches='tight')

# ========== Step 2: isolated legend ==========
fig2, ax2 = plt.subplots(figsize=(largura, 8), facecolor='white')
handles, labels = plot.get_legend_handles_labels()
new_labels = [simplify_label(label) for label in labels]
ax2.legend(
    handles, new_labels,
    loc='center', ncol=2,
    fontsize=fs_legenda, frameon=False
)
ax2.axis('off')
fig2.savefig(f"{mypath}q3_2_3_top_institutions_topic_v3_img2.jpg", bbox_inches='tight')

# ========== Step 3: image combination ==========
img1 = Image.open(f"{mypath}q3_2_3_top_institutions_topic_v3_img1.jpg")
img2 = Image.open(f"{mypath}q3_2_3_top_institutions_topic_v3_img2.jpg")

# Resize legend to same width of the plot
img2 = img2.resize((img1.width, img2.height+100))
combined_height = img1.height + img2.height
combined_img = Image.new('RGB', (img1.width, combined_height), color='white')
combined_img.paste(img1, (0, 0))
combined_img.paste(img2, (0, img1.height))

combined_img.save(f"{mypath}q3_2_3_top_institutions_topic_v3_final.jpg")

In [ ]:
# Execute data processing or visualization steps
# ##############################################################################################
# table by institution

import pandas as pd
from tabulate import tabulate

col_topic_name = "topic_name_cluster3_reduced"

# Ensure that columns are strings
df_data[col_topic_name] = df_data[col_topic_name].astype(str)



# Top 10 journals by number of documents
top_journals = df_data['first_institution'].value_counts().nlargest(10)

# List to store the rows of the table
table_rows = []

# Generate the data of the table
for journal in top_journals.index:
    jrn = journal.replace('INTERNATIONAL', 'INT.').replace("JOURNAL", "J.").replace("\&", "&")
    df_journal = df_data[df_data['first_institution'] == journal]
    journal_total = len(df_journal)

    # Add row of the journal (with total)
    table_rows.append([jrn, journal_total, '', ''])

    # Top 5 topics of the journal
    top_topics = df_journal[col_topic_name].value_counts().nlargest(5)
    for topic, count in top_topics.items():
        table_rows.append(['', '', topic, count])

# Create DataFrame of the table
df_table = pd.DataFrame(table_rows, columns=["Institution (Top 10)", "Total Docs", "Topics (Top 5)", "Total Docs"])

# Export as LaTeX using tabulate
latex_table = tabulate(df_table, headers='keys', tablefmt='LaTeX', showindex=False)

with open("q3_2_3/q3_2_3_top_institution_topic_table.tex", "w") as f:
    f.write(latex_table)

# Display or save
print(latex_table)

In [ ]:
# Execute data processing or visualization steps
import plotly.express as px

# 1. Count how many documents there are by journal and topic
df_counts = df_data.groupby(['journal', 'topic_name']).size().reset_index(name='count')

# 2. Select the top 10 journals
top_journals = df_counts.groupby('journal')['count'].sum().nlargest(10).index
df_counts = df_counts[df_counts['journal'].isin(top_journals)]

# 3. For each journal, keep only the top 5 topics
top_topics_per_journal = (
    df_counts.groupby('journal', group_keys=False)
    .apply(lambda g: g.nlargest(5, 'count'))
)

# 4. Interactive plot with Plotly
fig = px.sunburst(
    top_topics_per_journal,
    path=['journal', 'topic_name'],
    values='count',
    title='Top 5 topics in Each of the Top 10 Journals',
    width=800,
    height=800
)

fig.update_traces(textinfo='label+percent entry')
fig.show()

In [ ]:
# Execute data processing or visualization steps
x = df_data[df_data["year"].astype(int) < 2000]
# x.shape
x.to_csv("papers_up_to_1999.csv")

### Question S-3.4
Are there emerging research centers or countries that are increasing their contributions?56


In [ ]:
# Execute data processing or visualization steps
# based on the data of first author

fs1 = 50
fs2 = 34
fs3 = 28


# Filter years of interest
df_filtered = df_data[(df_data['year'] >= 2014) & (df_data['year'] <= 2025)]

# Grouping by institution and year
df_counts = (
    df_filtered
    .groupby(['first_institution', 'year'])
    .size()
    .reset_index(name='count')
)

# Top 10 most productive institutions (2014–2025)
top10_inst = (
    df_counts.groupby('first_institution')['count']
    .sum()
    .nlargest(20)
    .reset_index()
)

# Sort institutions based on the total - from largest to smallest
ordered_inst = top10_inst.sort_values('count', ascending=True)['first_institution'].tolist()

# Filter plot data
df_top10 = df_counts[df_counts['first_institution'].isin(ordered_inst)]

# Ensure correct order with ordered category
df_top10['first_institution'] = pd.Categorical(df_top10['first_institution'], categories=ordered_inst, ordered=True)

# Sort the DataFrame based on the desired order
df_top10 = df_top10.sort_values('first_institution')

# Create label column
df_top10['label'] = df_top10['count'].apply(lambda x: str(x) if x > 50 else "")
df_top10["first_institution"] = df_top10["first_institution"].apply(lambda x: x.replace('Institute', 'Inst.').replace("(IIT System)", "").replace("&", "&").replace(" - China", "").replace("(CSIR)", "").replace(" - India", ""))

# Plot
fig = px.scatter(
    df_top10,
    x='year',
    y='first_institution',
    size='count',
    # text='label',
    size_max=30,
    color_discrete_sequence=['salmon'] * len(df_top10),
)

# Visual adjustments
fig.update_traces(
    textposition='middle center',
    marker=dict(line=dict(width=1, color='black')),
    textfont=dict(size=fs3, color='black')
)

fig.update_layout(
    title=dict(
        text="Publications by First-Author Institution (2014–2025)",
        font=dict(size=fs1, color='black'),
        x=0.5,  # Centers the title horizontally
        xanchor='center',
        yanchor='top'
    ),
    margin=dict(t=100),  # Increases top margin
    xaxis_title="Year",
    xaxis=dict(
        title=dict(text="", font=dict(size=fs2)),   # X axis title
        tickfont=dict(size=fs2, color='black'),                      # Ticks (years)
        dtick=1,
        tickangle=-90,
        linecolor='black',             # X axis line color
        linewidth=2,                  # X axis line thickness
        showgrid=True,                # Activates X axis grid
        gridcolor='black',        # Grid color
        gridwidth=1,                  # Grid thickness
        griddash='dot',                # Dotted style

    ),
    yaxis=dict(
        title=dict(text="", font=dict(size=fs2)),
        tickfont=dict(size=fs2, color='black'),
        showline=True,
        linecolor='black',
        linewidth=2,
        showgrid=True,
        gridcolor='black',
        gridwidth=1,
        griddash='dot'
        # Without 'categoryorder' here
    ), 
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    height=1500*0.95,
    width =1700*0.80,
)


fig.write_image("q3_2_4/q3_2_4_pub_by_institutions_year.pdf") 

fig.show()

In [ ]:
# Execute data processing or visualization steps
# based on the data of the first author

# Pivot the DataFrame: institutions on the rows, years in the columns
df_pivot = df_top10.pivot_table(
    index='first_institution',
    columns='year',
    values='count',
    fill_value=0  # fills empty cells with 0
)

# Ensure that all values are integers (no decimal places)
df_pivot = df_pivot.astype(int)

# Sort columns by year
df_pivot = df_pivot[sorted(df_pivot.columns)]

# Optional: rename the index (row name)
df_pivot.index.name = "Institution"

# Generate LaTeX
latex_table = df_pivot.to_latex(
    index=True,
    caption="Publications per year for the top 20 most productive institutions (2014--2025).",
    label="tab:q3_2_4_pub_by_institutions_year",
    longtable=False,
    escape=False,
    bold_rows=False
)

# Save to file
with open("q3_2_4/q3_2_4_pub_by_institutions_year.tex", "w") as f:
    f.write(latex_table)

print(latex_table)  # Display the LaTeX table in the console

In [ ]:
# Execute data processing or visualization steps
# based on the data of the last author

fs1 = 50
fs2 = 34
fs3 = 28


# Filter years of interest
df_filtered = df_data[(df_data['year'] >= 2014) & (df_data['year'] <= 2025) & (df_data['last_institution'] != "")]

# Grouping by institution and year
df_counts = (
    df_filtered
    .groupby(['last_institution', 'year'])
    .size()
    .reset_index(name='count')
)

# Top 10 most productive institutions (2014–2025)
top10_inst = (
    df_counts.groupby('last_institution')['count']
    .sum()
    .nlargest(20)
    .reset_index()
)

# Sort institutions based on the total - from largest to smallest
ordered_inst = top10_inst.sort_values('count', ascending=True)['last_institution'].tolist()

# Filter plot data
df_top10 = df_counts[df_counts['last_institution'].isin(ordered_inst)]

# Ensure correct order with ordered category
df_top10['last_institution'] = pd.Categorical(df_top10['last_institution'], categories=ordered_inst, ordered=True)

# Sort the DataFrame based on the desired order
df_top10 = df_top10.sort_values('last_institution')

# Create label column
df_top10['label'] = df_top10['count'].apply(lambda x: str(x) if x > 50 else "")
df_top10["last_institution"] = df_top10["last_institution"].apply(lambda x: x.replace('Institute', 'Inst.').replace("(IIT System)", "").replace("\\&", "&").replace(" - China", "").replace("(CSIR)", "").replace(" - India", ""))

# Plot
fig = px.scatter(
    df_top10,
    x='year',
    y='last_institution',
    size='count',
    text='label',
    size_max=40,
    color_discrete_sequence=['salmon'] * len(df_top10),
)

# Visual adjustments
fig.update_traces(
    textposition='middle center',
    marker=dict(line=dict(width=1, color='black')),
    textfont=dict(size=fs3, color='black')
)

fig.update_layout(
    title=dict(
        text="Publications by Last Author's Institution (2014–2025)",
        font=dict(size=fs1, color='black'),
        x=0.5,  # Centers the title horizontally
        xanchor='center',
        yanchor='top'
    ),
    margin=dict(t=100),  # Increases top margin
    xaxis_title="Year",
    xaxis=dict(
        title=dict(text="", font=dict(size=fs2, color='black')),   # X axis title
        tickfont=dict(size=fs2, color='black'),                       # Ticks (years)
        dtick=1,
        tickangle=-45,
        linecolor='black',             # X axis line color
        linewidth=2,                  # X axis line thickness
        showgrid=True,                # Activates X axis grid
        gridcolor='black',        # Grid color
        gridwidth=1,                  # Grid thickness
        griddash='dot'                # Dotted style

    ),
    yaxis=dict(
        title=dict(text="", font=dict(size=fs2, color='black')),
        tickfont=dict(size=fs2, color='black'),
        showline=True,
        linecolor='black',
        linewidth=2,
        showgrid=True,
        gridcolor='black',
        gridwidth=1,
        griddash='dot'
        # Without 'categoryorder' here
    ), 
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    height=1500,
    width =1700,
)


fig.write_image("q3_2_4/q3_2_4_pub_by_last_institutions_year.pdf") 

fig.show()

In [ ]:
# Execute data processing or visualization steps
# based on the data of the last author

# Pivot the DataFrame: institutions on the rows, years in the columns
df_pivot = df_top10.pivot_table(
    index='last_institution',
    columns='year',
    values='count',
    fill_value=0  # fills empty cells with 0
)

# Ensure that all values are integers (no decimal places)
df_pivot = df_pivot.astype(int)

# --- START OF SORTING MODIFICATION ---

# 1. Create the 'Total' column to calculate the sum of publications for each institution
df_pivot['Total'] = df_pivot.sum(axis=1)

# 2. Sort the DataFrame by the 'Total' column in descending order
# The parameter ascending=False reverses the order (from largest to smallest)
df_sorted = df_pivot.sort_values(by='Total', ascending=False)

# 3. Remove the 'Total' column, as it was only an auxiliary step for sorting
# and should not appear in the final table.
df_final = df_sorted.drop(columns='Total')

# --- END OF MODIFICATION ---


# Sort columns by year (this step remains the same)
df_final = df_final[sorted(df_final.columns)]

# Optional: rename the index (row name)
df_final.index.name = "Institution"

# Generate LaTeX using the sorted final DataFrame (df_final)
latex_table = df_final.to_latex(
    index=True,
    caption="Publications per year for the top 20 most productive Last Authors's institutions (2014--2025).",
    label="tab:q3_2_4_pub_by_last_institutions_year",
    longtable=False,
    escape=False,
    bold_rows=False
)
# Save to file
with open("q3_2_4/q3_2_4_pub_by_last_institutions_year.tex", "w") as f:
    f.write(latex_table)

print(latex_table)  # Display the LaTeX table in the console

In [ ]:
# Execute data processing or visualization steps
import plotly.graph_objects as go
import plotly.express as px

# # calculate the relevance index of the institution
# # Function to calculate the relevance index of a topic
# year_stats = (
# df_data.groupby("year")["citations"]
# .agg(["mean", "std"])
# .rename(columns={"mean": "year_mean", "std": "year_std"})
# .reset_index()
# )
# current_year = 2025
# def compute_topic_index(topic_df):
# df = topic_df.copy()
# df = df[df["citations"].notna() & df["year"].notna()]
    
# # Calculate citations by year
# df["citations_per_year"] = df["citations"] / (current_year - df["year"] + 1 + 0.25)  # 0.25 to consider only the 3 first months of 2025

# # Merge with statistics of the year
# df = df.merge(year_stats, on="year", how="left")
    
# # Z-score by year
# df["z_score"] = (df["citations"] - df["year_mean"]) / df["year_std"]
# df["z_score"] = df["z_score"].fillna(0)

# # Identify top 10% by year
# top_percentile_flags = []
# for year in df["year"].unique():
# year_notes = df_data[df_data["year"] == year]["citations"]
# if len(year_notes) >= 10:
# threshold = np.percentile(year_notes, 90)
# top_percentile_flags.extend(df[df["year"] == year]["citations"] >= threshold)
# else:
# top_percentile_flags.extend([False] * sum(df["year"] == year))
# df["top_10_percent"] = top_percentile_flags[:len(df)]

# # Calculation of the composite index
# mean_cpy = df["citations_per_year"].mean()
# mean_z = df["z_score"].mean()
# prop_top10 = df["top_10_percent"].mean()
# n_docs = len(df)

# # Composite index (weighted)
# index = (
# 0.5 * mean_cpy +
# 0.3 * prop_top10 +
# 0.2 * np.log1p(n_docs)
# )

# return {
# "n_documents": n_docs,
# "mean_citations_per_year": mean_cpy,
# "mean_z_score": mean_z,
# "prop_top_10_percent": prop_top10,
# "composite_index": index
# }

# # Apply to all topics
# instituicoes = list(set(df_data["frst_inst"].to_list()))
# years = list(set(df_data["year"].to_list()))
# relevance_data = []
# for inst in tqdm(instituicoes):
# for year in years:
# df = df_data[ (df_data["frst_inst"] == inst) & (df_data["year"] == year)]
# idx = compute_topic_index(df)
# relevance_data.append((inst, year, idx["composite_index"]))



# # Transform to DataFrame
# df_relevance = pd.DataFrame(relevance_data, columns=['frst_inst', 'year', 'relevance'])

# df_relevance.to_pickle("df_relavance.pkl")
df_relevance = pd.read_pickle("df_relavance.pkl")

# Ensure that the types are correct
df_data['year'] = df_data['year'].astype(int)
df_relevance['year'] = df_relevance['year'].astype(int)

# Filter years
df_filtered = df_data[df_data['year'].between(2015, 2025)]

# Count of publications by inst/year
pubs = df_filtered.groupby(['frst_inst', 'year']).size().reset_index(name='pub_count')

# Join with relevance index
df_plot = pd.merge(pubs, df_relevance, on=['frst_inst', 'year'], how='left')

# Select top 10 institutions by total publications
top_insts = df_plot.groupby('frst_inst')['pub_count'].sum().nlargest(10).index.tolist()
df_plot = df_plot[df_plot['frst_inst'].isin(top_insts)]

# Sort y axis
df_plot['frst_inst'] = pd.Categorical(
    df_plot['frst_inst'],
    categories=df_plot.groupby('frst_inst')['pub_count'].sum().sort_values(ascending=False).index,
    ordered=True
)

# Fonts
font_title = 30
font_axis = 24
font_labels = 20

# Create the figure
fig = go.Figure()

relevance_min = df_plot['relevance'].min()
relevance_max = df_plot['relevance'].max()

for i, inst in enumerate(df_plot['frst_inst'].cat.categories):
    df_inst = df_plot[df_plot['frst_inst'] == inst]
    fig.add_trace(go.Scatter(
        x=df_inst['year'],
        y=[inst] * len(df_inst),
        mode='markers+text',
        marker=dict(
            size=df_inst['pub_count']*0.5,
            color=df_inst['relevance'],
            colorscale='YlGnBu',
            cmin=relevance_min,
            cmax=relevance_max,
            colorbar=dict(
                title='Relevance Index',
                title_font=dict(size=font_axis),
                tickfont=dict(size=font_labels)
            ) if i == 0 else None,
            line=dict(width=1, color='DarkSlateGrey')
        ),
        text=[
            str(p) if p > 100 else '' for p in df_inst['pub_count']
        ],
        textposition='middle center',
        textfont=dict(size=font_labels),
        hovertemplate=(
            "Institution: %{y}<br>" +
            "Year: %{x}<br>" +
            "Publications: %{marker.size}<br>" +
            "Relevance: %{marker.color:.2f}<extra></extra>"
        ),
        showlegend=False
    ))

# Layout
fig.update_layout(
    width=1500,
    height=900,
    title=dict(text="Publications and Relevance by Institution (2014–2025)", font_size=font_title),
    xaxis=dict(
        title='',
        tickangle=45,
        title_font_size=font_axis,
        showline=True,
        linecolor='black',
        gridcolor='lightgray',
        griddash='dot'
    ),
    yaxis=dict(
        title='',
        title_font_size=font_axis,
        showline=True,
        linecolor='black',
        gridcolor='light

### Question S-3.5
What are the most active authors based on publication frequency and citations?

In [ ]:
# Execute data processing or visualization steps
# spider chart of first author

# Font sizes
fs1 = 50  # title
fs2 = 38  # legend and labels
fs3 = 30  # values on points

label_offset = 0.12  # increased label distance

# --- Grouping and metrics ---
df_authors = df_data.copy()
df_authors['first_author'] = df_authors['first_author'].str.strip().str.title()

agg_df = df_authors.groupby('first_author').agg(
    num_publications=('title', 'count'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean'),
    first_country=('first_country', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
).reset_index()

agg_df['perc_publications'] = agg_df['num_publications'] / agg_df['num_publications'].sum() * 100

top_n = 5
top_authors = agg_df.sort_values(by='num_publications', ascending=False).head(top_n).reset_index(drop=True)

# --- Robust normalization ---
def robust_minmax(series, lower=5, upper=95):
    q_min = np.percentile(series, lower)
    q_max = np.percentile(series, upper)
    return (series - q_min) / (q_max - q_min + 1e-9), q_min, q_max

features = ['num_publications', 'total_citations', 'mean_citations', 'perc_publications']
top_authors_scaled = top_authors[features].copy()
scaling_params = {}

for col in features:
    scaled, q_min, q_max = robust_minmax(top_authors[col])
    top_authors_scaled[col] = scaled
    scaling_params[col] = (q_min, q_max)

# --- Radar chart setup ---
labels = ['# publications', 'citations', 'mean citations', '% publications']
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# --- 1) CALCULATE and STORE label positions ---

label_positions = {}

for i, row in top_authors_scaled.iterrows():
    values_original = top_authors.loc[i, features].tolist()
    values_original += values_original[:1]
    label = top_authors['first_author'][i]
    label_positions[label] = []

    for angle, feat, orig_val in zip(angles, features + [features[0]], values_original):
        q_min, q_max = scaling_params[feat if feat != features[0] else features[0]]
        scaled_val = (orig_val - q_min) / (q_max - q_min + 1e-9)

        r_label = scaled_val + label_offset
        label_positions[label].append((angle, r_label))

# --- Custom rotation mapping by feature ---
rotation_map = {
    'perc_publications': 90,
    'total_citations': -90
}

# --- 2) PLOT using stored positions ---

fig, ax = plt.subplots(figsize=(18, 18), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for i, row in top_authors_scaled.iterrows():
    values_scaled = row.tolist() + row.tolist()[:1]
    label = top_authors['first_author'][i]

    ax.plot(angles, values_scaled, label=label)
    ax.fill(angles, values_scaled, alpha=0.1)

    for idx, ((angle, r_label), feat, orig_val) in enumerate(zip(label_positions[label], features + [features[0]], top_authors.loc[i, features].tolist() + [top_authors.loc[i, features[0]]])):
        q_min, q_max = scaling_params[feat]
        scaled_val = (orig_val - q_min) / (q_max - q_min + 1e-9)

        ax.plot([angle, angle], [scaled_val, r_label], color='gray', linewidth=1)

        rotation = rotation_map.get(feat, 0)
        # Modify here:
        if feat in ['num_publications', 'total_citations']:
            display_val = f'{int(orig_val)}'  # no decimal
        elif feat == 'perc_publications':
            display_val = f'{orig_val:.3f}'  # no decimal
        else:
            display_val = f'{orig_val:.1f}'    # 1 decimal place

        ax.text(angle, r_label, display_val, ha='center', va='center',
                fontsize=fs3, rotation=rotation)

# --- Styling the plot ---
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# --- Styling ---
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Remove the '% publications' and 'citations' grid labels
custom_labels = ['# publications', '', 'mean citations', '']
ax.set_thetagrids(np.degrees(angles[:-1]), custom_labels, fontsize=fs2)

# Force the drawing to obtain correct limits
plt.draw()

# --- Manually add rotated text boxes in the right places ---

# Angular position of removed labels
angle_dict = dict(zip(labels, angles[:-1]))

# Text box for '% publications'
angle_perc = angle_dict['% publications']
ax.text(angle_perc, 1.35, '% publications',
        fontsize=fs2, rotation=90,
        ha='center', va='center')

# Text box for 'citations'
angle_cit = angle_dict['citations']
ax.text(angle_cit, 1.35, 'citations',
        fontsize=fs2, rotation=-90,
        ha='center', va='center')

# Title
ax.set_title('Top 5 Authors by number of publications', fontsize=fs1, pad=30)

# Remove the radial axis labels
ax.set_yticklabels([])

# Legend
legend = ax.legend(loc='lower right', bbox_to_anchor=(1.25, -0.1), fontsize=fs2)
legend.get_frame().set_facecolor('white')

plt.tight_layout()


plt.savefig("q3_2_5/q3_2_5_top5_authors_metrics.pdf") 

plt.show()

In [ ]:
# Execute data processing or visualization steps
# table of the plot above, but with top 10 authors

# --- Grouping and metrics ---
df_authors = df_data.copy()
df_authors['first_author'] = df_authors['first_author'].str.strip().str.title()

agg_df = df_authors.groupby('first_author').agg(
    num_publications=('title', 'count'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean'),
    first_country=('first_country', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    first_institution=('first_institution', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
).reset_index()

# agg_df['perc_publications'] = agg_df['num_publications'] / agg_df['num_publications'].sum() * 100

top_n = 20
top_authors = agg_df.sort_values(by='num_publications', ascending=False).head(top_n).reset_index(drop=True)

latex = top_authors.to_latex(
    index=False,
    float_format="%.2f",
    column_format="lcccccc",
    header=["Author", "# Publications", "Total Citations", "Mean Citations", "First Country", "First Institution" ]#"% Publications"]
)
with open("q3_2_5/q3_2_5_top10_authors_metrics.tex", "w") as f:
    f.write(latex)
    
# Display or save
print(latex)

In [ ]:
# Execute data processing or visualization steps
author_counts = df_data['first_author'].value_counts()
author_counts

In [ ]:
# Execute data processing or visualization steps
# spider chart of corresponding author

# Font sizes
fs1 = 50  # title
fs2 = 38  # legend and labels
fs3 = 30  # values on points

label_offset = 0.12  # increased label distance

# --- Grouping and metrics ---
df_authors = df_data.copy()
df_authors['first_author'] = df_authors['corresp_author'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ''
)
df_authors['first_country'] = df_authors['corresp_country'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ''
)
df_authors['first_author'] = df_authors['first_author'].str.strip().str.title()

agg_df = df_authors.groupby('first_author').agg(
    num_publications=('title', 'count'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean'),
    first_country=('first_country', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
).reset_index()

agg_df['perc_publications'] = agg_df['num_publications'] / agg_df['num_publications'].sum() * 100

top_n = 5
top_authors = agg_df.sort_values(by='num_publications', ascending=False).head(top_n).reset_index(drop=True)

# --- Robust normalization ---
def robust_minmax(series, lower=5, upper=95):
    q_min = np.percentile(series, lower)
    q_max = np.percentile(series, upper)
    return (series - q_min) / (q_max - q_min + 1e-9), q_min, q_max

features = ['num_publications', 'total_citations', 'mean_citations', 'perc_publications']
top_authors_scaled = top_authors[features].copy()
scaling_params = {}

for col in features:
    scaled, q_min, q_max = robust_minmax(top_authors[col])
    top_authors_scaled[col] = scaled
    scaling_params[col] = (q_min, q_max)

# --- Radar chart setup ---
labels = ['# publications', 'citations', 'mean citations', '% publications']
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# --- 1) CALCULATE and STORE label positions ---

label_positions = {}

for i, row in top_authors_scaled.iterrows():
    values_original = top_authors.loc[i, features].tolist()
    values_original += values_original[:1]
    label = top_authors['first_author'][i]
    label_positions[label] = []

    for angle, feat, orig_val in zip(angles, features + [features[0]], values_original):
        q_min, q_max = scaling_params[feat if feat != features[0] else features[0]]
        scaled_val = (orig_val - q_min) / (q_max - q_min + 1e-9)

        r_label = scaled_val + label_offset
        label_positions[label].append((angle, r_label))

# --- Custom rotation mapping by feature ---
rotation_map = {
    'perc_publications': 90,
    'total_citations': -90
}

# --- 2) PLOT using stored positions ---

fig, ax = plt.subplots(figsize=(18, 18), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for i, row in top_authors_scaled.iterrows():
    values_scaled = row.tolist() + row.tolist()[:1]
    label = top_authors['first_author'][i]

    ax.plot(angles, values_scaled, label=label)
    ax.fill(angles, values_scaled, alpha=0.1)

    for idx, ((angle, r_label), feat, orig_val) in enumerate(zip(label_positions[label], features + [features[0]], top_authors.loc[i, features].tolist() + [top_authors.loc[i, features[0]]])):
        q_min, q_max = scaling_params[feat]
        scaled_val = (orig_val - q_min) / (q_max - q_min + 1e-9)

        ax.plot([angle, angle], [scaled_val, r_label], color='gray', linewidth=1)

        rotation = rotation_map.get(feat, 0)
        # Modify here:
        if feat in ['num_publications', 'total_citations']:
            display_val = f'{int(orig_val)}'  # no decimal
        elif feat == 'perc_publications':
            display_val = f'{orig_val:.3f}'  # no decimal
        else:
            display_val = f'{orig_val:.1f}'    # 1 decimal place

        ax.text(angle, r_label, display_val, ha='center', va='center',
                fontsize=fs3, rotation=rotation)

# --- Styling the plot ---
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# --- Styling ---
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Remove the '% publications' and 'citations' grid labels
custom_labels = ['# publications', '', 'mean citations', '']
ax.set_thetagrids(np.degrees(angles[:-1]), custom_labels, fontsize=fs2)

# Force the drawing to obtain correct limits
plt.draw()

# --- Manually add rotated text boxes in the right places ---

# Angular position of removed labels
angle_dict = dict(zip(labels, angles[:-1]))

# Text box for '% publications'
angle_perc = angle_dict['% publications']
ax.text(angle_perc, 1.35, '% publications',
        fontsize=fs2, rotation=90,
        ha='center', va='center')

# Text box for 'citations'
angle_cit = angle_dict['citations']
ax.text(angle_cit, 1.35, 'citations',
        fontsize=fs2, rotation=-90,
        ha='center', va='center')

# Title
ax.set_title('Top 5 Corresponding Authors by number of publications', fontsize=fs1, pad=30)

# Remove the radial axis labels
ax.set_yticklabels([])

# Legend
legend = ax.legend(loc='lower right', bbox_to_anchor=(1.25, -0.1), fontsize=fs2)
legend.get_frame().set_facecolor('white')

plt.tight_layout()


plt.savefig("q3_2_5/q3_2_5_top5_corresp_authors_metrics.pdf") 

plt.show()

In [ ]:
# Execute data processing or visualization steps
# table of last authors (NOT related to the plot above)


# --- Part 1: Calculation of the Statistics by Author and Institution ---
print("Calculating statistics per pair (author, institution)...")
# THE MAIN CHANGE IS HERE: Group by both columns.
author_institution_stats = df_data.groupby(['last_author', 'last_institution']).agg(
    publications=('citations_tot', 'count'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean')
).reset_index() # .reset_index() transforms the groups ('last_author', 'last_institution') into columns

# --- Part 2: Sort and Select the Top 20 ---
print("Sorting and formatting the final table...")
# Sort by the results of the new grouping and get the top 20 pairs.
df_final = author_institution_stats.sort_values('publications', ascending=False)
df_top20 = df_final.head(20).copy()


# --- Part 3: Rename and Format Columns ---
# Renaming columns for the final presentation
df_top20.rename(columns={
    'last_author': 'Author',
    'last_institution': 'Affiliation',
    'publications': '# Pubs',
    'total_citations': 'Tot Cites',
    'mean_citations': 'Mean Cites'
}, inplace=True)

# Data type conversion
df_top20['# Pubs'] = df_top20['# Pubs'].astype(int)
df_top20['Tot Cites'] = df_top20['Tot Cites'].astype(int)
df_top20['Mean Cites'] = df_top20['Mean Cites'].round(1)

# REORDERING THE COLUMNS so that 'Institution' is second
df_top20 = df_top20[['Author', 'Affiliation', '# Pubs', 'Tot Cites', 'Mean Cites']]


# --- Part 4: Generate LaTeX Table ---
print("Generating LaTeX CODE for the table...")

# Defining the format of the columns for LaTeX into 5 columns.
# p{width} = allows line breaks in long texts.
# l = left-aligned, r = right-aligned.
column_format = "p{3.5cm} p{4.5cm} r r r"
new_header=["Author",  'Affiliation', "\\shortstack[r]{{#\\\\Pubs}}", '\\shortstack[r]{{Tot\\\\Cites}}', '\\shortstack[r]{{Mean\\\\Cites}}']
# new_header=["Author",  'Affiliation', "{{sdfgsdf\}}", 'sdfgsdfg', 'sdfgdsfg']

latex_table = df_top20.to_latex(
    buf=None,
    caption="Top 20 Author-Institution Pairs by Publications",
    label="tab:top20_author_institution_pairs",
    index=False,  # CHANGED: use 'False' to not include a rank/index column.
    column_format=column_format,
    escape=True,
    float_format="%.1f",
    header=new_header
)

print("\n--- LATEX CODE OF THE TABLE ---")
print(latex_table)

In [ ]:
# Execute data processing or visualization steps
# table of the top 20 last authors with their last affiliated institution

# --- Part 1: Calculation of General Statistics by Author ---
print("Calculating general statistics by author...")
# THE groupby is now done only by 'last_author'
author_stats = df_data.groupby('last_author').agg(
    publications=('citations_tot', 'count'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean')
).reset_index()


# --- Part 2: Determine the Most Recent Affiliation of Each Author ---
print("Determining the most recent affiliation of each author...")
# Sort the data by author and, descendingly, by year
df_sorted_by_year = df_data.sort_values(['last_author', 'year'], ascending=[True, False])
# For each author, get the first entry, which corresponds to the most recent year
latest_affiliations = df_sorted_by_year.drop_duplicates(subset='last_author', keep='first')[['last_author', 'last_institution']]


# --- Part 3: Combine Statistics and Recent Affiliation ---
print("Combining general statistics with the most recent affiliation...")
# Join the two DataFrames by the 'last_author' column
df_merged = pd.merge(author_stats, latest_affiliations, on='last_author')


# --- Part 4: Sort, Select Top 20 and Format ---
# Sort by the author's total publication count
df_final = df_merged.sort_values('publications', ascending=False)
df_top20 = df_final.head(20).copy()

df_top20['last_author'] = df_top20['last_author'].str.title()  # Name formatting

# Renaming columns
df_top20.rename(columns={
    'last_author': 'Author',
    'last_institution': 'Last Affiliation',
    'publications': '# Pubs',
    'total_citations': 'Tot Cites',
    'mean_citations': 'Mean Cites'
}, inplace=True)

# Data type conversion
df_top20['# Pubs'] = df_top20['# Pubs'].astype(int)
df_top20['Tot Cites'] = df_top20['Tot Cites'].astype(int)
df_top20['Mean Cites'] = df_top20['Mean Cites'].round(1)

# REORDERING THE COLUMNS so that 'Last Affiliation' is LAST
df_top20 = df_top20[['Author', '# Pubs', 'Tot Cites', 'Mean Cites', 'Last Affiliation']]


# --- Part 5: Generate LaTeX Table ---
print("Generating LaTeX CODE for the table...")

# Format for 5 columns: Author (text), 3x Stats (numbers), Affiliation (text)
column_format = "p{4cm} r r r p{5cm}"

# Adjusting the header order to match the new column order
# final order: Author, # Pubs, Tot Cites, Mean Cites, Last Affiliation
custom_header = [
    "Author",
    "\\shortstack[r]{{\\#\\\\Pubs}}",
    '\\shortstack[r]{{Tot\\\\Cites}}',
    '\\shortstack[r]{{Mean\\\\Cites}}',
    'Last Affiliation'
]

latex_table = df_top20.to_latex(
    buf=None,
    caption="Top 20 Last Authors by Total Publications and Their Most Recent Affiliation",
    label="tab:top20_authors_by_pubs",
    index=False,
    column_format=column_format,
    escape=False,  # IMPORTANT: False so LaTeX commands in the header work
    float_format="%.1f",
    header=custom_header
)

print("\n--- LATEX CODE OF THE TABLE ---")
print(latex_table)

In [ ]:
# Execute data processing or visualization steps
# table top 20 last author

# --- Grouping and metrics ---
df_authors = df_data.copy()
df_authors['last_author'] = df_authors['last_author'].str.strip().str.title()

agg_df = df_authors.groupby('last_author').agg(
    num_publications=('title', 'count'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean'),
    first_country=('last_country', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
).reset_index()

# agg_df['perc_publications'] = agg_df['num_publications'] / agg_df['num_publications'].sum() * 100


top_n = 20
top_authors = agg_df.sort_values(by='num_publications', ascending=False).head(top_n).reset_index(drop=True)

latex = top_authors.to_latex(
    index=False,
    float_format="%.2f",
    column_format="lccccc",
    header=["Author", "# Publications", "Total Citations", "Mean Citations", "First Country"]
)
with open("q3_2_5/q3_2_5_top20_last_authors_metrics.tex", "w") as f:
    f.write(latex)
    
# Display or save
print(latex)

### Question S-3.6
How of the international collaborations influence the advances in H2 production?

In [ ]:
# Execute data processing or visualization steps
# Font sizes
fs1 = 40  # title
fs2 = 30  # Axes title
fs3 = 28  # data labels on Axes

# Remove duplicates in the list of countries per document
df_data['unique_countries'] = df_data['countries'].apply(lambda x: list(set(x)))
df_data['n_countries'] = df_data['unique_countries'].apply(len)

# Calculate citations per year of each paper
current_year = pd.Timestamp.now().year
df_data['citations_per_year'] = df_data['citations_tot'] / (current_year - df_data['year'] + 1)

# Group: for each N countries, get a list of the average citations per year of the documents
grouped = df_data.groupby('n_countries')['citations_per_year'].apply(list).sort_index()

# Plot
plt.figure(figsize=(20, 10))
plt.boxplot(grouped.values, positions=grouped.index, patch_artist=True)

plt.yscale('log')

# Customization
plt.title('Distribution of Average Citations per Year vs Number of Countries per Document', fontsize=fs1)
plt.xlabel('Number of Countries (without duplicates per document)', fontsize=fs2)
plt.ylabel('Average Citations per Year (per document)', fontsize=fs2)
plt.xticks(fontsize=fs3)
plt.yticks(fontsize=fs3)
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.gca().set_facecolor('white')  # White background

plt.tight_layout()
plt.show()

In [ ]:
# Execute data processing or visualization steps
# Fonts
fs1 = 44  # title
fs2 = 34  # Axes title
fs3 = 32  # data labels on Axes

# Remove duplicates in the lists of countries per document
df_data['unique_countries'] = df_data['countries'].apply(lambda x: list(set(x)))
df_data['n_countries'] = df_data['unique_countries'].apply(len)

# Calculate citations per year
current_year = pd.Timestamp.now().year
df_data['citations_per_year'] = df_data['citations_tot'] / (current_year - df_data['year'] + 1)

# Create plot
plt.figure(figsize=(12, 8))

# Boxplot without visual outliers
sns.boxplot(
    x='n_countries',
    y='citations_per_year',
    data=df_data,
    showfliers=False,     # Hides the outliers from the boxplot (we'll display them with a stripplot)
    width=0.6,
    palette='pastel'
)

# Stripplot with transparency to show density of points
sns.stripplot(
    x='n_countries',
    y='citations_per_year',
    data=df_data,
    color='black',
    alpha=0.3,
    jitter=True,         # Spreads the points horizontally
    size=4
)

# Log scale on the Y axis
plt.yscale('log')

# Customization
plt.title('Mean Citations/Year at\nDifferent Levels of Collaboration', fontsize=fs1)
plt.xlabel('Cross-Country Collaboration', fontsize=fs2)
plt.ylabel('Mean citations per year\n(log scale)', fontsize=fs2)
plt.xticks(fontsize=fs3)
plt.yticks(fontsize=fs3)
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.gca().set_facecolor('white')  # White background

plt.tight_layout()

plt.savefig("q3_2_6/q3_2_6_cross_country_collaboration_citations.pdf") 
plt.savefig("q3_2_6/q4_6_cross_country_collaboration_citations.pdf") 

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Filter only positive values
df_plot = df_data[df_data['citations_per_year'] > 0]

# Calculate grouped statistics by number of countries
summary_table = df_plot.groupby('n_countries')['citations_per_year'].agg([
    ('# Documents', 'count'),
    ('Min', 'min'),
    ('1st Quartile', lambda x: x.quantile(0.25)),
    ('Median', 'median'),
    ('3rd Quartile', lambda x: x.quantile(0.75)),
    ('Max', 'max'),
    ('Mean', 'mean'),
]).reset_index()

# Rename the number of countries column
summary_table.rename(columns={'n_countries': 'Countries'}, inplace=True)

# Save as LaTeX
latex_output = summary_table.to_latex(index=False, float_format="%.2f")

# Save in .tex file
with open("q3_2_6/q3_2_6_cross_country_collaboration_citations.tex", "w") as f:
    f.write(latex_output)

# Display (optional)
print(latex_output)

In [ ]:
# Execute data processing or visualization steps
from itertools import combinations

# Fonts
fs1 = 40  # Title
fs2 = 30  # Axes Title
fs3 = 28  # Data Label

# Dataframe example
# df_data = pd.read_csv("yourfile.csv")


import plotly.colors as pc

def generate_color_bar_shapes(scale, log_min, log_max, x_start=1.0, x_end=1.015, y_start=0.0, y_end=1.0, steps=100):
    colors = pc.sample_colorscale(scale, [i / steps for i in range(steps)])
    shapes = []

    for i in range(steps):
        y0 = y_start + (y_end - y_start) * i / steps
        y1 = y_start + (y_end - y_start) * (i + 1) / steps

        shapes.append(dict(
            type="rect",
            xref="paper", yref="paper",
            x0=x_start, x1=x_end,
            y0=y0, y1=y1,
            fillcolor=colors[i],
            line=dict(width=0)
        ))

    return shapes

# Remove duplicates in the lists of countries
df_data['countries_unique'] = df_data['countries'].apply(lambda x: list(set(x)))

# Count of publications by main country
pubs_by_country = df_data['first_country'].value_counts().reset_index()
pubs_by_country.columns = ['country', 'count']

# create column with logarithmic values for better visualization
pubs_by_country['log_count'] = np.log1p(pubs_by_country['count']) 

# Count collaborations between pairs
collaborations = Counter()
for country_list in df_data['countries_unique']:
    if len(country_list) > 1:
        for pair in combinations(sorted(country_list), 2):
            collaborations[pair] += 1

# Approximate coordinates per country
country_coords = {
    'United States': (39.5, -98.35),
    'Brazil': (-14.235, -51.9253),
    'China': (35.8617, 104.1954),
    'Germany': (51.1657, 10.4515),
    'India': (20.5937, 78.9629),
    'United Kingdom': (55.3781, -3.435973),
    'France': (46.6034, 2.2137),
    'Canada': (56.1304, -106.3468),
    'Japan': (36.2048, 138.2529),
    'Australia': (-25.2744, 133.7751),
    'Italy': (41.8719, 12.5674),
    'Spain': (40.4637, -3.7492),
    'Russia': (61.5240, 105.3188),
    'South Korea': (35.9078, 127.7669),
    'South Africa': (-30.5595, 22.9375),
    
    # new added countries
    'United States of America': (39.5, -98.35),
    'Saudi Arabia': (23.8859, 45.0792),
    # 'Singapore': (1.3521, 103.8198),
    # 'Pakistan': (30.3753, 69.3451),
    # 'Taiwan': (23.6978, 120.9605),
    # 'Iran': (32.4279, 53.6880),
    # 'Russian Federation': (61.5240, 105.3188),
    # 'Switzerland': (46.8182, 8.2275),
    # 'Poland': (51.9194, 19.1451),
    # 'Egypt': (26.8206, 30.8025),
    # 'Sweden': (60.1282, 18.6435),
    # 'Portugal': (39.3999, -8.2245),
    # 'Czechia': (49.8175, 15.4730),
    # 'Denmark': (56.2639, 9.5018),
    # 'Viet Nam': (14.0583, 108.2772),
    # 'Turkey': (38.9637, 35.2433),
    # 'Malaysia': (4.2105, 101.9758),
    # 'Israel': (31.0461, 34.8516),
    # 'United Arab Emirates': (23.4241, 53.8478),
    # 'Mexico': (23.6345, -102.5528),
    # 'Norway': (60.4720, 8.4689),
    # 'Netherlands': (52.1326, 5.2913)
}

# Choropleth map for publications by country
fig = px.choropleth(
    pubs_by_country,
    locations="country",
    locationmode="country names",
    color="log_count",
    color_continuous_scale="Reds",
    projection="natural earth",
)

# Add the collaboration lines
for (country1, country2), weight in collaborations.items():
    if country1 in country_coords and country2 in country_coords:
        lat1, lon1 = country_coords[country1]
        lat2, lon2 = country_coords[country2]
        fig.add_trace(go.Scattergeo(
            lon=[lon1, lon2],
            lat=[lat1, lat2],
            mode='lines',
            line=dict(width=0.5 + weight**0.3, color='chocolate'),
            opacity=0.5,
            hoverinfo='none',
        ))






# Define the desired real values and their logs
ticks_real = [1, 10, 100, 1000, 5000, pubs_by_country['count'].max()]
ticks_log = np.log1p(ticks_real)

annotations = []

for real, logval in zip(ticks_real, ticks_log):
    y_pos = (logval - ticks_log.min()) / (ticks_log.max() - ticks_log.min())
    annotations.append(dict(
        x=1.02,
        y=y_pos,
        xref='paper',
        yref='paper',
        text=f'{real if real < 1000 else f"{int(real/1000)}K"}',
        showarrow=False,
        font=dict(size=fs2),
        xanchor='left'
    ))

# Color bar title
annotations.append(dict(
    x=0.9,
    y=1.10,
    xref='paper',
    yref='paper',
    text='Publications',
    showarrow=False,
    font=dict(size=fs2),
    xanchor='left'
))

log_min = np.log1p(min(ticks_real))
log_max = np.log1p(max(ticks_real))
color_bar_shapes = generate_color_bar_shapes(
    scale='Reds',
    log_min=log_min,
    log_max=log_max,
    x_start=0.985,
    x_end=1.005,
    y_start=0,
    y_end=1,
    steps=100
)






# final layout
fig.update_layout(
    title_text='International Collaboration',
    title_font_size=fs1,
    title_x=0.5,
    geo=dict(
        showland=True,
        landcolor='white',
        showframe=False,
        showcountries=True,
    ),
    margin=dict(l=0, r=120, t=80, b=0),  # Right margin space
    showlegend=False,
    height=600,
    width=1200,
    annotations=annotations,
    coloraxis_showscale=False,
    shapes=color_bar_shapes
)

fig.write_image("q3_2_6/q3_2_6_cross_country_collaboration.pdf") 

fig.show()



# 1. Extract all the countries from the DataFrame
# Include countries from the 'first_country' column and from the 'countries_unique' lists
all_countries = [c for sublist in df_data['countries_unique'] for c in sublist]

# 2. Count number of documents by country (how many times it appears in 'countries_unique')
country_counts = Counter(all_countries)

# 3. Set of countries with defined coordinates
countries_with_coords = set(country_coords.keys())

# 4. Find countries from the DataFrame that are missing in the coordinates dictionary
missing_countries = {
    country: count for country, count in country_counts.items()
    if country not in countries_with_coords
}

# 5. Sort descending by number of documents
missing_countries_sorted = sorted(missing_countries.items(), key=lambda x: x[1], reverse=True)

# 6. Display
print(f"Number of countries without coordinates: {len(missing_countries_sorted)}")
print("Missing countries and number of documents:")
for country, count in missing_countries_sorted:
    print(f"{country}: {count}")

In [ ]:
# Execute data processing or visualization steps
# Filter collaborations with coordinates available for both countries
visible_collab = {
    pair: count for pair, count in collaborations.items()
    if pair[0] in country_coords and pair[1] in country_coords
}

# Sort by number of collaborations (descending)
sorted_collab = sorted(visible_collab.items(), key=lambda x: x[1], reverse=True)

# Header of the LaTeX table
latex_table = r"""\begin{table}[ht]
\centering
\caption{Collaborations between countries with visible lines on the map}
\label{tab:collaborations_visible}
\begin{tabular}{l l r}
\toprule
\textbf{country 1} & \textbf{country 2} & \textbf{collaborations} \\
\midrule
"""

# Add the rows
for (country1, country2), count in sorted_collab:
    latex_table += f"{country1} & {country2} & {count} \\\\\n"

# Close the table
latex_table += r"""\bottomrule
\end{tabular}
\end{table}
"""

# Displays the LaTeX table
print(latex_table)

## Extras

### Last authors and their collaboration period

In [ ]:
# Execute data processing or visualization steps
import matplotlib.ticker as mticker

fs1 = 24
fs2 = 18

# 1. Find the 10 authors with the most publications
top_10_last_authors = df_data['last_author'].value_counts().nlargest(15).index

# 2. Filter the DataFrame to include only the 10 main authors
df_top_10 = df_data[df_data['last_author'].isin(top_10_last_authors)]

# 3. Calculate the collaboration period for each author-coauthor pair
collaboration_periods = df_top_10.groupby(['last_author', 'first_author'])['year'].agg(['min', 'max']).reset_index()
collaboration_periods['period'] = collaboration_periods['max'] - collaboration_periods['min'] + 1

# 4. For each author, find the minimum, maximum, and average collaboration period
author_collaboration_summary = collaboration_periods.groupby('last_author')['period'].agg(['min', 'max', 'mean']).reset_index()

# Sort the authors to match the order of the top 10
author_collaboration_summary['last_author'] = pd.Categorical(
    author_collaboration_summary['last_author'],
    categories=top_10_last_authors,
    ordered=True
)
author_collaboration_summary = author_collaboration_summary.sort_values('last_author')
author_collaboration_summary['last_author'] = author_collaboration_summary['last_author'].str.title()  # Name formatting

# --- 5. Create the horizontal bar plot (image style) ---

# Define a color palette inspired by the image
num_authors = len(author_collaboration_summary)
cmap = plt.get_cmap('cividis') 
colors = cmap(np.linspace(0.2, 0.8, num_authors))

# Figure settings
fig, ax = plt.subplots(figsize=(10, 8))
# fig.patch.set_facecolor('#FBF5EF') # Cream/beige background color
# ax.set_facecolor('#FBF5EF')
fig.patch.set_facecolor('white') # Cream/beige background color
ax.set_facecolor('white')

# Y position for the bars
y_pos = np.arange(len(author_collaboration_summary['last_author']))

# The width of the bars is the difference between the maximum and minimum period
bar_widths = author_collaboration_summary['max'] - author_collaboration_summary['min']

# The starting point of the bars is the minimum period
bar_starts = author_collaboration_summary['min']

# Plot the horizontal bars that represent the interval [min, max]
ax.barh(y_pos, bar_widths, left=bar_starts, align='center', height=0.5,
        color=colors, alpha=0.8, edgecolor=colors, linewidth=1.5)

# Plot the average markers over the bars
ax.scatter(author_collaboration_summary['mean'], y_pos,
           marker='|', color='black', s=100, zorder=10, linewidths=1.5)

# --- Style and Label Adjustments ---

# Move the X axis to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Remove the plot borders (spines) for a cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Configure the Y axis ticks
ax.set_yticks(y_pos)
ax.set_yticklabels(author_collaboration_summary['last_author'], fontfamily='serif', fontsize=fs2)
ax.invert_yaxis()  # Invert the axis so the main author is on top

# Limit the X axis with a small margin
ax.set_xlim(0, author_collaboration_summary['max'].max() * 1.05)

# Add a subtle vertical grid
ax.grid(axis='x', linestyle='-', color='gray', alpha=0.3, linewidth=0.5)
ax.grid(axis='y', linestyle='', alpha=0) # Remove Y axis grid

# Force the display of all whole numbers on the X axis
ax.xaxis.set_major_locator(mticker.MultipleLocator(base=1))

# Add titles, subtitles and font using fig.text for precise positioning
# fig.text(0.1, 0.95, 'THE WALK OF THE COAUTHORS',
# fontsize=16, fontweight='bold', ha='left', va='top', fontfamily='serif')
fig.text(0.1, 0.92, 'Collaboration longevity for the top 15 authors',
         fontsize=fs1, ha='left', va='top', fontfamily='serif')
# fig.text(0.1, 0.02, 'SOURCE: Data analysis from your institution',
# fontsize=8, ha='left', va='bottom', color='gray', fontfamily='serif')

# X axis label
ax.set_xlabel('Longevity in years', fontsize=fs2, fontfamily='serif', labelpad=10)
ax.tick_params(axis='x', labelsize=fs2)


plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.9]) # Adjust layout to fit texts

plt.savefig("q3_2_5/q3_2_5_top15_longevity.pdf") 

plt.show()



# plt.savefig('collaboration_period.png')

In [ ]:
# Execute data processing or visualization steps
# Collaboration period by topic

import matplotlib.ticker as mticker
import textwrap


fs1 = 28
fs2 = 22

# 1. Calculate the collaboration period for each pair ('last_author', 'first_author')
collaboration_periods = df_data.groupby(['last_author', 'first_author'])['year'].agg(['min', 'max']).reset_index()
collaboration_periods['period'] = collaboration_periods['max'] - collaboration_periods['min'] + 1

# 2. Associate each collaboration period to the corresponding topics
# First, we obtain the unique author-topic pairs from the original DataFrame
unique_author_topics = df_data[['last_author', 'first_author', 'topic_name_cluster3_reduced']].drop_duplicates()

# Now, we join the collaboration periods with the topics
topic_collaboration_data = pd.merge(
    unique_author_topics,
    collaboration_periods[['last_author', 'first_author', 'period']],
    on=['last_author', 'first_author']
)

# 3. For each topic, find the minimum, maximum and average collaboration duration
topic_summary = topic_collaboration_data.groupby('topic_name_cluster3_reduced')['period'].agg(['min', 'max', 'mean']).reset_index()

# Sort the topics by name for consistency in the plot
topic_summary = topic_summary.sort_values('topic_name_cluster3_reduced')


# -- make the y labels use two lines
wrapped_labels = [
    "\n".join(textwrap.wrap(label, width=55))  # adjust the width as needed
    for label in topic_summary['topic_name_cluster3_reduced']
]

# --- 4. Create the horizontal bar plot ---

# Define a color palette
num_topics = len(topic_summary)
cmap = plt.get_cmap('cividis')
colors = cmap(np.linspace(0.2, 0.8, num_topics))

# Figure settings
fig, ax = plt.subplots(figsize=(11, 14))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Y position for the bars
y_pos = np.arange(len(topic_summary['topic_name_cluster3_reduced']))

# The width of the bars is the difference between the maximum and minimum period
bar_widths = topic_summary['max'] - topic_summary['min']
# The starting point of the bars is the minimum period
bar_starts = topic_summary['min']

# Plot the horizontal bars that represent the interval [min, max]
ax.barh(y_pos, bar_widths, left=bar_starts, align='center', height=0.5,
        color=colors, alpha=0.8, edgecolor=colors, linewidth=1.5)

# Plot the average markers over the bars
ax.scatter(topic_summary['mean'], y_pos,
           marker='|', color='black', s=150, zorder=10, linewidths=2)

# --- Style and Label Adjustments ---

# Move the X axis to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Remove the plot borders (spines)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Configure the Y axis ticks
# ax.set_yticks(y_pos)
# ax.set_yticklabels(topic_summary['topic_name_cluster3_reduced'], fontfamily='serif', fontsize=fs2)
ax.set_yticks(y_pos)
ax.set_yticklabels(wrapped_labels, fontfamily='serif', fontsize=fs2)
ax.invert_yaxis()  # Invert the axis so the first topic is on top

# Limit the X axis starting at 0 with a small margin
ax.set_xlim(0, topic_summary['max'].max() * 1.05)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))

# Add vertical grid
ax.grid(axis='x', linestyle='-', color='gray', alpha=0.3, linewidth=0.5)
ax.grid(axis='y', linestyle='', alpha=0)



# Add titles and labels
fig.text(0.15, 0.95, 'Collaboration Longevity by Topic',
         fontsize=fs1, ha='left', va='top', fontfamily='serif')

ax.set_xlabel('Longevity in years', fontsize=fs2, fontfamily='serif', labelpad=10)
ax.tick_params(axis='x', labelsize=fs2, labelrotation=45)

plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.93])

plt.savefig("q3_2_5/q3_2_5_topic_longevity.pdf") 

plt.show()

In [ ]:
# Execute data processing or visualization steps
# Get the top 20 universities based on the document count
top_20_universities = df_data['last_institution'].value_counts().nlargest(20).index

# Filter the DataFrame to include only the top 20 universities
df_top_20 = df_data[df_data['last_institution'].isin(top_20_universities)]

# Count the number of unique 'last_author' for each of the top 20 universities
author_counts_by_university = df_top_20.groupby('last_institution')['last_author'].nunique()

# Sort the results for better visualization
author_counts_by_university = author_counts_by_university.sort_values(ascending=False)

# Print the result
# print("Count of 'last_author' per university (Top 20):")
print(author_counts_by_university)

In [ ]:
# Execute data processing or visualization steps
from adjustText import adjust_text

# Define variables for font size
sa = 20  # Font size for subplot titles
sb = 20  # Font size for Axes and legends
sc = 26
sd = 16  # Font size for author names       

qtd_authors = 20

# Processes data for first authors
first_author_stats = df_data.groupby('first_author').agg(
    publications=('first_author', 'size'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean')
).reset_index()
top_10_first_authors = first_author_stats.nlargest(qtd_authors, 'publications')

# Processes data for last authors
last_author_stats = df_data.groupby('last_author').agg(
    publications=('last_author', 'size'),
    total_citations=('citations_tot', 'sum'),
    mean_citations=('citations_tot', 'mean')
).reset_index()
top_10_last_authors = last_author_stats.nlargest(qtd_authors, 'publications')

# Formats the author names to capitalize the first letter
top_10_first_authors['first_author'] = top_10_first_authors['first_author'].str.title()
top_10_last_authors['last_author'] = top_10_last_authors['last_author'].str.title()


# Create the plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 15)) # Increased height slightly

# Add a general title to the entire figure
fig.suptitle(f'Analysis of Top {qtd_authors} Authors\nby Publication and Citation Metrics',
             fontsize=sc)           # Larger size than subplot titles (sa=20)



# First plot: Top 10 First Authors
scatter1 = ax1.scatter(top_10_first_authors['publications'], top_10_first_authors['total_citations'],
                       c=top_10_first_authors['mean_citations'], cmap='viridis', s=150)

# --- MODIFICATION 1: Collect texts for adjustment ---
texts_ax1 = []
for i, txt in enumerate(top_10_first_authors['first_author']):
    texts_ax1.append(ax1.text(top_10_first_authors['publications'].iloc[i],
                             top_10_first_authors['total_citations'].iloc[i],
                             txt,
                             fontsize=sd))

# Calls the adjust_text function to position texts without overlapping
adjust_text(texts_ax1, ax=ax1, arrowprops=dict(arrowstyle='->', color='grey', lw=0.5))
# --------------------------------------------------

ax1.set_title('(A) First authors', loc='left', fontsize=sa)
ax1.set_xlabel('Number of Publications', fontsize=sb)
ax1.set_ylabel('Total Citations', fontsize=sb)
ax1.tick_params(axis='both', which='major', labelsize=sb)
ax1.set_facecolor('white')
ax1.grid(False)
cbar1 = fig.colorbar(scatter1, ax=ax1)
cbar1.set_label('Mean Citations', fontsize=sb)
cbar1.ax.tick_params(labelsize=sb)


# Second plot: Top 10 Last Authors
scatter2 = ax2.scatter(top_10_last_authors['publications'], top_10_last_authors['total_citations'],
                       c=top_10_last_authors['mean_citations'], cmap='viridis', s=150)

# --- MODIFICATION 2: Collect texts for adjustment ---
texts_ax2 = []
for i, txt in enumerate(top_10_last_authors['last_author']):
    texts_ax2.append(ax2.text(top_10_last_authors['publications'].iloc[i],
                             top_10_last_authors['total_citations'].iloc[i],
                             txt,
                             fontsize=sd))

# Calls the adjust_text function again for the second plot
adjust_text(texts_ax2, ax=ax2, arrowprops=dict(arrowstyle='->', color='grey', lw=0.5))
# --------------------------------------------------

ax2.set_title('(B) Last authors', loc='left', fontsize=sa)
ax2.set_xlabel('Number of Publications', fontsize=sb)
ax2.set_ylabel('Total Citations', fontsize=sb)
ax2.tick_params(axis='both', which='major', labelsize=sb)
ax2.set_facecolor('white')
ax2.grid(False)
cbar2 = fig.colorbar(scatter2, ax=ax2)
cbar2.set_label('Mean Citations', fontsize=sb)
cbar2.ax.tick_params(labelsize=sb)


# Adjust layout to prevent overlap and save the figure
plt.tight_layout()

plt.savefig("q3_2_5/q3_2_5_top20_first_last_authors.pdf") 
plt.savefig("q3_2_5/q4_5_top20_first_last_authors.pdf") 

plt.show()

### Institution, first author, last author, country relation

In [ ]:
# Execute data processing or visualization steps
# --- Step 2: Cleaning and Pre-calculations ---

# skip data where 'last_institution' is null or an empty/space string.
df_data['last_institution'].replace(r'^\s*$', np.nan, regex=True, inplace=True)
df_data.dropna(subset=['last_institution'], inplace=True)

# Add a column for the total citations to facilitate the weighted average calculation
df_data['citations_sum'] = df_data['citations_tot']

# put author names with capitalized first letter
df_data['last_author'] = df_data['last_author'].str.title()
df_data['first_author'] = df_data['first_author'].str.title()

# Aggregations
agg_inst = df_data.groupby('last_institution').agg(
    total_pubs=('last_author', 'count'),
    total_cites=('citations_sum', 'sum')
).sort_values('total_pubs', ascending=False)
agg_inst['avg_cites'] = (agg_inst['total_cites'] / agg_inst['total_pubs'])

agg_last_author = df_data.groupby(['last_institution', 'last_author']).agg(
    total_pubs=('first_author', 'count'),
    total_cites=('citations_sum', 'sum')
).sort_values(['last_institution', 'total_pubs'], ascending=[True, False])
agg_last_author['avg_cites'] = (agg_last_author['total_cites'] / agg_last_author['total_pubs'])

agg_first_author = df_data.groupby(['last_institution', 'last_author', 'first_author']).agg(
    total_pubs=('year', 'count'),
    total_cites=('citations_sum', 'sum')
).sort_values(['last_institution', 'last_author', 'total_pubs'], ascending=[True, True, False])
agg_first_author['avg_cites'] = (agg_first_author['total_cites'] / agg_first_author['total_pubs'])


# --- Step 3: Auxiliary Text Wrapping Function ---
def wrap_text(text, width=25):
    """Wraps the text and formats it for LaTeX with \shortstack."""
    if len(text) <= width:
        return text
    wrapped_lines = textwrap.wrap(text, width, break_long_words=True, break_on_hyphens=False)
    latex_lines = r' \\ '.join(wrapped_lines)
    return f"\\shortstack[l]{{{latex_lines}}}"

# --- Step 4: Building the LaTeX String ---

top_15_institutions = agg_inst.head(15).index

latex_string = []

# Add commands to compact the table
latex_string.append(r"\begingroup")
latex_string.append(r"\setlength{\tabcolsep}{3pt} % Reduces the space between columns")
latex_string.append(r"\renewcommand{\arraystretch}{0.6} % Reduces row height")

# LaTeX Table Header
# **MODIFICATION 1: The last two columns are now 'r' for right alignment.**
latex_string.append(r"\begin{longtable}{p{3.8cm} p{3.8cm} p{4.2cm} r r}")
latex_string.append(r"\captionsetup{justification=centering, skip=0cm}")
latex_string.append(r"\caption{Top 15 Institutions With Breakdown by\\Top 3 Last Authors and Top 5 First Authors.}\\")
latex_string.append(r"\label{tab:top15_last_authors_by_inst} \\")
latex_string.append(r"\hline")
# **MODIFICATION 2 (Recommended): Numeric column headers aligned to the right.**
latex_string.append(r"\textbf{Institution} & \textbf{Last Author} & \textbf{First Author} & \textbf{\shortstack[r]{Total\\Pubs}} & \textbf{\shortstack[r]{Avg.\\Cites}} \\")
latex_string.append(r"\hline")
latex_string.append(r"\endfirsthead")
latex_string.append(r"\multicolumn{

In [ ]:
# Execute data processing or visualization steps
N_TOP_INSTITUTIONS = 5
N_TOP_LAST_AUTHORS = 3
N_TOP_FIRST_AUTHORS = 3



# --- Part 2: Procedural Sankey Construction Logic ---
print(f"Starting Sankey construction for: Top {N_TOP_INSTITUTIONS} Institutions, Top {N_TOP_LAST_AUTHORS} Last Authors, Top {N_TOP_FIRST_AUTHORS} First Authors.")

node_labels, node_colors, source_indices, target_indices, link_values = [], [], [], [], []
label_to_index = {}

def add_node(label, color):
    if label not in label_to_index:
        label_to_index[label] = len(node_labels)
        node_labels.append(label)
        node_colors.append(color)

inst_totals = df_data['last_institution'].value_counts()
top_inst = inst_totals.nlargest(N_TOP_INSTITUTIONS)

for inst_name, inst_pubs in top_inst.items():
    inst_label = f"{inst_name} ({inst_pubs})"
    add_node(inst_label, 'rgba(31, 119, 180, 0.8)')
    inst_index = label_to_index[inst_label]

    df_inst = df_data[df_data['last_institution'] == inst_name]
    
    la_totals = df_inst['last_author'].value_counts()
    top_la = la_totals.nlargest(N_TOP_LAST_AUTHORS)
    
    la_to_process = list(top_la.items())
    
    others_la_pubs = la_totals.iloc[N_TOP_LAST_AUTHORS:].sum()
    if others_la_pubs > 0:
        others_la_name = f"Other Authors ({inst_name})"
        la_to_process.append((others_la_name, others_la_pubs))

    # --- START OF MODIFICATION 1 ---
    # Sort the last_authors list to ensure a descending order with "other" at the end
    la_to_process.sort(key=lambda x: (x[0].startswith("Other"), -x[1]))
    # --- END OF MODIFICATION 1 ---

    for la_name, la_pubs in la_to_process:
        la_label = f"{la_name.split(' (')[0]} ({la_pubs})"
        add_node(la_label, 'rgba(255, 127, 14, 0.8)')
        la_index = label_to_index[la_label]
        
        source_indices.append(inst_index)
        target_indices.append(la_index)
        link_values.append(la_pubs)

        if la_name.startswith("Other Authors"):
            top_la_names = top_la.index
            df_la = df_inst[~df_inst['last_author'].isin(top_la_names)]
        else:
            df_la = df_inst[df_inst['last_author'] == la_name]

        fa_totals = df_la['first_author'].value_counts()
        top_fa = fa_totals.nlargest(N_TOP_FIRST_AUTHORS)
        
        fa_to_process = list(top_fa.items())

        others_fa_pubs = fa_totals.iloc[N_TOP_FIRST_AUTHORS:].sum()
        if others_fa_pubs > 0:
            others_fa_label_base = f"Other Primary Authors ({la_name.split(' (

### Co-authorship graph

In [ ]:
# Execute data processing or visualization steps
# co-authorship graph

import networkx as nx

# Create graph
G = nx.Graph()
for _, row in pair_counts.iterrows():
    G.add_edge(row['last_author'], row['first_author'], weight=row['Num_Articles'])

# Use only the top authors to simplify visuals
top_edges = pair_counts[pair_counts['Num_Articles'] > 7]
G_sub = G.edge_subgraph([(row['last_author'], row['first_author']) for _, row in top_edges.iterrows()])

# Plot
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_sub, k=0.5)
weights = [G_sub[u][v]['weight'] for u,v in G_sub.edges()]
nx.draw(G_sub, pos, with_labels=True, node_size=500, font_size=8, width=weights, edge_color='gray')
plt.title("Co-authorship Network: Last vs First Authors")
plt.show()

### Citation correlation

In [ ]:
# Execute data processing or visualization steps
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load df_cross
df_cross = pd.read_pickle("df_cross.pkl")

# Explode referenced DOIs into rows
df_long = df_cross.explode("dois_ref")

# Filter only references between internal documents
internal_dois = set(df_cross["doi"])
df_internal_refs = df_long[df_long["dois_ref"].isin(internal_dois)]

# Create count table (who cites who)
pivot = pd.crosstab(df_internal_refs["doi"], df_internal_refs["dois_ref"])

# Display just a slice for easier visualization
subset = pivot.iloc[:30, :30]

plt.figure(figsize=(12, 10))
sns.heatmap(subset, cmap="Blues", linewidths=0.5)
plt.title("Heatmap of Citations between Internal Documents")
plt.xlabel("Referenced")
plt.ylabel("Referencing")
plt.show()

In [ ]:
# Execute data processing or visualization steps
# citation histograms

# Plot style
sns.set_theme(style="whitegrid")

# Calculate a derived column: external citations
df_data['citations_ext'] = df_data['citations_tot'] - df_data['citacoes_int']
# df_temp = df_data[df_data["references_doi"].apply(lambda x: len(x) > 100)]
df_temp = df_data

# Define variables for plotting
variaveis = ['references_int', 'references_ext', 'citacoes_int', 'citations_ext']
titulos = ['Internal References', 'External References', 'Internal Citations', 'External Citations']

# Create a figure with 4 subplots
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs = axs.flatten()


# Generate histograms
for i, var in enumerate(variaveis):
    sns.histplot(df_temp[var], bins=100, kde=False, ax=axs[i], color='steelblue')
    axs[i].set_title(titulos[i], fontsize=14)
    axs[i].set_xlabel('')
    axs[i].set_ylabel('Frequency')

# Adjust layout
plt.tight_layout()
plt.show()

for var in variaveis:
    print(f"{var}: {min(df_temp[var])} - {max(df_temp[var])}")

print()
print("Internal references: amount of documents referenced by the articles and that are part of the corpus")
print("External references: amount of documents referenced by the articles and that are not part of the corpus")
print("Internal citations: amount of citations the article received from documents that are part of the corpus")
print("External citations: amount of citations the article received from documents that are not part of the corpus (total citations - internal citations)")

print(f"Average amount of internal references: {df_temp['references_int'].mean():.2f}")
print(f"Average amount of external references: {df_temp['references_ext'].mean():.2f}")
print(f"Documents without internal references: {len(df_temp[df_temp['references_int'].apply(lambda x: x == 0)]):.0f}")

print(f"Average amount of internal citations: {df_temp['citacoes_int'].mean():.2f}")
print(f"Average amount of external citations: {df_temp['citations_ext'].mean():.2f}")
print(f"Documents without internal citations: {len(df_temp[df_temp['citacoes_int'].apply(lambda x: x == 0)]):.0f}")

In [ ]:
# Basic English comment: this cell performs code processing or visualization.
x = df_data[df_data["citacoes_int"].apply(lambda x: x <= 100)]
len(x)

In [ ]:
# Execute data processing or visualization steps
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load DataFrame
df_cross = pd.read_pickle("df_cross.pkl")
df_long = df_cross.explode("dois_ref")

# Set of internal DOIs
internal_dois = set(df_cross["doi"])

# Tag for each row: if the cited document is internal or external
df_long["tipo_referencia"] = df_long["dois_ref"].apply(lambda x: "interna" if x in internal_dois else "externa")

In [ ]:
# Execute data processing or visualization steps
# 1. the 10 MOST INTERNALLY CITED documents
mais_citados_internos = (
    df_long[df_long["tipo_referencia"] == "interna"]["dois_ref"]
    .value_counts()
    .head(10)
    .reset_index()
)
mais_citados_internos.columns = ["doi", "num_citacoes_internas"]

# 2. the 10 MOST EXTERNALLY CITED documents
mais_citados_externos = (
    df_long[df_long["tipo_referencia"] == "externa"]["dois_ref"]
    .value_counts()
    .head(10)
    .reset_index()
)
mais_citados_externos.columns = ["doi", "num_citacoes_externas"]

# 3. the 10 documents that CITED THE MOST INTERNAL ARTICLES
mais_citadores_internos = (
    df_long[df_long["tipo_referencia"] == "interna"]["doi"]
    .value_counts()
    .head(10)
    .reset_index()
)
mais_citadores_internos.columns = ["doi", "citacoes_a_internos"]

# 4. the 10 documents that CITED THE MOST EXTERNAL ARTICLES
mais_citadores_externos = (
    df_long[df_long["tipo_referencia"] == "externa"]["doi"]
    .value_counts()
    .head(10)
    .reset_index()
)
mais_citadores_externos.columns = ["doi", "citacoes_a_externos"]

# Show tables in the terminal
print("\n1. MOST INTERNALLY cited documents:\n", mais_citados_internos)
print("\n2. MOST EXTERNALLY cited documents:\n", mais_citados_externos)
print("\n3. Documents that cited the most INTERNAL ARTICLES:\n", mais_citadores_internos)
print("\n4. Documents that cited the most EXTERNAL ARTICLES:\n", mais_citadores_externos)

In [ ]:
# Execute data processing or visualization steps
def plot_bar(df, x, y, title, xlabel):
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df, x=x, y=y, palette="viridis")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("DOI")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# 1. Most INTERNALLY cited
plot_bar(mais_citados_internos, "num_citacoes_internas", "doi", 
         "Top 10 - Most Internally Cited", "Number of Internal Citations")

# 2. Most EXTERNALLY cited
plot_bar(mais_citados_externos, "num_citacoes_externas", "doi", 
         "Top 10 - Most Externally Cited", "Number of External Citations")

# 3. Most citers of INTERNAL ARTICLES
plot_bar(mais_citadores_internos, "citacoes_a_internos", "doi", 
         "Top 10 - Most Citing Internal Articles", "Number of Citations to Internal")

# 4. Most citers of EXTERNAL ARTICLES
plot_bar(mais_citadores_externos, "citacoes_a_externos", "doi", 
         "Top 10 - Most Citing External Articles", "Number of Citations to External")

In [ ]:
# Basic English comment: this cell performs code processing or visualization.
import seaborn as sns

data_melted = pd.melt(citacoes_por_doc.reset_index(), 
                      id_vars='doi', 
                      value_vars=['citacoes_internas', 'citacoes_externas'], 
                      var_name='tipo', 
                      value_name='citacoes')

plt.figure(figsize=(8, 6))
sns.violinplot(x='tipo', y='citacoes', data=data_melted, scale='width', inner='quartile')
plt.yscale("log")
plt.title("Logarithmic Distribution of Internal vs External Citations", fontsize=16)
plt.xlabel("Citation Type")
plt.ylabel("Number of Citations (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# Execute data processing or visualization steps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Internal Histogram
axes[0].hist(citacoes_por_doc["citacoes_internas"], bins=[-0.5, 0.5, 1.5, 2.5], color='blue', edgecolor='black')
axes[0].set_title("Distribution of Internal Citations", fontsize=16)
axes[0].set_xlabel("No of Internal Citations")
axes[0].set_ylabel("Frequency")

# External Histogram (log scale)
axes[1].hist(citacoes_por_doc["citacoes_externas"], bins=30, color='gray', edgecolor='black')
axes[1].set_yscale("log")
axes[1].set_title("Distribution of External Citations", fontsize=16)
axes[1].set_xlabel("No of External Citations")
axes[1].set_ylabel("Frequency (log)")

plt.tight_layout()
plt.show()

In [ ]:
# Basic English comment: this cell performs code processing or visualization.
media_internas = citacoes_por_doc["citacoes_internas"].mean()
media_externas = citacoes_por_doc["citacoes_externas"].mean()
sem_citacoes_internas = (citacoes_por_doc["citacoes_internas"] == 0).sum()

print(f"Number of documents without internal citations: {sem_citacoes_internas} -> {sem_citacoes_internas/len(citacoes_por_doc)*100:.0f}%")
print(f"Average internal citations: {media_internas:.2f}")
print(f"Average external citations: {media_externas:.2f}")